# VoltRelay Energy — Battery-Swap Network Analytics
**Gradient Learnings Data Analytics Hackathon · final submission notebook: load → clean → analyse → model → findings**

Runs top-to-bottom in Colab (High-RAM runtime recommended: `swap_events` is ~730 MB / 3.9 M rows; ~10 min end to end). Every random step uses the dataset seed `59500`.
Set `CFG["QUICK_RIDER_FRAC"] = 0.15` for a fast smoke test.

| Part | Content | Core question |
|---|---|---|
| 0 | Setup, config, helpers, automatic figure export | – |
| 1 | Download + memory-light loading of the 8 tables | – |
| 2 | Data understanding (grain, keys, which fields exist for which event type) | – |
| 3 | Cleaning (every data-quality note handled **and logged**), unit-economics model with **measured** battery wear | – |
| 4 | Network performance over time | **Q1** |
| 5 | Service failures & customer experience | **Q2** |
| 6 | Station & geographic patterns | **Q3** |
| 7 | Battery & equipment performance | **Q4** |
| 8 | Pricing & partner economics (event study, difference-in-differences, partner P&L) | **Q5** |
| 9 | Retention root-cause analysis (cohorts, curves, model + group importance) | **Q6** |
| 10 | Optional deep dives: expansion waves, ticket text, anomalies, budget decision | Optional |
| 11 | Follow-up analyses that test and correct Parts 4–10 (heat × generation, wear sensitivity, pilot, retention, waves, station priorities) | Q1–Q6 |
| 12 | Models: next-day station failure-risk forecast, rider early-warning, Gen1 fix break-even | Optional |
| 13 | Key findings, recommendations, limitations, exports | – |

**Design rules used throughout**
* A *service failure* = `failed_no_charged_battery` + `abandoned_queue` + `failed_system_error`. `cancelled_by_rider` is tracked but is the rider's choice, not a service fault.
* Revenue / cost fields are only used on `swap_completed` rows (other event types do not carry them).
* Timestamps are fixed (firmware bug) **before** any hour-of-day or date logic; the duplicate check runs **before** any counting.
* **Battery wear uses measured degradation** (pack cost × SoH lost per swap ÷ usable SoH window), not an assumed cycle life. The flat 1,500-cycle model is kept as `wear_cost_flat` and reported as a sensitivity, because the two give opposite answers on the *level* of loaded margin.
* "Associated with" is not "caused by": the data is observational and simulated; where a mechanism is not proven the text says so.
* Assumptions live in the `CFG` dict (cycle life for the sensitivity model, duplicate gap, km cap, retention window …).
* Numbers quoted in the **Reading it** blocks come from the executed run of this analysis (seeded, reproducible); if a re-run prints different figures, the printed output is authoritative.


## 0 · Setup

In [ ]:
!pip -q install -U gdown
import os, re, glob, time, warnings, gc
import numpy as np, pandas as pd
import matplotlib.pyplot as plt, matplotlib.ticker as mtick, seaborn as sns
warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams.update({"figure.dpi": 100, "axes.titlesize": 12, "axes.titleweight": "bold", "axes.labelsize": 10})

# every figure shown from here on is also saved to OUT (zip is created in Part 13)
OUT = "/content/voltrelay_outputs"; os.makedirs(OUT, exist_ok=True)
_orig_show = plt.show; _FIGN = [0]
def _show_and_save(*a, **k):
    _FIGN[0] += 1; plt.gcf().savefig(f"{OUT}/fig_{_FIGN[0]:02d}.png", dpi=150, bbox_inches="tight"); _orig_show(*a, **k)
plt.show = _show_and_save
pd.set_option("display.width", 250, "display.max_columns", 80, "display.max_rows", 150,
              "display.float_format", lambda v: f"{v:,.3f}")
SEED = 59500                      # generation seed of the dataset, reused for all sampling
rng = np.random.default_rng(SEED)
T_NB = time.time()

CFG = dict(
    FOLDER_URL = "https://drive.google.com/drive/folders/1HPrlSewrkQJ6LEWuxInYLuSW0OLx7n_y",
    DATA_DIR   = "/content/voltrelay_data",
    QUICK_RIDER_FRAC = None,   # e.g. 0.15 -> keep only 15% of riders (all their swaps/tickets) for a fast smoke test
    DUP_GAP_SEC  = 180,        # offline-sync retries: same rider/station/batteries/type within this many seconds
    KM_MAX       = 300,        # km_since_last_swap above this (or < 0) = odometer reset -> NaN
    CYCLE_LIFE   = 1500,       # ASSUMPTION: full cycles a pack delivers before replacement -> wear cost per swap = pack cost / CYCLE_LIFE
    EARLY_DAYS   = 14,         # retention study: "early experience" window after a rider's first attempt
    RET_FROM     = 30,         # retention study: rider counts as retained if active between day RET_FROM ...
    RET_TO       = 60,         #                  ... and day RET_TO after the first attempt
    MIN_ATTEMPTS = 500,        # minimum attempts for a station / zone to be ranked
)
DATA_DIR   = CFG["DATA_DIR"]
DATA_START = pd.Timestamp("2024-01-01")
DATA_END   = pd.Timestamp("2025-06-30 23:59:59")
T0W        = pd.Timestamp("2024-01-01")        # Monday -> week index origin
FIND = {}                                      # key numbers are collected here and printed in part 11
CLEAN_LOG = []                                 # every cleaning step is logged here

def log(step, n_affected, note=""):
    CLEAN_LOG.append(dict(step=step, rows_affected=int(n_affected), note=note))
    print(f"[clean] {step}: {int(n_affected):,} rows  {note}")

def seg_metrics(df, by, min_n=0):
    """Service-quality metrics for any grouping of the swap-attempt table."""
    g = df.groupby(by, observed=True)
    out = pd.DataFrame({
        "attempts":      g.size(),
        "svc_fails":     g["service_fail"].sum(),
        "service_fail_%": g["service_fail"].mean() * 100,
        "no_batt_%":     g["is_nobatt"].mean() * 100,
        "abandon_%":     g["is_abandon"].mean() * 100,
        "sys_err_%":     g["is_syserr"].mean() * 100,
        "cancel_%":      g["is_cancel"].mean() * 100,
        "queue_mean_s":  g["queue_wait_sec"].mean(),
        "queue_p90_s":   g["queue_wait_sec"].quantile(.9),
    })
    return out[out.attempts >= min_n]

EVENTS = {}
def mark_events(ax, events=None):
    ev = EVENTS if events is None else events
    for name, d in ev.items():
        if d is None or pd.isna(d): continue
        ax.axvline(d, ls="--", lw=1, color="grey", alpha=.8)
        ax.text(d, ax.get_ylim()[1], " " + name, rotation=90, va="top", ha="right", fontsize=7, color="dimgrey")

def pct_axis(ax, axis="y"):
    (ax.yaxis if axis == "y" else ax.xaxis).set_major_formatter(mtick.PercentFormatter(decimals=0))

def top_bottom(df, col, k=5):
    s = df.sort_values(col)
    return pd.concat([s.head(k), s.tail(k)])

## 1 · Data loading

In [ ]:
NEED = ["swap_events", "station_hourly_status", "riders", "batteries", "support_tickets",
        "stations", "city_daily_context", "fleet_partners"]

def find(name):
    hits = [p for p in glob.glob(f"{DATA_DIR}/**/*", recursive=True)
            if os.path.isfile(p) and os.path.basename(p).startswith(name) and p.endswith((".csv", ".csv.gz"))]
    assert hits, f"{name} not found under {DATA_DIR} - check the folder listing"
    return sorted(hits)[0]

def have_all():
    try:
        for n in NEED: find(n)
        return True
    except AssertionError:
        return False

os.makedirs(DATA_DIR, exist_ok=True)
if have_all():
    print("Data already present - skipping download")
else:
    import gdown
    gdown.download_folder(url=CFG["FOLDER_URL"], output=DATA_DIR, quiet=True, use_cookies=False)
for n in NEED:
    p = find(n); print(f"{os.path.getsize(p)/1e6:8.1f} MB  {p}")

In [ ]:
t = time.time()
# ---- swap_events: only needed columns, compact dtypes (event_id / attempt_seq / payment_mode are constant or unused)
sw_cat   = ["rider_id", "station_id", "event_type", "battery_in_id", "battery_out_id", "tariff_code", "station_firmware", "sync_mode"]
sw_float = ["soc_in_pct", "soh_in_pct", "soc_out_pct", "soh_out_pct", "km_since_last_swap",
            "list_price_inr", "discount_inr", "amount_charged_inr", "energy_to_recharge_kwh"]
sw_dtype = {**{c: "category" for c in sw_cat}, **{c: "float32" for c in sw_float}, "queue_wait_sec": "int32"}
sw_use   = sw_cat + sw_float + ["event_ts", "queue_wait_sec"]
swaps = pd.read_csv(find("swap_events"), usecols=sw_use, dtype=sw_dtype)
swaps["event_ts"] = pd.to_datetime(swaps["event_ts"])

# ---- hourly telemetry
hr_float = ["charged_2w_avg", "charged_2w_min", "charged_3w_min", "packs_charging", "packs_quarantined",
            "ambient_temp_c", "cabinet_temp_c", "avg_charge_minutes", "grid_kwh"]
hourly = pd.read_csv(find("station_hourly_status"),
                     dtype={**{c: "float32" for c in hr_float}, "station_id": "category", "telemetry_status": "category",
                            "chargers_online": "int16", "outage_minutes": "int16"})
hourly["hour_start"] = pd.to_datetime(hourly["hour_start"])

# ---- small tables
riders    = pd.read_csv(find("riders"), parse_dates=["signup_date"])
batteries = pd.read_csv(find("batteries"), parse_dates=["manufacture_date", "commission_date", "retired_date"])
tickets   = pd.read_csv(find("support_tickets"), parse_dates=["created_ts"])
stations  = pd.read_csv(find("stations"), parse_dates=["commissioned_date", "decommissioned_date",
                                                       "firmware_updated_date", "competitor_within_1_5km_since"])
city_ctx  = pd.read_csv(find("city_daily_context"), parse_dates=["date"])
partners  = pd.read_csv(find("fleet_partners"), parse_dates=["contract_start_date", "amendment_date"])

if CFG["QUICK_RIDER_FRAC"]:
    keep = set(riders.rider_id.sample(frac=CFG["QUICK_RIDER_FRAC"], random_state=SEED))
    riders  = riders[riders.rider_id.isin(keep)].reset_index(drop=True)
    swaps   = swaps[swaps.rider_id.isin(keep)].reset_index(drop=True)
    tickets = tickets[tickets.rider_id.isin(keep)].reset_index(drop=True)
    print(f"QUICK mode: kept {len(keep):,} riders")

print(f"loaded in {time.time()-t:.0f}s | swaps {swaps.shape} ({swaps.memory_usage(deep=True).sum()/1e9:.2f} GB) | hourly {hourly.shape}")
for n, d in dict(riders=riders, batteries=batteries, tickets=tickets, stations=stations, city_ctx=city_ctx, partners=partners).items():
    print(f"{n:10s} {d.shape}")
gc.collect();

## 2 · Data understanding

In [ ]:
print("swap_events  :", swaps.event_ts.min(), "->", swaps.event_ts.max())
print("hourly       :", hourly.hour_start.min(), "->", hourly.hour_start.max())
print("tickets      :", tickets.created_ts.min(), "->", tickets.created_ts.max())
print("riders signup:", riders.signup_date.min().date(), "->", riders.signup_date.max().date())

# --- key / foreign-key integrity
fk = pd.Series({
    "swaps.rider_id in riders":          swaps.rider_id.isin(riders.rider_id).mean(),
    "swaps.station_id in stations":      swaps.station_id.isin(stations.station_id).mean(),
    "swaps.battery_in_id in batteries":  swaps.battery_in_id.dropna().isin(batteries.battery_id).mean(),
    "swaps.battery_out_id in batteries": swaps.battery_out_id.dropna().isin(batteries.battery_id).mean(),
    "tickets.rider_id in riders":        tickets.rider_id.isin(riders.rider_id).mean(),
    "tickets.station_id in stations":    tickets.station_id.dropna().isin(stations.station_id).mean(),
    "tickets.battery_id in batteries":   tickets.battery_id.dropna().isin(batteries.battery_id).mean(),
    "riders.partner_id in partners":     riders.partner_id.dropna().isin(partners.partner_id).mean(),
    "hourly.station_id in stations":     hourly.station_id.isin(stations.station_id).mean(),
}).rename("share matching")
print("\nForeign-key integrity"); print(fk.to_frame().T.T.round(4).to_string())
print("\nrider_id unique:", riders.rider_id.is_unique, "| station_id unique:", stations.station_id.is_unique,
      "| battery_id unique:", batteries.battery_id.is_unique, "| hourly (station,hour) unique:",
      not hourly.duplicated(["station_id", "hour_start"]).any())

# --- which fields are populated for which event type (drives the revenue / battery analyses)
cols = ["battery_in_id", "battery_out_id", "soc_in_pct", "soc_out_pct", "km_since_last_swap",
        "list_price_inr", "amount_charged_inr", "energy_to_recharge_kwh"]
pop = swaps[cols].notna().groupby(swaps.event_type, observed=True).mean().round(3)
print("\nShare of rows with the field populated, by event_type"); print(pop.to_string())
print("\namount_charged_inr by event_type"); print(swaps.groupby("event_type", observed=True).amount_charged_inr.describe().round(1).to_string())
print("\ntariff_code x month-of-first-use (first date each tariff appears)")
print(swaps.groupby("tariff_code", observed=True).event_ts.agg(["min", "max", "size"]).to_string())
print("\nstation firmware x sync_mode"); print(pd.crosstab(swaps.station_firmware, swaps.sync_mode))

**Reading it.** The raw export has 3,877,013 attempts; 62,031 rows at the `STN-TST` test stations are removed, leaving 3,814,982 attempts (3,586,675 completed). Every foreign key matches 100 %, and all primary keys are unique.
Which fields exist for which outcome matters for every later calculation: `battery_out_id`, `soc_out_pct` and `energy_to_recharge_kwh` exist **only** for completed swaps; `battery_in_id` / `soc_in_pct` / `km_since_last_swap` exist for completed, failed-no-battery and abandoned attempts but not for rider cancellations or system errors; `amount_charged_inr` is exactly 0 for every non-completed outcome. So revenue, energy and wear are computed on completed swaps only, and failure rates on all attempts.
The tariff history shows the two pricing events directly: `STD` and `PARTNER` run for the whole period, `PEAK` / `OFFPEAK` first appear on 1 Oct 2024 (the pilot). `payment_mode` is constant in this export and is dropped. Firmware `v3.2.0` (the timestamp bug) runs at 1.56 M attempts, 14 % of them `offline_batch`.

## 3 · Cleaning
Every data-quality note from the problem statement is handled explicitly and logged in `CLEAN_LOG`:

| Issue | Handling | Outcome |
|---|---|---|
| Internal test stations `STN-TST*` | excluded from swaps, telemetry, tickets, stations | 62,031 swap rows removed |
| Inconsistent `riders.home_city` spellings | normalised via a mapping, cross-checked against the clean `home_zone` prefix | 585 rows changed; 100 % zone agreement |
| Firmware v3.2.0 timestamp bug (−5 h 30 min, 10 Mar–14 Apr 2025) | detected with an hour-of-day diagnostic, corrected **before** any time feature is derived | 141,280 rows shifted |
| Offline-sync near-duplicates | flagged on rider + station + packs + event type within `DUP_GAP_SEC`; looser key sets probed in 11.1 | none found (0 removed) |
| Odometer resets / SoC > 100 | `km_since_last_swap` outside `[0, KM_MAX]` → NaN; SoC/SoH clipped to 100 | 15,287 km values; ~3 k readings per SoC/SoH field |
| Non-completed attempts | revenue / cost / battery metrics use `swap_completed` only | – |
| Missing telemetry | never treated as zero; aggregations skip NaN; `telemetry_status` retained | 3.2 % partial/missing, concentrated at poor-connectivity stations |
| CSAT not missing at random | response rate shown by resolution time; never averaged naively | response rate 43 % for tickets resolved < 24 h vs 22 % beyond |
| Completed swaps logged on a pack **after** its retirement date | investigated and tested as a sensitivity in 11.3 | 45,813 swaps (1.3 %) |
| Battery wear cost | measured degradation per lot (3.12); flat 1,500-cycle model kept as sensitivity | headline model changed |


In [ ]:
# ---------- 3.1 test stations
test_ids = [s for s in stations.station_id if str(s).startswith("STN-TST")]
print("test stations:", test_ids)
if test_ids:
    ts_rows = swaps[swaps.station_id.isin(test_ids)]
    print(f"rows at test stations: {len(ts_rows):,} | share with amount==0: {(ts_rows.amount_charged_inr.fillna(0)==0).mean():.1%}")
    n0 = len(swaps)
    swaps   = swaps[~swaps.station_id.isin(test_ids)].reset_index(drop=True)
    hourly  = hourly[~hourly.station_id.isin(test_ids)].reset_index(drop=True)
    tickets = tickets[~tickets.station_id.isin(test_ids)].reset_index(drop=True)
    stations = stations[~stations.station_id.isin(test_ids)].reset_index(drop=True)
    log("exclude test stations", n0 - len(swaps), "swap rows removed")
swaps["station_id"]  = swaps["station_id"].cat.remove_unused_categories()
hourly["station_id"] = hourly["station_id"].cat.remove_unused_categories()

# ---------- 3.2 rider city standardisation (+ cross-check with zone prefix)
CITY_MAP = {"bengaluru": "Bengaluru", "bangalore": "Bengaluru", "blr": "Bengaluru",
            "delhi ncr": "Delhi NCR", "delhi": "Delhi NCR", "new delhi": "Delhi NCR", "gurgaon": "Delhi NCR", "gurugram": "Delhi NCR", "noida": "Delhi NCR", "del": "Delhi NCR",
            "hyderabad": "Hyderabad", "hyd": "Hyderabad",
            "pune": "Pune", "pun": "Pune",
            "mumbai": "Mumbai", "bombay": "Mumbai", "mum": "Mumbai",
            "jaipur": "Jaipur", "jai": "Jaipur"}
ZONE_PREFIX = {"BLR": "Bengaluru", "DEL": "Delhi NCR", "HYD": "Hyderabad", "PUN": "Pune", "MUM": "Mumbai", "JAI": "Jaipur"}
riders["home_city_std"] = riders.home_city.str.strip().str.lower().map(CITY_MAP)
riders["city_from_zone"] = riders.home_zone.str.split("-").str[0].map(ZONE_PREFIX)
print("unmapped home_city strings:", sorted(riders.loc[riders.home_city_std.isna(), "home_city"].unique()))
agree = (riders.home_city_std == riders.city_from_zone).mean()
print(f"home_city (standardised) agrees with home_zone prefix: {agree:.2%}")
riders["rider_city"] = riders.city_from_zone.fillna(riders.home_city_std)
log("standardise rider city", (riders.home_city != riders.home_city_std).sum(), f"rows whose raw spelling changed; zone-agreement {agree:.2%}")
stations["city"] = stations.city.str.strip()
print("station cities:", sorted(stations.city.unique()), "| rider cities:", sorted(riders.rider_city.dropna().unique()),
      "| city_ctx cities:", sorted(city_ctx.city.unique()))

# ---------- 3.3 rider helper attributes
partners_i = partners.set_index("partner_id")
riders["rider_group"] = np.where(riders.partner_id.notna(), "partner", "independent")
riders["surcharge_billable"] = riders.partner_id.map(partners_i.peak_surcharge_billable)
riders["seg"] = np.where(riders.partner_id.isna(), "independent",
                 np.where(riders.surcharge_billable == "Y", "partner_surcharge_Y", "partner_surcharge_N"))
riders["partner_key"] = riders.partner_id.fillna("Independent")
print(riders.seg.value_counts().to_string())

In [ ]:
# ---------- 3.4 firmware timestamp bug (v3.2.0, 10 Mar - 14 Apr 2025, logged 5h30 early)
FW_BUG = "v3.2.0"; SHIFT = pd.Timedelta(hours=5, minutes=30)
BUG_FROM = pd.Timestamp("2025-03-10 00:00") - SHIFT      # recorded (wrong) time of the first affected event
BUG_TO   = pd.Timestamp("2025-04-15 00:00") - SHIFT
swaps["event_ts_raw"] = swaps["event_ts"]
bug = (swaps.station_firmware == FW_BUG) & (swaps.event_ts >= BUG_FROM) & (swaps.event_ts < BUG_TO)
ref = (swaps.station_firmware != FW_BUG) & (swaps.event_ts >= pd.Timestamp("2025-03-10")) & (swaps.event_ts < pd.Timestamp("2025-04-15"))
prof = lambda ts: ts.dt.hour.value_counts(normalize=True).sort_index().reindex(range(24), fill_value=0)
p_ref, p_raw = prof(swaps.loc[ref, "event_ts"]), prof(swaps.loc[bug, "event_ts"])
p_fix = prof(swaps.loc[bug, "event_ts"] + SHIFT)
tv = lambda a, b: 0.5 * (a - b).abs().sum()
print(f"affected rows: {bug.sum():,}")
print(f"hour-of-day profile distance (total variation) vs unaffected firmware in same window:  raw={tv(p_raw,p_ref):.3f}  corrected={tv(p_fix,p_ref):.3f}")
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(p_ref.index, p_ref.values, "k-", lw=2, label="firmware v3.2.1, same window (reference)")
ax.plot(p_raw.index, p_raw.values, "r--", label="v3.2.0 as logged")
ax.plot(p_fix.index, p_fix.values, "g-", label="v3.2.0 after +5h30")
ax.set_xlabel("hour of day"); ax.set_ylabel("share of swaps"); ax.set_title("Timestamp bug: hour-of-day profile before / after correction"); ax.legend(); plt.show()
swaps.loc[bug, "event_ts"] = swaps.loc[bug, "event_ts"] + SHIFT
log("firmware timestamp fix", bug.sum(), "event_ts shifted +5h30 (v3.2.0, 10 Mar-14 Apr 2025)")
FIND["timestamp_bug_rows"] = int(bug.sum())
swaps.drop(columns="event_ts_raw", inplace=True); del bug, ref; gc.collect();

**Reading it.** The fix is validated. Compared with the unaffected firmware in the same window, the hour-of-day profile of the 141,280 `v3.2.0` rows is far off as logged (total-variation distance 0.499) and almost identical after the +5 h 30 min shift (0.017). Without the correction these rows would have inflated early-morning activity and distorted every hour-of-day view for 10 Mar–14 Apr 2025.

In [ ]:
# ---------- 3.5 near-duplicate offline-sync records
keys = ["rider_id", "station_id", "battery_in_id", "battery_out_id", "event_type"]
off = swaps.loc[swaps.sync_mode == "offline_batch", keys + ["event_ts"]].copy()
for c in keys: off[c] = off[c].cat.codes
off = off.sort_values(keys + ["event_ts"])
K = off[keys].values
same = np.r_[False, (K[1:] == K[:-1]).all(axis=1)]
tvals = off.event_ts.values
gap = np.r_[np.nan, (tvals[1:] - tvals[:-1]) / np.timedelta64(1, "s")]
has_batt = (off.battery_in_id.values >= 0)             # only trust the match when a returned pack is identified
cand = same & has_batt
print("same-key consecutive pairs (offline rows, pack identified):", cand.sum())
print("gap quantiles (s):", pd.Series(gap[cand]).quantile([.1, .5, .9, .99]).round(0).to_dict())
for thr in (60, 120, 180, 300, 600):
    print(f"  gap <= {thr:>3}s -> {(cand & (gap <= thr)).sum():,} flagged")
is_dup = cand & (gap <= CFG["DUP_GAP_SEC"])
dup_idx = off.index[is_dup]
share_off = len(dup_idx) / (swaps.sync_mode == "offline_batch").sum()
print(f"removing {len(dup_idx):,} duplicates = {len(dup_idx)/len(swaps):.2%} of all rows ({share_off:.1%} of offline_batch rows)")
dup_tier = swaps.loc[dup_idx, "station_id"].map(stations.set_index("station_id").connectivity_tier).value_counts(normalize=True)
print("duplicates by connectivity tier:", dup_tier.round(3).to_dict())
swaps = swaps.drop(index=dup_idx).reset_index(drop=True)
log("remove offline-sync duplicates", len(dup_idx), f"gap <= {CFG['DUP_GAP_SEC']}s, same rider/station/packs/type")
FIND["duplicates_removed"] = int(len(dup_idx)); del off, K, same, tvals, gap, cand, is_dup; gc.collect();

In [ ]:
# ---------- 3.6 outliers & impossible values
km = swaps.km_since_last_swap
print("km_since_last_swap quantiles:", km.quantile([0, .001, .01, .5, .99, .999, 1]).round(1).to_dict())
bad_km = (km < 0) | (km > CFG["KM_MAX"])
swaps.loc[bad_km, "km_since_last_swap"] = np.nan
log("invalid km_since_last_swap -> NaN", bad_km.sum(), f"<0 or >{CFG['KM_MAX']}")
for c in ["soc_in_pct", "soc_out_pct", "soh_in_pct", "soh_out_pct"]:
    n_over = int((swaps[c] > 100).sum()); n_neg = int((swaps[c] < 0).sum())
    swaps[c] = swaps[c].clip(lower=0, upper=100)
    log(f"clip {c} to [0,100]", n_over + n_neg, f">100: {n_over:,}, <0: {n_neg:,}")
print("queue_wait_sec quantiles:", swaps.queue_wait_sec.quantile([0, .5, .9, .99, .999, 1]).round(0).to_dict())

# price consistency on completed swaps (net = list - discount, surcharge handled separately)
dn = swaps.event_type == "swap_completed"
resid = (swaps.loc[dn, "amount_charged_inr"] - (swaps.loc[dn, "list_price_inr"] - swaps.loc[dn, "discount_inr"]))
print("\namount - (list - discount) on completed swaps, by tariff:")
print(resid.groupby(swaps.loc[dn, "tariff_code"], observed=True).describe().round(2).to_string())
print("completed swaps with amount == 0:", int((swaps.loc[dn, "amount_charged_inr"] == 0).sum()))
del dn, resid

In [ ]:
# ---------- 3.7 derived columns
swaps["is_done"]    = swaps.event_type == "swap_completed"
swaps["is_nobatt"]  = swaps.event_type == "failed_no_charged_battery"
swaps["is_abandon"] = swaps.event_type == "abandoned_queue"
swaps["is_cancel"]  = swaps.event_type == "cancelled_by_rider"
swaps["is_syserr"]  = swaps.event_type == "failed_system_error"
swaps["service_fail"] = swaps.is_nobatt | swaps.is_abandon | swaps.is_syserr

swaps["date"]  = swaps.event_ts.dt.normalize()
swaps["month"] = swaps.date.values.astype("datetime64[M]").astype("datetime64[ns]")
swaps["wk"]    = ((swaps.date - T0W).dt.days // 7).astype("int16")
swaps["hour"]  = swaps.event_ts.dt.hour.astype("int8")
swaps["dow"]   = swaps.event_ts.dt.dayofweek.astype("int8")
m_ = swaps.event_ts.dt.month
swaps["season"] = pd.Categorical(np.select([m_.isin([12, 1, 2]), m_.isin([3, 4, 5]), m_.isin([6, 7, 8, 9])],
                                           ["Winter", "Summer", "Monsoon"], "Post-monsoon"),
                                 categories=["Winter", "Summer", "Monsoon", "Post-monsoon"])
del m_

# ---------- 3.8 enrichment with station / rider attributes (map on categories = fast, no 3.9M-row joins)
def attach(df, key, lk_df, lk_key, cols):
    lk = lk_df.set_index(lk_key)
    for c in cols:
        out = df[key].map(lk[c])
        if out.dtype == object or str(out.dtype) == "category": out = out.astype("category")
        elif out.dtype == "float64": out = out.astype("float32")
        df[c] = out

ST_COLS = ["city", "zone", "location_type", "host_type", "charger_generation", "expansion_wave",
           "connectivity_tier", "commissioned_date", "grid_tariff_inr_kwh"]
RD_COLS = ["partner_id", "partner_key", "vehicle_class", "rider_city", "plan_type", "signup_channel", "signup_date",
           "declared_shift", "age_band", "kyc_verified", "rider_group", "seg"]
attach(swaps, "station_id", stations, "station_id", ST_COLS)
attach(swaps, "rider_id", riders, "rider_id", RD_COLS)
swaps["is_partner_rider"] = swaps.rider_group == "partner"
swaps["station_age_m"] = ((swaps.event_ts - swaps.commissioned_date).dt.days / 30.44).astype("float32")
print("riders without a profile row:", int(swaps.vehicle_class.isna().sum()))
print(swaps.memory_usage(deep=True).sum() / 1e9, "GB")

In [ ]:
# ---------- 3.9 unit-economics model (ASSUMPTIONS in CFG): per completed swap
#   revenue        = amount_charged_inr
#   energy_cost    = energy_to_recharge_kwh x station grid tariff
#   wear_cost      = FIRST-PASS flat model: purchase cost of the returned pack / CYCLE_LIFE. Superseded in 3.12 by measured degradation;
#                    the flat version is kept as wear_cost_flat (sensitivity)
#   cm_var         = revenue - energy_cost - wear_cost                (variable contribution)
#   fixed (rent+maintenance) is allocated per month in part 4 -> "loaded" contribution
bat_i = batteries.set_index("battery_id")
swaps["pack_cost_in"] = swaps.battery_in_id.map(bat_i.purchase_cost_inr).astype("float32")
med_cost = float(swaps.loc[swaps.is_done, "pack_cost_in"].median())
swaps["energy_cost"] = (swaps.energy_to_recharge_kwh * swaps.grid_tariff_inr_kwh).astype("float32")
med_energy = float(swaps.loc[swaps.is_done, "energy_cost"].median())
swaps["energy_cost"] = swaps.energy_cost.fillna(med_energy).where(swaps.is_done, 0).astype("float32")
swaps["wear_cost"]   = (swaps.pack_cost_in.fillna(med_cost) / CFG["CYCLE_LIFE"]).where(swaps.is_done, 0).astype("float32")
swaps["revenue"]     = swaps.amount_charged_inr.fillna(0).where(swaps.is_done, 0).astype("float32")
swaps["cm_var"]      = (swaps.revenue - swaps.energy_cost - swaps.wear_cost).astype("float32")
print(f"median pack cost {med_cost:,.0f} INR | median energy cost per swap {med_energy:.1f} INR")
print(swaps.loc[swaps.is_done, ["revenue", "energy_cost", "wear_cost", "cm_var"]].describe().round(2).to_string())

done = swaps[swaps.is_done].copy()            # completed swaps only
print(f"\ncompleted swaps: {len(done):,} of {len(swaps):,} attempts")
FIND["attempts_clean"] = len(swaps); FIND["completed_clean"] = len(done)

In [ ]:
# ---------- 3.10 hourly telemetry, tickets: enrichment and data-quality checks
HR_ATTRS = ["city", "zone", "location_type", "host_type", "charger_generation", "expansion_wave", "connectivity_tier"]
attach(hourly, "station_id", stations, "station_id", HR_ATTRS)
hourly["stockout2w"] = (hourly.charged_2w_min == 0).astype("float32").where(hourly.charged_2w_min.notna())
hourly["tele_ok"]    = (hourly.telemetry_status == "ok")
print("telemetry_status share:", hourly.telemetry_status.value_counts(normalize=True).round(3).to_dict())
print("telemetry_status by connectivity tier:"); print(pd.crosstab(hourly.connectivity_tier, hourly.telemetry_status, normalize="index").round(3).to_string())

attach(tickets, "station_id", stations, "station_id", HR_ATTRS)
tickets["rider_group"] = tickets.rider_id.map(riders.set_index("rider_id").rider_group)
print("\nticket category x resolution_status:"); print(pd.crosstab(tickets.category, tickets.resolution_status).to_string())

# CSAT is not missing at random -> show response rate and mean by resolution time
tickets["res_bucket"] = pd.cut(tickets.resolution_hours, [0, 4, 12, 24, 48, 1e6], labels=["<4h", "4-12h", "12-24h", "24-48h", ">48h"])
cs = tickets.groupby("res_bucket", observed=True).agg(tickets=("ticket_id", "size"), csat_response_rate=("csat_score", lambda s: s.notna().mean()),
                                                    csat_mean=("csat_score", "mean"))
print("\nCSAT response rate / mean by resolution time (naive mean is biased towards fast-resolved tickets):"); print(cs.round(3).to_string())
log("tickets: CSAT kept as-is, flagged biased", tickets.csat_score.isna().sum(), "NaN CSAT not imputed")

print("\n===== CLEANING LOG =====")
print(pd.DataFrame(CLEAN_LOG).to_string(index=False))

## 3.11–3.12 · Battery ledger and the measured-degradation wear model
The unit-economics model first used a flat cycle life (3.9). Pack degradation is observable, so the headline now uses **measured wear per swap by manufacturing lot**; the flat model stays as a sensitivity (`wear_cost_flat`). All margin figures from Part 4 onward use the measured model.

In [ ]:
# ---------- 3.11 battery ledger, degradation regression, supplier and lot scorecards (moved up from 7.1: the wear model in 3.12 needs it)
END = DATA_END.normalize()
bat = batteries.copy()
bat["end_date"] = bat.retired_date.fillna(END)
bat["age_months"] = ((bat.end_date - bat.commission_date).dt.days / 30.44).clip(lower=.5)
bat["soh_loss"] = bat.initial_soh_pct - bat.current_soh_pct
bat["soh_loss_per_month"] = bat.soh_loss / bat.age_months
cyc = done.battery_in_id.value_counts(); cyc.index = cyc.index.astype(str)
bat["cycles_obs"] = bat.battery_id.map(cyc).fillna(0)            # returns of this pack in completed swaps (Jan-24 on)
bat["months_obs"] = ((bat.end_date - bat.commission_date.clip(lower=DATA_START)).dt.days / 30.44).clip(lower=.5)
bat["swaps_per_month"] = bat.cycles_obs / bat.months_obs
bat["is3w"] = (bat.pack_type.str.startswith("3W")).astype(int)
bat["retired"] = bat.retired_date.notna()
bat["wear_inr_per_cycle"] = bat.purchase_cost_inr / CFG["CYCLE_LIFE"]

# degradation beyond what age + usage + pack type explain (residual of an OLS) -> highlights cohorts that are 'worse than expected'
Xr = np.column_stack([np.ones(len(bat)), bat.age_months, bat.cycles_obs, bat.is3w]); coef, *_ = np.linalg.lstsq(Xr, bat.soh_loss.values, rcond=None)
bat["soh_loss_resid"] = bat.soh_loss - Xr @ coef
print("OLS: soh_loss ~ 1 + age_months + cycles_obs + is3w  coef =", np.round(coef, 4))

sup_t = bat.groupby("supplier").agg(packs=("battery_id", "size"), first_commission=("commission_date", "min"), cost_mean=("purchase_cost_inr", "mean"),
                                    initial_soh=("initial_soh_pct", "mean"), current_soh=("current_soh_pct", "mean"), loss_per_month=("soh_loss_per_month", "mean"),
                                    loss_resid=("soh_loss_resid", "mean"), swaps_per_month=("swaps_per_month", "mean"), retired_pct=("retired", lambda s: s.mean() * 100))
print("Supplier scorecard"); print(sup_t.round(3).to_string())
print("\nSupplier x pack type"); print(bat.groupby(["supplier", "pack_type"]).agg(packs=("battery_id", "size"), loss_per_month=("soh_loss_per_month", "mean"), loss_resid=("soh_loss_resid", "mean"),
                                                                              current_soh=("current_soh_pct", "mean")).round(3).to_string())
print("\nRetirement reasons by supplier"); print(pd.crosstab(bat.supplier, bat.retirement_reason.fillna("in service")).to_string())
print("\nBMS firmware"); print(bat.groupby("bms_firmware").agg(packs=("battery_id", "size"), loss_per_month=("soh_loss_per_month", "mean"), loss_resid=("soh_loss_resid", "mean")).round(3).to_string())

lot = bat.groupby(["supplier", "manufacturing_lot"]).agg(packs=("battery_id", "size"), commission_med=("commission_date", "median"), loss_per_month=("soh_loss_per_month", "mean"),
                                                         loss_resid=("soh_loss_resid", "mean"), current_soh=("current_soh_pct", "mean"), retired_pct=("retired", lambda s: s.mean() * 100)).reset_index()
lot = lot[lot.packs >= 15].copy()
lot["z_resid"] = (lot.loss_resid - lot.loss_resid.mean()) / lot.loss_resid.std()
print("\nLots whose degradation is unusually high after controlling for age/usage (|z|>2):"); print(lot[lot.z_resid.abs() > 2].sort_values("z_resid", ascending=False).round(3).to_string(index=False))
FIND["Q4 supplier loss/month"] = sup_t.loss_per_month.round(3).to_dict(); FIND["Q4 lots flagged |z|>2"] = int((lot.z_resid.abs() > 2).sum())

In [ ]:
# ---------- 3.12 battery wear from MEASURED degradation (headline model; the flat 1,500-cycle model is kept as wear_cost_flat)
#   wear per swap = pack cost x (SoH points lost per swap cycle, by lot) / (usable SoH window = initial SoH - end-of-life SoH)
#   CAVEAT: only Kyron packs are retired, so the end-of-life SoH for Cellora / Amptek is an extrapolation -> sensitivity table in 11.3
retired = bat[bat.retired]
EOL_SOH = float(retired.current_soh_pct.median())
print(f"retired packs: {len(retired):,} | median SoH at retirement {EOL_SOH:.1f}% | by supplier:", retired.groupby('supplier').current_soh_pct.median().round(1).to_dict())

# degradation per cycle by lot (fallback supplier x pack type); only packs commissioned >= Jan-2024 have all cycles observed
b2 = bat[bat.commission_date >= DATA_START]
key_l, key_s = ["supplier", "manufacturing_lot", "pack_type"], ["supplier", "pack_type"]
gl = b2.groupby(key_l).agg(loss=("soh_loss", "sum"), cyc_lot=("cycles_obs", "sum")).reset_index(); gl["lpc_lot"] = gl.loss / gl.cyc_lot
gs = b2.groupby(key_s).agg(loss_s=("soh_loss", "sum"), cyc_s=("cycles_obs", "sum")).reset_index(); gs["lpc_sup"] = gs.loss_s / gs.cyc_s
bat2 = bat.drop(columns=[c for c in ["lpc_lot", "cyc_lot", "lpc_sup", "lpc", "usable_soh", "wear_per_swap", "implied_life_swaps"] if c in bat.columns])
bat2 = bat2.merge(gl[key_l + ["lpc_lot", "cyc_lot"]], on=key_l, how="left").merge(gs[key_s + ["lpc_sup"]], on=key_s, how="left")
bat2["lpc"] = np.where(bat2.cyc_lot.fillna(0) >= 3000, bat2.lpc_lot, bat2.lpc_sup)      # SoH points lost per swap cycle
bat2["usable_soh"] = (bat2.initial_soh_pct - EOL_SOH).clip(lower=5)
bat2["wear_per_swap"] = bat2.purchase_cost_inr * bat2.lpc / bat2.usable_soh                # cost x share of usable life consumed per swap
bat2["implied_life_swaps"] = bat2.usable_soh / bat2.lpc
bat_i2 = bat2.set_index("battery_id")
print("\nImplied pack life and wear cost per swap:")
print(bat2.groupby(["supplier", "pack_type"]).agg(packs=("battery_id", "size"), life_swaps=("implied_life_swaps", "median"), wear_per_swap=("wear_per_swap", "mean")).round(1).to_string())
print("\nWorst lots by wear per swap:"); print(bat2.groupby(["supplier", "manufacturing_lot"]).agg(packs=("battery_id", "size"), life_swaps=("implied_life_swaps", "median"), wear_per_swap=("wear_per_swap", "mean")).query("packs>=100").sort_values("wear_per_swap", ascending=False).head(6).round(1).to_string())
BAD_LOTS = lot.loc[lot.z_resid > 2, "manufacturing_lot"].tolist()
print("\nlots flagged by the degradation z-score (> 2):", BAD_LOTS)

# make the measured model the headline in swaps and done; keep the flat model as a sensitivity column
for df_ in (swaps, done): df_["wear_cost_flat"] = df_.wear_cost
w_new = swaps.battery_in_id.map(bat_i2.wear_per_swap).astype("float64")
w_new = w_new.fillna(w_new[swaps.is_done].median())
swaps["wear_cost"] = w_new.where(swaps.is_done, 0).astype("float32")
swaps["cm_var"]    = (swaps.revenue - swaps.energy_cost - swaps.wear_cost).astype("float32")
done["wear_cost"]  = swaps.loc[done.index, "wear_cost"]
done["cm_var"]     = swaps.loc[done.index, "cm_var"]
print(f"\nmean wear per completed swap: flat model INR {done.wear_cost_flat.mean():.1f} -> measured model INR {done.wear_cost.mean():.1f} | variable CM per swap {done.cm_var.mean():.2f}")
log("wear cost model", len(done), f"measured degradation (end-of-life SoH {EOL_SOH:.1f}%) replaces flat {CFG['CYCLE_LIFE']}-cycle assumption; flat kept as wear_cost_flat")
del retired, b2, gl, gs, w_new


# PART 4 · Q1 — Network performance over time
**Question:** how have completed swaps, revenue, failure rate and contribution margin per swap trended — do they tell the same story?

In [ ]:
# ---------- 4.1 detect the key events from the data itself (no hard-coded dates)
px = done.groupby(["date", "tariff_code"], observed=True).list_price_inr.median().unstack()
std = px["STD"].dropna()
lvl = std.reset_index(); lvl.columns = ["date", "price"]
print("STD list price levels:"); print(lvl.groupby("price").date.agg(["min", "max", "count"]).to_string())
chg = std[std.diff().fillna(0) != 0]
PRICE_CHANGE_DATE = chg.index[0] if len(chg) else None
print("-> base price change detected on:", PRICE_CHANGE_DATE)

pil = done.tariff_code.isin(["PEAK", "OFFPEAK"])
PILOT_START = done.loc[pil, "date"].min()
pil_share = pil.groupby(done.city, observed=True).mean()
PILOT_CITIES = pil_share[pil_share > 0.01].index.astype(str).tolist()
print("-> peak/off-peak pilot start:", PILOT_START, "| pilot cities:", PILOT_CITIES)
print(pil_share.round(4).to_string())

w = stations.groupby("expansion_wave").commissioned_date.agg(["count", "min", "median", "max"]); print("\n", w.to_string())
WAVE1 = w.loc[[i for i in w.index if "Wave1" in i], "min"].min() if any("Wave1" in i for i in w.index) else None
WAVE2 = w.loc[[i for i in w.index if "Wave2" in i], "min"].min() if any("Wave2" in i for i in w.index) else None
sup = batteries.groupby("supplier").commission_date.agg(["count", "min", "max"]).sort_values("min"); print("\n", sup.to_string())
NEW_SUPPLIER = sup.index[-1]; SUP_START = sup["min"].iloc[-1]
print("-> new supplier (latest first commission):", NEW_SUPPLIER, SUP_START.date())
amend = partners.dropna(subset=["amendment_date"])[["partner_id", "partner_name", "amendment_date", "discount_pct", "discount_pct_after_amendment"]]
print("\nPartner amendments:\n", amend.to_string(index=False))
EVENTS.update({"Base price change": PRICE_CHANGE_DATE, "Pricing pilot": PILOT_START, "Wave 1": WAVE1, "Wave 2": WAVE2,
               f"{NEW_SUPPLIER} supplier": SUP_START, "Partner amendment": amend.amendment_date.min() if len(amend) else None})

In [ ]:
# ---------- 4.2 monthly KPI table
g = swaps.groupby("month")
monthly = pd.DataFrame({
    "attempts": g.size(), "completed": g["is_done"].sum(), "nobatt": g["is_nobatt"].sum(), "abandon": g["is_abandon"].sum(),
    "cancel": g["is_cancel"].sum(), "syserr": g["is_syserr"].sum(),
    "queue_mean": g["queue_wait_sec"].mean(), "queue_p90": g["queue_wait_sec"].quantile(.9),
    "revenue": g["revenue"].sum(), "energy_cost": g["energy_cost"].sum(), "wear_cost": g["wear_cost"].sum(), "cm_var": g["cm_var"].sum(),
    "active_riders": g["rider_id"].nunique(),
})
# station fixed costs: every station that exists in the month carries rent + maintenance
stn_f = stations[["station_id", "commissioned_date", "decommissioned_date", "monthly_rent_inr", "monthly_maintenance_inr"]].copy()
stn_f["fixed"] = stn_f.monthly_rent_inr + stn_f.monthly_maintenance_inr
def _fixed(m):
    end = m + pd.offsets.MonthEnd(0)
    act = (stn_f.commissioned_date <= end) & (stn_f.decommissioned_date.isna() | (stn_f.decommissioned_date >= m))
    return pd.Series({"active_stations": int(act.sum()), "fixed_cost": float(stn_f.loc[act, "fixed"].sum())})
monthly = monthly.join(pd.DataFrame({m: _fixed(m) for m in monthly.index}).T)
monthly["svc_fails"] = monthly.nobatt + monthly.abandon + monthly.syserr
monthly["svc_fail_%"]  = monthly.svc_fails / monthly.attempts * 100
for c in ["nobatt", "abandon", "syserr", "cancel"]: monthly[f"{c}_%"] = monthly[c] / monthly.attempts * 100
monthly["rev_per_swap"]    = monthly.revenue / monthly.completed
monthly["energy_per_swap"] = monthly.energy_cost / monthly.completed
monthly["wear_per_swap"]   = monthly.wear_cost / monthly.completed
monthly["cm_var_per_swap"] = monthly.cm_var / monthly.completed
monthly["fixed_per_swap"]  = monthly.fixed_cost / monthly.completed
monthly["cm_loaded_per_swap"] = monthly.cm_var_per_swap - monthly.fixed_per_swap
monthly["swaps_per_station"]  = monthly.completed / monthly.active_stations
# sensitivity: the same loaded margin under the flat 1,500-cycle wear assumption
monthly["wear_flat_per_swap"] = g["wear_cost_flat"].sum() / monthly.completed
monthly["cm_loaded_flat_per_swap"] = monthly.rev_per_swap - monthly.energy_per_swap - monthly.wear_flat_per_swap - monthly.fixed_per_swap
# per-swap allocation of station rent + maintenance, used by the partner and station economics below
done["fixed_alloc"] = done.month.map(monthly.fixed_per_swap).astype("float32")
done["cm_loaded"]   = (done.cm_var - done.fixed_alloc).astype("float32")
print(monthly[["attempts", "completed", "svc_fail_%", "queue_mean", "revenue", "rev_per_swap", "cm_var_per_swap", "cm_loaded_per_swap", "cm_loaded_flat_per_swap",
               "active_stations", "swaps_per_station", "active_riders"]].round(2).to_string())

In [ ]:
# ---------- 4.3 the six trend views
fig, axes = plt.subplots(3, 2, figsize=(17, 14))
x = monthly.index
ax = axes[0, 0]; ax.bar(x, monthly.completed / 1e3, width=20, color="tab:blue", alpha=.7); ax.set_ylabel("completed swaps (000)")
a2 = ax.twinx(); a2.plot(x, monthly.revenue / 1e6, "o-", color="tab:green"); a2.set_ylabel("revenue (INR m)", color="tab:green"); a2.grid(False)
ax.set_title("Volume and revenue"); mark_events(ax)

ax = axes[0, 1]
for c, l in [("svc_fail_%", "all service failures"), ("nobatt_%", "no charged battery"), ("abandon_%", "abandoned in queue"), ("syserr_%", "system error")]:
    ax.plot(x, monthly[c], marker="o", ms=3, label=l, lw=2.5 if c == "svc_fail_%" else 1.3)
ax.set_ylabel("% of attempts"); ax.set_title("Service-failure rate by type"); ax.legend(fontsize=8); mark_events(ax)

ax = axes[1, 0]; ax.plot(x, monthly.queue_mean, "o-", label="mean"); ax.plot(x, monthly.queue_p90, "s--", label="p90")
ax.set_ylabel("seconds"); ax.set_title("Queue wait"); ax.legend(); mark_events(ax)

ax = axes[1, 1]
for c, l in [("rev_per_swap", "revenue"), ("energy_per_swap", "energy cost"), ("wear_per_swap", "battery wear"),
             ("cm_var_per_swap", "variable contribution"), ("cm_loaded_per_swap", "loaded contribution (after rent/maint.)")]:
    ax.plot(x, monthly[c], marker="o", ms=3, label=l)
ax.plot(x, monthly.cm_loaded_flat_per_swap, "k--", lw=1.2, label="loaded, flat 1,500-cycle wear (sensitivity)")
ax.axhline(0, color="k", lw=.8); ax.set_ylabel("INR per completed swap"); ax.set_title("Unit economics (measured-degradation wear)"); ax.legend(fontsize=8); mark_events(ax)

ax = axes[2, 0]; ax.plot(x, monthly.active_stations, "o-", label="active stations"); ax.set_ylabel("stations")
a2 = ax.twinx(); a2.plot(x, monthly.swaps_per_station, "s--", color="tab:red"); a2.set_ylabel("completed swaps / station / month", color="tab:red"); a2.grid(False)
ax.set_title("Footprint vs throughput per station"); mark_events(ax)

ax = axes[2, 1]
idx = pd.DataFrame({"completed swaps": monthly.completed, "revenue": monthly.revenue, "service failures (count)": monthly.svc_fails,
                    "variable contribution (INR)": monthly.cm_var, "active riders": monthly.active_riders})
(idx / idx.iloc[0] * 100).plot(ax=ax, marker="o", ms=3); ax.axhline(100, color="k", lw=.8)
ax.set_title("Index (first month = 100): do all metrics move together?"); ax.set_ylabel("index"); mark_events(ax)
plt.tight_layout(); plt.show()

In [ ]:
# ---------- 4.4 first quarter vs last quarter + divergence table
qa = monthly[["attempts", "completed", "nobatt", "abandon", "syserr", "cancel", "revenue", "energy_cost", "wear_cost", "cm_var", "fixed_cost", "svc_fails"]
            ].groupby(monthly.index.to_period("Q")).sum()
qa["svc_fail_%"] = qa.svc_fails / qa.attempts * 100
qa["rev_per_swap"] = qa.revenue / qa.completed
qa["cm_var_per_swap"] = qa.cm_var / qa.completed
qa["cm_loaded_per_swap"] = (qa.cm_var - qa.fixed_cost) / qa.completed
print(qa[["attempts", "completed", "svc_fail_%", "revenue", "rev_per_swap", "cm_var_per_swap", "cm_loaded_per_swap"]].round(2).to_string())

first, last = qa.iloc[0], qa.iloc[-1]
div = pd.DataFrame({"first_quarter": first[["completed", "revenue", "svc_fail_%", "rev_per_swap", "cm_var_per_swap", "cm_loaded_per_swap"]],
                    "last_quarter":  last [["completed", "revenue", "svc_fail_%", "rev_per_swap", "cm_var_per_swap", "cm_loaded_per_swap"]]})
div["change_%"] = (div.last_quarter / div.first_quarter - 1) * 100
print("\nFirst vs last quarter"); print(div.round(2).to_string())
for k in ["completed", "revenue", "cm_var_per_swap", "cm_loaded_per_swap"]:
    FIND[f"Q1 {k}: first->last quarter"] = f"{div.loc[k,'first_quarter']:.2f} -> {div.loc[k,'last_quarter']:.2f} ({div.loc[k,'change_%']:+.0f}%)"

# The first and last quarter are different seasons (Jan-Mar 2024 vs Apr-Jun 2025), so the change in failure rate above is NOT a trend.
# Like-for-like: same quarter, one year apart.
ss = pd.DataFrame({"2024": [qa.loc[pd.Period(f"2024Q{k}"), "svc_fail_%"] for k in (1, 2)],
                   "2025": [qa.loc[pd.Period(f"2025Q{k}"), "svc_fail_%"] for k in (1, 2)]}, index=["Q1 (Jan-Mar)", "Q2 (Apr-Jun, heat season)"])
ss["change_pp"] = ss["2025"] - ss["2024"]
print("\nService-failure % - same season, year over year (the correct comparison)"); print(ss.round(2).to_string())
FIND["Q1 svc_fail_% like-for-like (Q1 24->25 | Q2 24->25)"] = f"{ss.iloc[0,0]:.2f}->{ss.iloc[0,1]:.2f} | {ss.iloc[1,0]:.2f}->{ss.iloc[1,1]:.2f}"
FIND["Q1 svc_fail_% first->last quarter (DIFFERENT seasons - not a trend)"] = f"{div.loc['svc_fail_%','first_quarter']:.2f}->{div.loc['svc_fail_%','last_quarter']:.2f}"


**Reading it — do the four metrics tell the same story? No.**

* **Volume and revenue grew ~3×.** Completed swaps per quarter rose from 324 k (Q1-24) to 896 k (Q2-25), revenue from INR 19.4 m to 58.2 m; active riders per month from 4.2 k to 12.0 k; swaps per station per month from 1,150 to 2,035 (+77 %).
* **The "failure rate doubled" headline (3.98 % → 7.92 %) is a seasonal illusion.** It compares Jan–Mar 2024 with Apr–Jun 2025. Like for like: Q1 3.98 % → 3.89 %, Q2 9.52 % → **7.92 %** (better, not worse). Outside the Apr–Jun heat season the failure rate is flat at ≈ 3.9 % for all 18 months; the spike repeats every summer (May-24 11.9 %, May-25 9.5 %). Within a charger generation the heat-season rate did not change (Delhi Gen1 16.35 % → 16.64 %); the 2025 improvement is mix shift (Gen1's share of attempts fell 57.8 % → 38.2 % as Gen3 grew from 0 to 24.6 %, see 11.2).
* **Revenue per swap** rose INR 5 with the July 2024 base-price change (≈ 59.9 → 65.0) and has been flat since; it did not follow volume.
* **Margin per swap depends on how battery wear is measured.** On the measured-degradation model, *variable* contribution per swap is flat (INR 7.4 in Q1-24, 7.0 in Q2-25) because the price rise and falling energy cost per swap (21.9 → 17.5) are eaten by higher wear (30.6 → 40.3). *Loaded* contribution (after site rent and maintenance) improves from −12.1 to −4.9 per swap, almost entirely because fixed cost per swap fell (19.5 → 11.9) as volume grew — and stays negative. On the flat 1,500-cycle assumption the same quarters read −5.5 → +11.7, i.e. profitable; **the level of loaded margin is assumption-dependent, the movements are not**:
  * the **July 2024 price rise** lifts loaded margin by ≈ +7.3 per swap (Jun-24 −7.8 → Jul-24 −0.5);
  * the **Kyron lots KY-2407/08/09** then take it back: Aug-24 −0.8 → Sep-24 −7.3 (−6.5) while the share of swaps on those lots goes from 0.8 % to 21.4 %. Per-swap profitability did erode after the July peak, and it has recovered only partially (−4.1 in Jun-25) through scale and lower energy cost.
* Sensitivity to the end-of-life SoH and to flat-cycle alternatives is tested in 11.3.

# PART 5 · Q2 — Service failures & customer experience
**Question:** how do queue wait, failed attempts and abandoned swaps vary by station, hour, season and vehicle class — spread evenly or concentrated?

In [ ]:
by_hour = seg_metrics(swaps, "hour"); by_dow = seg_metrics(swaps, "dow"); by_veh = seg_metrics(swaps, "vehicle_class")
by_city = seg_metrics(swaps, "city"); by_season = seg_metrics(swaps, "season"); by_year_q = seg_metrics(swaps, swaps.month.dt.to_period("Q").rename("quarter"))
print("By vehicle class"); print(by_veh.round(2).to_string())
print("\nBy city");        print(by_city.round(2).to_string())
print("\nBy season");      print(by_season.round(2).to_string())

fig, axes = plt.subplots(2, 3, figsize=(18, 9))
ax = axes[0, 0]; by_hour["service_fail_%"].plot(ax=ax, marker="o", color="tab:red"); ax.set_title("Service-failure rate by hour of day (corrected timestamps)"); ax.set_ylabel("%")
a2 = ax.twinx(); by_hour["queue_mean_s"].plot(ax=a2, marker="s", ls="--", color="tab:blue"); a2.set_ylabel("mean queue (s)", color="tab:blue"); a2.grid(False)
hm = swaps.groupby(["dow", "hour"], observed=True).service_fail.mean().unstack() * 100
sns.heatmap(hm, ax=axes[0, 1], cmap="Reds", cbar_kws={"label": "%"}); axes[0, 1].set_title("Failure rate: weekday x hour (0 = Mon)")
by_veh[["no_batt_%", "abandon_%", "sys_err_%", "cancel_%"]].plot.bar(stacked=True, ax=axes[0, 2]); axes[0, 2].set_title("Failure mix by vehicle class"); axes[0, 2].set_xticklabels(by_veh.index, rotation=0)
hc = swaps.groupby(["city", "season"], observed=True).service_fail.mean().unstack() * 100
sns.heatmap(hc, annot=True, fmt=".1f", cmap="Reds", ax=axes[1, 0]); axes[1, 0].set_title("Failure % by city x season")
vh = swaps.groupby(["hour", "vehicle_class"], observed=True).service_fail.mean().unstack() * 100
vh.plot(ax=axes[1, 1], marker="o", ms=3); axes[1, 1].set_title("Failure % by hour, per vehicle class"); axes[1, 1].set_ylabel("%")
by_city[["no_batt_%", "abandon_%", "sys_err_%"]].plot.bar(stacked=True, ax=axes[1, 2]); axes[1, 2].set_title("Failure mix by city"); axes[1, 2].tick_params(axis="x", rotation=30)
plt.tight_layout(); plt.show()

In [ ]:
# ---------- 5.2 weather / calendar context (city-day grain, rate = sum(fails)/sum(attempts))
cd = swaps.groupby(["city", "date"], observed=True).agg(attempts=("is_done", "size"), fails=("service_fail", "sum"), nobatt=("is_nobatt", "sum"),
                                                        abandon=("is_abandon", "sum"), q_mean=("queue_wait_sec", "mean")).reset_index()
cd["city"] = cd.city.astype(str)
cx = city_ctx.copy(); cx["city"] = cx.city.astype(str)
cd = cd.merge(cx, on=["city", "date"], how="left")
cd["rain_bin"]   = pd.cut(cd.rainfall_mm, [-.1, .1, 5, 20, 50, 1e4], labels=["dry", "<5mm", "5-20", "20-50", "50+"])
cd["temp_bin"]   = pd.cut(cd.max_temp_c, [-50, 25, 32, 36, 40, 60], labels=["<25", "25-32", "32-36", "36-40", ">40"])
cd["outage_bin"] = pd.cut(cd.grid_outage_hours, [-.1, 0, .5, 1, 2, 24], labels=["0", "0-0.5h", "0.5-1h", "1-2h", ">2h"])
cd["festival"]   = cd.festival_or_event.notna()

def rate_by(col):
    return cd.groupby(col, observed=True).apply(lambda g: pd.Series({
        "city_days": len(g), "attempts": g.attempts.sum(), "svc_fail_%": g.fails.sum() / g.attempts.sum() * 100,
        "no_batt_%": g.nobatt.sum() / g.attempts.sum() * 100, "abandon_%": g.abandon.sum() / g.attempts.sum() * 100,
        "queue_s": np.average(g.q_mean, weights=g.attempts)}))
ctx_cols = ["rain_bin", "temp_bin", "outage_bin", "heat_alert", "flood_disruption", "festival", "is_public_holiday", "competitor_promo_active", "ecommerce_sale_event"]
fig, axes = plt.subplots(3, 3, figsize=(17, 11))
for a, c in zip(axes.ravel(), ctx_cols):
    t = rate_by(c); t["svc_fail_%"].plot.bar(ax=a, color="tab:red", alpha=.8); a.set_title(f"Failure % by {c}"); a.set_xlabel(""); a.tick_params(axis="x", rotation=0)
    print(f"\n{c}"); print(t.round(2).to_string())
plt.tight_layout(); plt.show()

# controlled view: do the context flags matter once city and month are held fixed?
try:
    import statsmodels.formula.api as smf
    d = cd.dropna(subset=["rainfall_mm", "max_temp_c", "grid_outage_hours"]).copy()
    d["rate"] = d.fails / d.attempts; d["ym"] = d.date.dt.to_period("M").astype(str)
    d["rain_mm10"] = d.rainfall_mm / 10; d["hot"] = (d.max_temp_c > 38).astype(int)
    for c in ["heat_alert", "flood_disruption", "is_public_holiday", "competitor_promo_active", "ecommerce_sale_event", "festival"]: d[c] = d[c].astype(int)
    mdl = smf.wls("rate ~ rain_mm10 + hot + grid_outage_hours + heat_alert + flood_disruption + is_public_holiday + festival + ecommerce_sale_event + C(city) + C(ym)",
                  d, weights=d.attempts).fit(cov_type="HC1")
    co = mdl.summary2().tables[1]; print("\nWLS on failure rate (city + month fixed effects):"); print(co.loc[~co.index.str.startswith("C("), ["Coef.", "P>|z|"]].round(4).to_string())
except Exception as e:
    print("statsmodels step skipped:", e)

In [ ]:
# ---------- 5.3 is service quality spread evenly? station-level concentration
stn_perf = seg_metrics(swaps, "station_id").reset_index()
stn_perf["station_id"] = stn_perf.station_id.astype(str)
st_cols = ["station_id", "city", "zone", "location_type", "host_type", "charger_generation", "expansion_wave", "connectivity_tier", "commissioned_date",
           "slots_2w", "slots_3w", "inventory_target_2w", "inventory_target_3w", "monthly_rent_inr", "monthly_maintenance_inr", "grid_tariff_inr_kwh",
           "firmware_version", "competitor_within_1_5km_since"]
stn_perf = stn_perf.merge(stations[st_cols], on="station_id", how="left")
p0 = swaps.service_fail.mean()
stn_perf["z_vs_network"] = (stn_perf["service_fail_%"] / 100 - p0) / np.sqrt(p0 * (1 - p0) / stn_perf.attempts)

s = stn_perf.sort_values("svc_fails", ascending=False).reset_index(drop=True)
cum_f = s.svc_fails.cumsum() / s.svc_fails.sum(); cum_att = s.attempts.cumsum() / s.attempts.sum()
k10 = int(np.ceil(.10 * len(s)))
print(f"top 10% of stations ({k10}) by failure count: {cum_f.iloc[k10-1]:.1%} of all service failures vs {cum_att.iloc[k10-1]:.1%} of all attempts")
print(f"stations with failure rate > 3 sigma above network: {(stn_perf.z_vs_network > 3).sum()} of {len(stn_perf)}")
print("failure-rate spread across stations (%):", stn_perf["service_fail_%"].describe().round(2).to_dict())
FIND["Q2 top10% stations share of failures"] = f"{cum_f.iloc[k10-1]:.1%} (share of attempts {cum_att.iloc[k10-1]:.1%})"

fig, axes = plt.subplots(1, 3, figsize=(18, 4.5))
axes[0].plot(np.arange(1, len(s) + 1) / len(s), cum_f, label="share of failures"); axes[0].plot(np.arange(1, len(s) + 1) / len(s), cum_att, label="share of attempts")
axes[0].plot([0, 1], [0, 1], "k:", lw=.8); axes[0].set_title("Concentration curve (stations sorted by failures)"); axes[0].legend()
sns.histplot(stn_perf["service_fail_%"], bins=30, ax=axes[1]); axes[1].set_title("Distribution of station failure rates (%)")
sns.scatterplot(data=stn_perf, x="attempts", y="service_fail_%", hue="charger_generation", style="connectivity_tier", ax=axes[2]); axes[2].set_title("Station failure % vs volume")
plt.tight_layout(); plt.show()
show = ["station_id", "city", "zone", "location_type", "charger_generation", "connectivity_tier", "expansion_wave", "attempts", "service_fail_%", "no_batt_%", "abandon_%", "sys_err_%", "queue_mean_s", "z_vs_network"]
print("\nWorst 15 stations"); print(stn_perf[stn_perf.attempts >= CFG["MIN_ATTEMPTS"]].nlargest(15, "service_fail_%")[show].round(2).to_string(index=False))
print("\nBest 10 stations");  print(stn_perf[stn_perf.attempts >= CFG["MIN_ATTEMPTS"]].nsmallest(10, "service_fail_%")[show].round(2).to_string(index=False))

In [ ]:
# ---------- 5.4 link failures to what the station actually had in stock (hourly telemetry)
def station_hour_table(df):
    hs = df.event_ts.dt.floor("h").rename("hour_start")
    t = df.groupby(["station_id", hs], observed=True).agg(attempts=("is_done", "size"), fails=("service_fail", "sum"), nobatt=("is_nobatt", "sum"),
                                                         abandons=("is_abandon", "sum"), q_mean=("queue_wait_sec", "mean")).reset_index()
    t["station_id"] = t.station_id.astype(str)
    return t
hrs = hourly[["station_id", "hour_start", "charged_2w_min", "charged_2w_avg", "charged_3w_min", "chargers_online", "outage_minutes",
              "telemetry_status", "cabinet_temp_c", "packs_quarantined"]].copy()
hrs["station_id"] = hrs.station_id.astype(str)

sh = station_hour_table(swaps).merge(hrs, on=["station_id", "hour_start"], how="left")
ok = sh[sh.charged_2w_min.notna()].copy()
ok["stock_bucket"] = pd.cut(ok.charged_2w_min, [-1, 0, 1, 2, 4, 1000], labels=["0 (stock-out)", "1", "2", "3-4", "5+"])
t = ok.groupby("stock_bucket", observed=True).agg(station_hours=("attempts", "size"), attempts=("attempts", "sum"), nobatt=("nobatt", "sum"), fails=("fails", "sum"))
t["no_batt_%"] = t.nobatt / t.attempts * 100; t["svc_fail_%"] = t.fails / t.attempts * 100
print("2W swaps: failure rate by the minimum number of charged 2W packs in the hour"); print(t.round(2).to_string())
so = ok.charged_2w_min == 0
print(f"\nshare of 'no charged battery' failures that happen in hours flagged as stock-out: {ok.loc[so,'nobatt'].sum()/ok.nobatt.sum():.1%} "
      f"(stock-out hours are {so.mean():.1%} of station-hours with attempts)")

# 3W riders against the 3W stock minimum
sh3 = station_hour_table(swaps[swaps.vehicle_class == "3W"]).merge(hrs, on=["station_id", "hour_start"], how="left")
o3 = sh3[sh3.charged_3w_min.notna()].copy(); o3["so3"] = o3.charged_3w_min == 0
print("\n3W attempts: no-battery rate when 3W stock-out vs not:",
      (o3.groupby("so3").nobatt.sum() / o3.groupby("so3").attempts.sum() * 100).round(2).to_dict())

# queue length -> abandonment
q = sh[sh.attempts >= 5].copy(); q["q_bucket"] = pd.cut(q.q_mean, [-1, 60, 120, 240, 480, 1e6], labels=["<1m", "1-2m", "2-4m", "4-8m", ">8m"])
tq = q.groupby("q_bucket", observed=True).agg(station_hours=("attempts", "size"), attempts=("attempts", "sum"), abandons=("abandons", "sum"))
tq["abandon_%"] = tq.abandons / tq.attempts * 100
print("\nAbandonment rate by mean queue wait in the station-hour"); print(tq.round(2).to_string())

fig, axes = plt.subplots(1, 3, figsize=(17, 4))
t["no_batt_%"].plot.bar(ax=axes[0], color="tab:red"); axes[0].set_title("No-battery failure % by charged-2W stock"); axes[0].tick_params(axis="x", rotation=0)
tq["abandon_%"].plot.bar(ax=axes[1], color="tab:orange"); axes[1].set_title("Abandon % by mean queue in the hour"); axes[1].tick_params(axis="x", rotation=0)
sh_h = ok.assign(h=ok.hour_start.dt.hour).groupby("h").agg(a=("attempts", "sum"), s=("charged_2w_min", lambda v: (v == 0).mean() * 100))
sh_h["s"].plot(ax=axes[2], marker="o"); axes[2].set_title("% station-hours in stock-out, by hour of day"); axes[2].set_xlabel("hour")
plt.tight_layout(); plt.show()

**Reading it — where are failures concentrated?**

* **Time: heat, not hour of day.** Summer (Mar–May) fails 6.9 % and monsoon (Jun–Sep, which includes June heat) 6.2 %, against 3.9 % in winter and post-monsoon. By day maximum temperature the rate is ≈ 3.9 % below 32 °C, 11.9 % at 36–40 °C and 14.4 % above 40 °C; heat-alert days run 14.5 % vs 5.1 %. Days above 36 °C are 17.2 % of attempts but 40.6 % of all failures; ≈ 59.8 k "excess" heat failures are 28.2 % of the total (11.2). With city and month fixed effects the hot-day indicator (> 38 °C) adds ≈ 7.7 pp (p < 0.001); rainfall, flood days, public holidays, festivals and e-commerce sale days show no effect (their lower raw failure rates just reflect that they fall in cool months). Grid-outage hours add a small amount (≈ 0.6 pp per hour).
* **Queue wait is not the differentiator.** Mean queue wait is ≈ 243 s in every segment (vehicle class, city, season). Abandonment instead tracks the queue *within* a station-hour: 0.7 % when the mean wait is 2–4 min, 3.0 % at 4–8 min, 16.1 % above 8 min.
* **Concentrated, not even.** The worst 10 % of stations (15) generate 24.7 % of failures from 17.2 % of attempts; 36 of 150 stations sit more than 3σ above the network rate, and the 15 worst (8.1–8.7 %) are all Gen1 sites in Delhi NCR and Jaipur. Station failure rates span 3.7 %–8.7 %.
* **Type of failure:** 63 % no charged battery, 32 % abandoned in queue, 5 % system error.
* **Vehicle class:** 3W riders fail 9.95 % vs 5.02 % for 2W, and the gap persists on cool days (8.65 % vs 3.41 %) and in every city. It does not follow demand per allocated 3W pack (Spearman −0.02), so it is not a simple capacity shortfall.
* **Stock-outs are only part of the picture.** No-battery failures are 7.3 % in hours when the charged-2W minimum is 0, against 3.3 % with ≥ 5 charged packs — yet only 3.2 % of all no-battery failures occur in telemetry-flagged stock-out hours.

# PART 6 · Q3 — Station & geographic patterns
**Question:** how do charger generation, location type, commissioning date and city relate to service, battery turnaround time and complaints?

In [ ]:
# ---------- 6.1 station scorecard = swap metrics + telemetry + tickets
hg = hourly.groupby("station_id", observed=True).agg(
    tele_ok_share=("tele_ok", "mean"), avg_charge_min=("avg_charge_minutes", "mean"), cabinet_temp=("cabinet_temp_c", "mean"), ambient_temp=("ambient_temp_c", "mean"),
    quarantined=("packs_quarantined", "mean"), chargers_online=("chargers_online", "mean"), outage_min_per_day=("outage_minutes", "mean"),
    stockout_hours_pct=("stockout2w", "mean"), charged_2w_avg=("charged_2w_avg", "mean"))
hg.index = hg.index.astype(str); hg["outage_min_per_day"] *= 24; hg["stockout_hours_pct"] *= 100
tk = tickets.dropna(subset=["station_id"]).groupby("station_id", observed=True).agg(tickets=("ticket_id", "size"), tickets_unresolved=("resolution_status", lambda s: (s == "unresolved").mean()))
tk.index = tk.index.astype(str)
scorecard = stn_perf.set_index("station_id").join(hg).join(tk)
scorecard["tickets_per_1k"] = scorecard.tickets / scorecard.attempts * 1000
scorecard["slots"] = scorecard.slots_2w + scorecard.slots_3w
scorecard["days_active"] = (DATA_END.normalize() - scorecard.commissioned_date.clip(lower=DATA_START)).dt.days + 1
scorecard["attempts_per_slot_day"] = scorecard.attempts / scorecard.days_active / scorecard.slots.replace(0, np.nan)
scorecard["chargers_vs_slots"] = scorecard.chargers_online / scorecard.slots.replace(0, np.nan)
num = ["service_fail_%", "no_batt_%", "abandon_%", "sys_err_%", "queue_mean_s", "avg_charge_min", "stockout_hours_pct", "quarantined", "outage_min_per_day",
       "cabinet_temp", "chargers_vs_slots", "attempts_per_slot_day", "tickets_per_1k", "tele_ok_share"]
fig, axes = plt.subplots(1, 2, figsize=(17, 6))
sns.heatmap(scorecard[num].corr(method="spearman"), annot=True, fmt=".2f", cmap="coolwarm", center=0, ax=axes[0], cbar=False); axes[0].set_title("Station-level Spearman correlations")
sns.scatterplot(data=scorecard.reset_index(), x="attempts_per_slot_day", y="service_fail_%", hue="charger_generation", size="slots", ax=axes[1]); axes[1].set_title("Utilisation vs failure rate")
plt.tight_layout(); plt.show()
print(scorecard[num].describe().round(2).T.to_string())

In [ ]:
# ---------- 6.2 by attribute: charger generation, location, host, wave, connectivity, city
def attr_table(attr):
    a = seg_metrics(swaps, attr)[["attempts", "service_fail_%", "no_batt_%", "abandon_%", "sys_err_%", "queue_mean_s", "queue_p90_s"]]
    h = hourly.groupby(attr, observed=True).agg(avg_charge_min=("avg_charge_minutes", "mean"), stockout_hours_pct=("stockout2w", "mean"),
                                                quarantined=("packs_quarantined", "mean"), cabinet_temp=("cabinet_temp_c", "mean"), outage_min_hr=("outage_minutes", "mean"))
    h["stockout_hours_pct"] *= 100; h["outage_min_per_day"] = h.pop("outage_min_hr") * 24
    t = tickets.groupby(attr, observed=True).size().rename("tickets")
    out = a.join(h).join(t); out["tickets_per_1k"] = out.tickets / out.attempts * 1000
    out["stations"] = stations.groupby(attr).size()
    return out

ATTRS = ["charger_generation", "location_type", "host_type", "expansion_wave", "connectivity_tier", "city"]
tabs = {a: attr_table(a) for a in ATTRS}
for a, t in tabs.items():
    print(f"\n=== {a}"); print(t.round(2).to_string())
fig, axes = plt.subplots(3, 6, figsize=(26, 11))
for j, a in enumerate(ATTRS):
    for i, (c, ttl) in enumerate([("service_fail_%", "service-failure %"), ("avg_charge_min", "avg charge minutes"), ("tickets_per_1k", "tickets per 1k attempts")]):
        tabs[a][c].plot.bar(ax=axes[i, j], color=["tab:red", "tab:purple", "tab:green"][i]); axes[i, j].set_title(f"{ttl}\nby {a}", fontsize=10); axes[i, j].set_xlabel(""); axes[i, j].tick_params(axis="x", rotation=30)
plt.tight_layout(); plt.show()
FIND["Q3 failure % by charger generation"] = tabs["charger_generation"]["service_fail_%"].round(2).to_dict()
FIND["Q3 avg charge minutes by charger generation"] = tabs["charger_generation"]["avg_charge_min"].round(1).to_dict()

In [ ]:
# ---------- 6.3 commissioning age (ramp-up / new-site effect) and thermal behaviour per charger generation
swaps["age_bucket"] = pd.cut(swaps.station_age_m, [-100, 1, 3, 6, 12, 1000], labels=["<1m", "1-3m", "3-6m", "6-12m", "12m+"])
ab = seg_metrics(swaps, "age_bucket"); print("Failure by station age at event time"); print(ab.round(2).to_string())
aw = swaps.groupby(["expansion_wave", "age_bucket"], observed=True).service_fail.mean().unstack() * 100
print("\nFailure % by wave x station age"); print(aw.round(2).to_string())

hourly["temp_bin"] = pd.cut(hourly.cabinet_temp_c, [-50, 20, 25, 30, 35, 40, 45, 100])
th = hourly.groupby(["charger_generation", "temp_bin"], observed=True).agg(avg_charge_min=("avg_charge_minutes", "mean"), quarantined=("packs_quarantined", "mean"),
                                                                           stockout_pct=("stockout2w", "mean"), n=("avg_charge_minutes", "size")).reset_index()
th["stockout_pct"] *= 100
fig, axes = plt.subplots(1, 4, figsize=(21, 4.2))
aw.T.plot(ax=axes[0], marker="o"); axes[0].set_title("Failure % by station age, per wave"); axes[0].set_ylabel("%")
for ax, c, ttl in zip(axes[1:], ["avg_charge_min", "quarantined", "stockout_pct"], ["avg charge minutes", "packs quarantined", "% hours in stock-out"]):
    p = th[th.n > 500].pivot(index="temp_bin", columns="charger_generation", values=c); p.plot(ax=ax, marker="o"); ax.set_title(f"{ttl} vs cabinet temperature"); ax.tick_params(axis="x", rotation=30)
plt.tight_layout(); plt.show()

In [ ]:
# ---------- 6.4 complaints: what do riders complain about, where, and how well is it resolved?
tk_all = tickets.copy()
for a in ["charger_generation", "city", "location_type"]:
    print(f"\nTicket category mix (% of tickets) by {a}"); print((pd.crosstab(tk_all[a], tk_all.category, normalize="index") * 100).round(1).to_string())
res = tk_all.groupby("category").agg(tickets=("ticket_id", "size"), median_res_h=("resolution_hours", "median"), unresolved=("resolution_status", lambda s: (s == "unresolved").mean() * 100),
                                     duplicate=("resolution_status", lambda s: (s == "duplicate").mean() * 100), csat_resp=("csat_score", lambda s: s.notna().mean() * 100))
print("\nResolution quality by category"); print(res.round(1).to_string())
tm = tk_all.groupby(tk_all.created_ts.dt.to_period("M")).agg(tickets=("ticket_id", "size"))
tm.index = tm.index.to_timestamp()                                   # month start, same as monthly.index
tm["completed_swaps"] = monthly.completed.reindex(tm.index)           # NaN before Jan-2024 (tickets start Mar-2023)
tm["tickets_per_1k_swaps"] = tm.tickets / tm.completed_swaps * 1000
fig, axes = plt.subplots(1, 2, figsize=(15, 4))
pd.crosstab(tk_all.created_ts.dt.to_period("M"), tk_all.category).plot.area(ax=axes[0], stacked=True, alpha=.85); axes[0].set_title("Tickets per month by category"); axes[0].legend(fontsize=7)
tm.loc[tm.completed_swaps.notna(), "tickets_per_1k_swaps"].plot(ax=axes[1], marker="o"); axes[1].set_title("Tickets per 1,000 completed swaps")
plt.tight_layout(); plt.show()

**Reading it — stations and geography.**

| | Gen1 | Gen2 | Gen3 |
|---|---|---|---|
| stations (of 150) | 51 | 62 | 37 |
| share of attempts | 48 % | 42 % | 10 % |
| service-failure rate | **6.80 %** | 4.42 % | 4.40 % |
| average charge time | **91.8 min** | 64.2 | 42.1 |
| stock-out hours | 2.34 % | 0.40 % | 0.26 % |
| average cabinet temperature | 24.9 °C | 24.6 °C | 28.3 °C |
| failure % at 36–38 °C / above 40 °C | **14.4 / 18.9** | 7.4 / 9.3 | 5.0 / 5.3 |

* Gen1 is **not** the hottest cabinet (Gen3 is); its distinguishing feature is slow charging, and it is by far the most heat-sensitive. The mechanism linking the two is **not proven**: station-level charge time and stock-out hours correlate (0.65), but only 3.2 % of no-battery failures fall in stock-out hours and Gen1's no-battery failure rate is still 4.1 % when ≥ 5 charged packs are available (Gen2/3: 2.7 %). Treat it as "Gen1 + heat, mechanism not fully identified".
* The generation gap holds **inside each city** (hot days: Gen1 12.7–18.2 % vs Gen3 5.0–5.5 %), so it is equipment, not just geography. The geographic gap is largely Gen1 mix: Jaipur (7.4 %), Delhi NCR (6.9 %) and Hyderabad (6.3 %) have 56 %, 43 % and 50 % Gen1 stations against 9 % in Mumbai (4.0 %), 18 % in Pune (4.6 %) and 28 % in Bengaluru (4.1 %).
* Location type matters little (4.7 %–5.8 %) and is confounded with generation. Younger stations look better (4.1–4.3 % under 3 months vs 6.1 % at 6–12 months), but that is a generation/mix effect: the newest sites are Gen2/Gen3.
* **Complaints:** "no battery available" is the largest category everywhere (30 % of Gen1 tickets); tickets per 1,000 swaps vary 7–16 by station. Resolution quality is uniform across categories (median ≈ 13 h, ≈ 12 % unresolved), so support does not differentiate by problem type. 21.6 % of all tickets are vehicle-performance comments filed as `other` (11.10).

# PART 7 · Q4 — Battery & equipment performance
**Question:** how do state of health, cycles, supplier and lot relate to delivered range and swap frequency? Which cohorts stand out?

> **7.1 (battery ledger, SoH-loss regression, supplier and lot scorecards)** is computed in section **3.11**, because the measured-degradation wear model in 3.12 and every unit-economics number from Part 4 onward depend on it. The cells below build on it.

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
sns.boxplot(data=bat, x="supplier", y="soh_loss_per_month", hue="pack_type", showfliers=False, ax=axes[0, 0]); axes[0, 0].set_title("SoH loss per month in service")
sns.scatterplot(data=bat.sample(min(3000, len(bat)), random_state=SEED), x="age_months", y="current_soh_pct", hue="supplier", s=14, alpha=.6, ax=axes[0, 1]); axes[0, 1].set_title("Current SoH vs age")
sns.scatterplot(data=bat.sample(min(3000, len(bat)), random_state=SEED), x="cycles_obs", y="soh_loss", hue="supplier", s=14, alpha=.6, ax=axes[0, 2]); axes[0, 2].set_title("SoH loss vs observed cycles")
ls = lot.sort_values("z_resid"); axes[1, 0].barh(ls.supplier + " " + ls.manufacturing_lot, ls.z_resid, color=np.where(ls.z_resid.abs() > 2, "tab:red", "lightgrey"))
axes[1, 0].set_title("Lot degradation z-score (after age/usage)"); axes[1, 0].tick_params(axis="y", labelsize=5)
sns.boxplot(data=bat, x="supplier", y="swaps_per_month", hue="pack_type", showfliers=False, ax=axes[1, 1]); axes[1, 1].set_title("Swaps per month in service")
cm_ = bat.groupby([bat.commission_date.dt.to_period("Q"), "supplier"]).size().unstack(fill_value=0); cm_.plot.bar(stacked=True, ax=axes[1, 2]); axes[1, 2].set_title("Packs commissioned per quarter")
plt.tight_layout(); plt.show()

In [ ]:
# ---------- 7.2 delivered range vs SoH (event level), and swap frequency vs battery condition (rider level)
r = done.loc[done.km_since_last_swap.notna() & done.battery_in_id.notna() & done.soc_in_pct.notna(),
             ["rider_id", "battery_in_id", "soc_in_pct", "soh_in_pct", "km_since_last_swap", "vehicle_class", "is_partner_rider", "month"]].copy()
r["used_soc"] = 100 - r.soc_in_pct
r = r[r.used_soc >= 30]                                          # avoid dividing by tiny discharge depths
r["km_per_soc_pct"] = r.km_since_last_swap / r.used_soc           # range proxy: km per 1% of charge consumed (previous pack issued ~full)
for c in ["supplier", "manufacturing_lot", "pack_type"]: r[c] = r.battery_in_id.map(bat_i[c]).astype(str)
r["soh_bin"] = pd.cut(r.soh_in_pct, [0, 70, 75, 80, 85, 90, 95, 101], labels=["<70", "70-75", "75-80", "80-85", "85-90", "90-95", "95+"])
print("Range proxy (km per 1% SoC) by SoH of the returned pack and vehicle class")
print(r.pivot_table(index="soh_bin", columns="vehicle_class", values="km_per_soc_pct", aggfunc="mean", observed=True).round(3).to_string())
print("\nBy supplier x SoH bin"); print(r.pivot_table(index="soh_bin", columns="supplier", values="km_per_soc_pct", aggfunc="mean", observed=True).round(3).to_string())

Xe = np.column_stack([np.ones(len(r)), r.soh_in_pct.values, (r.vehicle_class == "3W").astype(int).values, r.used_soc.values])
ce, *_ = np.linalg.lstsq(Xe, r.km_per_soc_pct.values, rcond=None)
r["range_resid"] = r.km_per_soc_pct - Xe @ ce
print("\nOLS km_per_soc_pct ~ SoH + 3W + depth: coef =", np.round(ce, 4))
rs = r.groupby("supplier").agg(swaps=("range_resid", "size"), range_resid=("range_resid", "mean")); print("\nRange residual by supplier (negative = shorter range than SoH explains)"); print(rs.round(4).to_string())
rl = r.groupby(["supplier", "manufacturing_lot"]).agg(swaps=("range_resid", "size"), range_resid=("range_resid", "mean")).reset_index(); rl = rl[rl.swaps >= 300]
print("\nLots with the lowest / highest range residual"); print(pd.concat([rl.nsmallest(5, "range_resid"), rl.nlargest(5, "range_resid")]).round(4).to_string(index=False))

rd = done.groupby("rider_id", observed=True).agg(n_swaps=("is_done", "size"), active_days=("date", "nunique"), first=("date", "min"), last=("date", "max"),
                                                 soh_out=("soh_out_pct", "mean"), km=("km_since_last_swap", "mean")).reset_index()
rd["rider_id"] = rd.rider_id.astype(str); rd = rd.merge(riders[["rider_id", "vehicle_class", "rider_group"]], on="rider_id")
rd = rd[rd.n_swaps >= 20].copy(); rd["swaps_per_active_day"] = rd.n_swaps / rd.active_days
rd["soh_q"] = rd.groupby("vehicle_class").soh_out.transform(lambda s: pd.qcut(s, 5, labels=False, duplicates="drop"))
print("\nRider swaps per active day by quintile of average SoH of the packs they received (0 = worst packs)")
print(rd.pivot_table(index="soh_q", columns="vehicle_class", values="swaps_per_active_day", aggfunc="mean").round(3).to_string())
fig, axes = plt.subplots(1, 3, figsize=(18, 4.2))
r.pivot_table(index="soh_bin", columns="vehicle_class", values="km_per_soc_pct", aggfunc="mean", observed=True).plot(ax=axes[0], marker="o"); axes[0].set_title("Range proxy vs SoH of returned pack")
r.pivot_table(index="soh_bin", columns="supplier", values="km_per_soc_pct", aggfunc="mean", observed=True).plot(ax=axes[1], marker="o"); axes[1].set_title("Range proxy vs SoH by supplier")
rd.pivot_table(index="soh_q", columns="vehicle_class", values="swaps_per_active_day", aggfunc="mean").plot(ax=axes[2], marker="o"); axes[2].set_title("Swaps per active day vs pack health received"); axes[2].set_xlabel("SoH quintile (0 = lowest)")
plt.tight_layout(); plt.show()

# complaints about batteries by cohort (tickets that reference a pack), per 1,000 swaps
tb = tickets.dropna(subset=["battery_id"]).copy()
tb["supplier"] = tb.battery_id.map(bat_i.supplier); tb["lot"] = tb.battery_id.map(bat_i.manufacturing_lot)
sw_by_sup = done.battery_in_id.map(bat_i.supplier).value_counts()
print("\nTickets referencing a pack, per 1,000 completed swaps, by supplier:"); print((tb.supplier.value_counts() / sw_by_sup * 1000).round(2).to_string())

**Reading it — batteries and equipment.**

* **(a) SoH explains delivered range for every supplier — with one exception.** Range per 1 % of charge climbs steadily with SoH (2W: 0.42 km below 70 % SoH → 0.75 km above 95 %). At the same SoH Kyron packs deliver ≈ 4–6 % less (e.g. 0.755 vs 0.80 km per 1 % at 95 %+); the range residual is −0.027 for Kyron vs +0.008 for the others, concentrated in lots KY-2407/08/09.
* **(b) Degradation is cohort-driven, not just age/usage-driven.** Kyron loses 4.9 SoH points per month vs 2.3 (Cellora) and 4.1 (Amptek); after controlling for age, usage and pack type Kyron is +4.5 points worse than expected, Amptek −2.0, Cellora −1.4. Three lots stand out (z = 3.4–3.6): KY-2407, KY-2408, KY-2409 — 1,461 packs, 1,438 of them already retired at end of life (none of the other suppliers' packs has been). Implied life is ≈ 511 swaps for these 2W packs vs ≈ 1,217 for Cellora/Amptek; wear cost per swap INR 62.6 vs 26.3. Replacement cost already incurred: **INR 47.9 m, 20.9 % of period revenue**. BMS firmware shows no effect.
* **(c) A hidden service cost.** Riders who receive weaker packs swap more often per active day (2W: 1.42 in the worst-SoH quintile vs 1.30 in the best; 3W 1.17 vs 1.14).
* **(d) The new supplier shows up in tickets:** 10.7 pack-related tickets per 1,000 swaps for Kyron vs 7.8–7.9; battery-type complaints per 1,000 swaps are 5.0–5.3 in the three Kyron lots vs ≈ 3.1 next-worst. Riders on Kyron packs report dying at a median 45 km vs 54 km on others.
* `cycles_obs` only counts cycles from Jan 2024. 45,813 completed swaps (1.3 %) are logged on packs after their retirement date — investigated in 11.3.

# PART 8 · Q5 — Pricing & partner economics
**Question:** how do the base price change, the peak/off-peak pilot and partner contract terms relate to swap timing, revenue and contribution margin — and do they hit every rider segment the same way?

In [ ]:
# ---------- 8.1 tariff mix and effective prices
mix = pd.crosstab(done.month, done.tariff_code, normalize="index") * 100
lp = done.groupby(["month", "tariff_code"], observed=True).list_price_inr.mean().unstack()
disc = done.groupby(["month", "rider_group"], observed=True).apply(lambda g: g.discount_inr.sum() / g.list_price_inr.sum() * 100).unstack()
rev_g = done.groupby(["month", "rider_group"], observed=True).agg(rev=("revenue", "mean"), cm=("cm_var", "mean")).unstack()
fig, axes = plt.subplots(1, 4, figsize=(22, 4.2))
mix.plot.area(ax=axes[0], alpha=.85); axes[0].set_title("Tariff mix (% of completed swaps)"); mark_events(axes[0])
lp.plot(ax=axes[1], marker="o", ms=3); axes[1].set_title("Average list price by tariff"); mark_events(axes[1])
disc.plot(ax=axes[2], marker="o", ms=3); axes[2].set_title("Effective discount % of list"); mark_events(axes[2])
rev_g["cm"].plot(ax=axes[3], marker="o", ms=3); axes[3].set_title("Variable contribution per swap (INR)"); mark_events(axes[3])
plt.tight_layout(); plt.show()
print(lp.round(1).iloc[[0, 6, 12, -1]].to_string())

In [ ]:
# ---------- 8.2 rider-week table + event-study helper
codes = done.rider_id.cat.codes.values
rw = pd.DataFrame({"rc": codes, "wk": done.wk.values}).groupby(["rc", "wk"]).size().rename("n").reset_index()
rw["rider_id"] = np.asarray(done.rider_id.cat.categories)[rw.rc]; rw.drop(columns="rc", inplace=True)
rw = rw.merge(riders[["rider_id", "rider_group", "seg", "plan_type", "rider_city", "vehicle_class", "partner_key"]], on="rider_id", how="left")
LAST_WK = int(rw.wk.max())
print("rider-week rows:", len(rw))

def event_study(event_date, group_col, pre=8, post=8, lookback=4, subset=None):
    """Riders active in the `lookback` weeks before the event; weekly swaps per cohort rider, indexed to the pre-period (=1)."""
    we = int((pd.Timestamp(event_date) - T0W).days // 7)
    d = rw if subset is None else rw[subset(rw)]
    coh = d[(d.wk >= we - lookback) & (d.wk < we)].rider_id.unique()
    sub = d[d.rider_id.isin(coh) & d.wk.between(we - pre, we + post - 1)].copy(); sub["rel"] = sub.wk - we
    size = riders[riders.rider_id.isin(coh)].groupby(group_col).size()
    tot = sub.groupby([group_col, "rel"]).n.sum().unstack(0)
    per = tot / size
    base = per.loc[-lookback:-1].mean()
    return (per / base).reindex(range(-pre, post)), size

if PRICE_CHANGE_DATE is not None:
    fig, axes = plt.subplots(1, 3, figsize=(18, 4.2))
    for ax, gcol in zip(axes, ["rider_group", "plan_type", "rider_city"]):
        idx, size = event_study(PRICE_CHANGE_DATE, gcol); idx.plot(ax=ax, marker="o"); ax.axvline(0, color="k", ls="--"); ax.axhline(1, color="grey", lw=.8)
        ax.set_title(f"Base price change: weekly swaps per cohort rider (pre=1), by {gcol}"); ax.set_xlabel("weeks from price change")
        print(f"\n{gcol} cohort sizes:", size.to_dict()); print(idx.loc[[-2, -1, 0, 1, 2, 4, 7]].round(3).to_string())
    plt.tight_layout(); plt.show()
else:
    print("No base price change detected - skip")

**Reading it — the July 2024 base price change.** Partner-billed riders do not see the price (their employer pays), so they are a natural comparison group for independent riders. Both groups show the same profile after the change (activity index 7 weeks out: independent 0.917, partner-billed 0.902; pay-as-you-go 0.917, prepaid 0.917), and the gap between groups is small. There is **no evidence that the +INR 5 price rise reduced activity beyond the common pattern** (a cohort defined by prior activity regresses to the mean in both groups, so read the gap, not the absolute drop). Revenue per swap rose by the full INR 5 and loaded margin by ≈ 7.3.

In [ ]:
# ---------- 8.3 peak / off-peak pilot: did swaps move in time? (difference-in-differences)
pl = done[(done.city.isin(PILOT_CITIES)) & (done.date >= PILOT_START) & (done.seg == "independent")]
tc = pd.crosstab(pl.hour, pl.tariff_code, normalize="index")
print("Tariff mix by hour in pilot cities (independent riders, after pilot start):"); print((tc * 100).round(1).T.to_string())
PEAK_HOURS = tc.index[tc.get("PEAK", 0) > 0.5].tolist() if "PEAK" in tc else []
OFFPEAK_HOURS = tc.index[tc.get("OFFPEAK", 0) > 0.5].tolist() if "OFFPEAK" in tc else []
print("PEAK hours:", PEAK_HOURS, "| OFFPEAK hours:", OFFPEAK_HOURS)
if not PEAK_HOURS:
    PEAK_HOURS = swaps.groupby("hour").size().nlargest(4).index.tolist(); print("fallback: busiest hours", PEAK_HOURS)
done["is_peak_hour"] = done.hour.isin(PEAK_HOURS)
done["is_offpeak_hour"] = done.hour.isin(OFFPEAK_HOURS)
swaps["is_peak_hour"] = swaps.hour.isin(PEAK_HOURS)

def did(df, metric, start, weeks=12):
    d = df[(df.date >= start - pd.Timedelta(weeks=weeks)) & (df.date < start + pd.Timedelta(weeks=weeks))].copy()
    d["post"] = d.date >= start; d["pilot"] = d.city.isin(PILOT_CITIES)
    t = d.groupby(["seg", "pilot", "post"], observed=True)[metric].mean().unstack("post"); t.columns = ["pre", "post"]
    t["delta"] = t.post - t.pre
    o = t["delta"].unstack("pilot"); o.columns = ["control_delta", "pilot_delta"]; o["DiD"] = o.pilot_delta - o.control_delta
    return o

print("\nDiD (12 weeks before vs 12 weeks after pilot start; pilot vs non-pilot cities), by rider segment")
for metric, lab in [("is_peak_hour", "share of swaps in PEAK hours"), ("is_offpeak_hour", "share in OFF-PEAK hours"), ("revenue", "revenue per completed swap (INR)"), ("cm_var", "variable contribution per swap (INR)")]:
    print(f"\n-- {lab}"); print(did(done, metric, PILOT_START).round(4).to_string())
print("\n-- service-failure rate (all attempts)"); print((did(swaps, "service_fail", PILOT_START) * 100).round(3).to_string())

wkly = done.groupby(["wk", done.city.isin(PILOT_CITIES).rename("pilot_city"), "seg"], observed=True).is_peak_hour.mean().reset_index()
wkly["date"] = T0W + pd.to_timedelta(wkly.wk * 7, unit="D")
fig, axes = plt.subplots(1, 3, figsize=(19, 4.2), sharey=True)
for ax, sg in zip(axes, ["independent", "partner_surcharge_Y", "partner_surcharge_N"]):
    for pc, lab in [(True, "pilot cities"), (False, "other cities")]:
        s_ = wkly[(wkly.seg == sg) & (wkly.pilot_city == pc)]; ax.plot(s_.date, s_.is_peak_hour * 100, label=lab)
    ax.axvline(PILOT_START, color="k", ls="--"); ax.set_title(f"Share of swaps in peak hours - {sg}"); ax.legend()
plt.tight_layout(); plt.show()

try:
    import statsmodels.formula.api as smf
    cw = done[done.seg == "independent"].groupby(["city", "wk"], observed=True).is_peak_hour.mean().reset_index()
    cw["city"] = cw.city.astype(str); cw["pilot"] = cw.city.isin(PILOT_CITIES).astype(int); cw["post"] = (cw.wk >= int((PILOT_START - T0W).days // 7)).astype(int)
    m_did = smf.ols("is_peak_hour ~ pilot:post + C(city) + C(wk)", cw).fit(cov_type="HC1")
    print("\nTwo-way FE DiD on weekly peak share (independent riders): effect of pilot =", round(m_did.params["pilot:post"], 4), "| p =", round(m_did.pvalues["pilot:post"], 4))
    FIND["Q5 pilot effect on peak share (independents, DiD)"] = round(float(m_did.params["pilot:post"]), 4)
except Exception as e:
    print("statsmodels DiD skipped:", e)

In [ ]:
# ---------- 8.4 partner economics: value after energy, MEASURED battery wear and station costs (single partner table; reused in 11.6)
done["fixed_alloc"] = done.month.map(monthly.fixed_per_swap).astype("float32")
BUSY_HOURS = swaps.groupby("hour").size().nlargest(5).index.tolist()
done["in_busy"] = done.hour.isin(BUSY_HOURS)
g = done.groupby("partner_key", observed=True)
pt = pd.DataFrame({"swaps": g.size(), "riders": g["rider_id"].nunique(), "revenue": g["revenue"].sum(), "list_value": g["list_price_inr"].sum(),
                   "discount": g["discount_inr"].sum(), "energy": g["energy_cost"].sum(), "wear": g["wear_cost"].sum(), "fixed_alloc": g["fixed_alloc"].sum(),
                   "cm_var": g["cm_var"].sum(), "busy_hour_share": g["in_busy"].mean()})
pt["eff_discount_%"] = pt.discount / pt.list_value * 100
for c in ["revenue", "energy", "wear", "cm_var"]: pt[f"{c}_per_swap"] = pt[c] / pt.swaps
pt["cm_loaded"] = pt.cm_var - pt.fixed_alloc; pt["cm_loaded_per_swap"] = pt.cm_loaded / pt.swaps
pt["share_swaps_%"] = pt.swaps / pt.swaps.sum() * 100
neg_ = pt.cm_loaded.clip(upper=0); pt["share_of_total_loss_%"] = neg_ / neg_.sum() * 100        # share of all losses made by loss-making partners (the total itself is negative)
pt["swaps_per_rider"] = pt.swaps / pt.riders
sv = seg_metrics(swaps, "partner_key")[["service_fail_%", "queue_mean_s"]]; pt = pt.join(sv)
meta = partners.set_index("partner_id")[["partner_name", "partner_segment", "vehicle_class", "contract_type", "discount_pct", "discount_pct_after_amendment", "peak_surcharge_billable", "payment_terms_days"]]
pt = pt.join(meta)
pt["list_per_swap"] = pt.list_value / pt.swaps
pt["var_break_even_disc_%"] = (1 - (pt.energy_per_swap + pt.wear_per_swap) / pt.list_per_swap) * 100                        # discount at which VARIABLE margin = 0
pt["break_even_disc_%"]     = (1 - (pt.energy_per_swap + pt.wear_per_swap + pt.fixed_alloc / pt.swaps) / pt.list_per_swap) * 100  # incl. site cost
pt["headroom_var_pp"] = pt["var_break_even_disc_%"] - pt["eff_discount_%"]
cols = ["partner_name", "partner_segment", "contract_type", "discount_pct", "discount_pct_after_amendment", "peak_surcharge_billable", "riders", "swaps", "share_swaps_%", "swaps_per_rider",
        "eff_discount_%", "revenue_per_swap", "energy_per_swap", "wear_per_swap", "cm_var_per_swap", "cm_loaded_per_swap", "cm_loaded", "share_of_total_loss_%", "var_break_even_disc_%", "headroom_var_pp", "busy_hour_share", "service_fail_%"]
print(pt.sort_values("swaps", ascending=False)[cols].round(2).to_string())
largest = pt.drop(index="Independent").swaps.idxmax()
print(f"\nLargest partner by swaps: {largest} ({pt.loc[largest,'partner_name']}) | rank by loaded CM per swap among partners: "
      f"{int(pt.drop(index='Independent').cm_loaded_per_swap.rank(ascending=False)[largest])} of {len(pt)-1} | rank by total loaded CM: {int(pt.drop(index='Independent').cm_loaded.rank(ascending=False)[largest])}")
FIND["Q5 largest partner"] = f"{largest} {pt.loc[largest,'partner_name']}: loaded CM/swap {pt.loc[largest,'cm_loaded_per_swap']:.1f} vs network {pt.cm_loaded.sum()/pt.swaps.sum():.1f}"

fig, axes = plt.subplots(1, 3, figsize=(20, 5)); p = pt.sort_values("cm_loaded_per_swap")
axes[0].barh(p.index.astype(str) + " " + p.partner_name.fillna(""), p.cm_loaded_per_swap, color=np.where(p.cm_loaded_per_swap < 0, "tab:red", "tab:green")); axes[0].set_title("Loaded contribution per swap (INR)")
axes[1].scatter(pt["eff_discount_%"], pt.cm_var_per_swap, s=pt.swaps / pt.swaps.max() * 800 + 20, alpha=.6)
for i, rr in pt.iterrows(): axes[1].annotate(str(i), (rr["eff_discount_%"], rr.cm_var_per_swap), fontsize=8)
axes[1].set_xlabel("effective discount %"); axes[1].set_ylabel("variable CM per swap"); axes[1].set_title("Discount vs margin (bubble = volume)")
pt[["revenue_per_swap", "energy_per_swap", "wear_per_swap"]].plot.bar(ax=axes[2]); axes[2].set_title("Revenue vs cost per swap, by partner"); axes[2].tick_params(axis="x", rotation=45)
plt.tight_layout(); plt.show()

In [ ]:
# ---------- 8.5 renegotiated contract: before / after the amendment
for _, a in amend.iterrows():
    pid, d0 = a.partner_id, a.amendment_date
    x = done[done.partner_key == pid]
    m = x.groupby("month").agg(swaps=("is_done", "size"), rev=("revenue", "mean"), cm=("cm_var", "mean"), disc=("discount_inr", "sum"), lst=("list_price_inr", "sum"))
    m["eff_disc_%"] = m.disc / m.lst * 100
    print(f"\n=== {pid} {partners_i.loc[pid,'partner_name']} | amendment {d0.date()} | contract {a.discount_pct}% -> {a.discount_pct_after_amendment}%")
    print(m[["swaps", "rev", "cm", "eff_disc_%"]].round(2).to_string())
    pre, post = x[(x.date >= d0 - pd.Timedelta(days=90)) & (x.date < d0)], x[(x.date >= d0) & (x.date < d0 + pd.Timedelta(days=90))]
    print(f"90d pre vs 90d post: swaps/day {len(pre)/90:.0f} -> {len(post)/90:.0f}; revenue per swap {pre.revenue.mean():.1f} -> {post.revenue.mean():.1f}; "
          f"variable CM per swap {pre.cm_var.mean():.1f} -> {post.cm_var.mean():.1f}; total CM per day {pre.cm_var.sum()/90:,.0f} -> {post.cm_var.sum()/90:,.0f}")
    fig, ax = plt.subplots(1, 2, figsize=(14, 3.8))
    m.swaps.plot(ax=ax[0], marker="o"); ax[0].axvline(d0, color="k", ls="--"); ax[0].set_title(f"{pid} monthly completed swaps")
    m["eff_disc_%"].plot(ax=ax[1], marker="o", color="tab:red"); ax[1].axvline(d0, color="k", ls="--"); ax[1].set_title(f"{pid} effective discount %")
    plt.tight_layout(); plt.show()

**Reading it — pricing and partner economics.**

* **Base price (+INR 5, Jul 2024):** passed through in full to revenue per swap, no visible volume response (8.2).
* **Peak/off-peak pilot (Bengaluru + Pune from 1 Oct 2024; peak +INR 15, off-peak −INR 8).** Riders did shift: the peak-hour share of independent swaps fell 2.6 pp (DiD, p < 0.001). Revenue per swap rose INR 7.8 for independents and INR 5.9 for partners that are billed the surcharge, and **0.2 for partners that are not** — so segments are not affected equally. But independents made **0.56 fewer swaps per week (−10 %, p < 0.001)** while revenue per rider-week rose only INR 5.6 (p = 0.155, not significant). Only two cities were treated (two of the three lowest-failure cities), so the result may not generalise. The per-swap effect overstates the commercial case (11.7).
* **Largest partner ≠ most valuable.** ZipDrop (FP-03) is 15.3 % of all swaps, the **least valuable 2W partner per swap and the biggest loss in total**: loaded contribution −INR 16.5 per swap, ≈ −INR 9.1 m over the period. Its discount went from 12 % to 28 % on 1 Nov 2024 (effective 21.9 % over the whole period); the deeper discount bought only +2.0 % incremental volume (network share 15.39 % → 15.69 %). Variable break-even discount is **17.2 %** (energy + wear only); the loaded break-even is below zero, so no discount level covers site cost on these swaps. Moving 28 % → 12 % is worth ≈ INR 10.7 per swap ≈ INR 1.2 m per quarter at current volume (ignoring any volume loss).
* **On measured wear every partner is loss-making after site cost** (−INR 3.9 to −19.9 per swap); only independent riders are positive (+0.6). 3W partners (CargoTuk, HaulKing) are the worst (−19 to −20) because wear and energy per swap are ≈ 65 and 41 against ≈ 100 of revenue. Partner swaps are concentrated in busy hours (busy-hour share 0.66 for food-delivery partners vs 0.17–0.26 for cargo, e-commerce and bike-taxi partners).

# PART 9 · Q6 — Root cause of new-rider retention
**Design.** New riders = signed up on/after 1 Jan 2024 and made ≥ 1 attempt. Outcome `retained` = at least one attempt between day `RET_FROM` and `RET_TO` after the first attempt (only riders observable for the whole window). Features come **only from the first `EARLY_DAYS` days**, so they precede the outcome. Evidence is built in layers: (1) univariate retention gaps → (2) survival-style curves → (3) multivariate model with **group permutation importance** → (4) stratified checks to separate primary from secondary drivers and confounders (partner vs independent).

In [ ]:
# ---------- 9.1 cohort definition
new_ids = riders.loc[riders.signup_date >= DATA_START, "rider_id"]
first_ts = swaps.groupby("rider_id", observed=True).event_ts.min(); first_ts.index = first_ts.index.astype(str)
coh = riders[riders.rider_id.isin(new_ids)].copy()
coh["first_ts"] = coh.rider_id.map(first_ts)
print(f"new riders (signup >= {DATA_START.date()}): {len(coh):,} | with at least one attempt: {coh.first_ts.notna().sum():,}")
coh = coh.dropna(subset=["first_ts"]).copy()
coh["observable"] = coh.first_ts + pd.Timedelta(days=CFG["RET_TO"]) <= DATA_END
coh["days_signup_to_first"] = (coh.first_ts.dt.normalize() - coh.signup_date).dt.days

mask_new = swaps.rider_id.isin(coh.rider_id)
ev = swaps.loc[mask_new, ["rider_id", "event_ts", "is_done", "service_fail", "is_nobatt", "is_abandon", "is_syserr", "queue_wait_sec", "station_id", "list_price_inr",
                          "discount_inr", "amount_charged_inr", "soh_out_pct", "km_since_last_swap", "hour", "charger_generation", "location_type",
                          "connectivity_tier", "expansion_wave"]].copy()
ev["rider_id"] = ev.rider_id.astype(str); ev["station_id"] = ev.station_id.astype(str)
for c in ["charger_generation", "location_type", "connectivity_tier", "expansion_wave"]: ev[c] = ev[c].astype(str)
ev["first_ts"] = ev.rider_id.map(first_ts)
ev["rel_d"] = (ev.event_ts - ev.first_ts).dt.total_seconds() / 86400
ret_ids = set(ev.loc[(ev.rel_d >= CFG["RET_FROM"]) & (ev.rel_d < CFG["RET_TO"]), "rider_id"])
coh["retained"] = coh.rider_id.isin(ret_ids).astype(int)
obs = coh[coh.observable]
print(f"observable riders: {len(obs):,} | retention (day {CFG['RET_FROM']}-{CFG['RET_TO']}): {obs.retained.mean():.1%}")
print("\nretention by signup month:"); print(obs.groupby(obs.first_ts.dt.to_period("M")).retained.agg(["size", "mean"]).round(3).T.to_string())
FIND["Q6 new riders / observable / retention"] = f"{len(coh):,} / {len(obs):,} / {obs.retained.mean():.1%}"

In [ ]:
# ---------- 9.2 early-experience features (first EARLY_DAYS days, first attempts)
e = ev[ev.rel_d <= CFG["EARLY_DAYS"]].sort_values(["rider_id", "event_ts"]).copy()
e["k"] = e.groupby("rider_id").cumcount(); e["day"] = e.event_ts.dt.normalize()
g = e.groupby("rider_id")
F = pd.DataFrame({"n_attempts": g.size(), "n_done": g.is_done.sum(), "fail_rate": g.service_fail.mean(),
                  "n_nobatt": g.is_nobatt.sum(), "n_abandon": g.is_abandon.sum(), "n_syserr": g.is_syserr.sum(),
                  "queue_mean": g.queue_wait_sec.mean(), "queue_max": g.queue_wait_sec.max(),
                  "n_stations": g.station_id.nunique(), "active_days": g.day.nunique(),
                  "list_price_mean": g.list_price_inr.mean(), "amount_mean": g.amount_charged_inr.mean(),
                  "disc_share": g.discount_inr.sum() / g.list_price_inr.sum().replace(0, np.nan),
                  "soh_out_mean": g.soh_out_pct.mean(), "km_mean": g.km_since_last_swap.mean()})
f3 = e[e.k < 3].groupby("rider_id").service_fail.max().rename("any_fail_first3").astype(int)
f1 = e[e.k == 0].set_index("rider_id")
F = F.join(f3).join(f1[["service_fail"]].rename(columns={"service_fail": "first_failed"}).astype(int))
F = F.join(f1[["hour", "charger_generation", "location_type", "connectivity_tier", "expansion_wave", "station_id"]].rename(
    columns={"hour": "first_hour", "charger_generation": "f_charger_generation", "location_type": "f_location_type",
             "connectivity_tier": "f_connectivity_tier", "expansion_wave": "f_expansion_wave", "station_id": "f_station"}))

tk_e = tickets.copy(); tk_e["first_ts"] = tk_e.rider_id.map(first_ts); tk_e["rel_d"] = (tk_e.created_ts - tk_e.first_ts).dt.total_seconds() / 86400
tk_e = tk_e[(tk_e.rel_d >= 0) & (tk_e.rel_d <= CFG["EARLY_DAYS"])]
tkg = tk_e.groupby("rider_id").agg(n_tickets=("ticket_id", "size"), n_tickets_unresolved=("resolution_status", lambda s: (s == "unresolved").sum()), ticket_res_hours=("resolution_hours", "mean"))
D = coh.set_index("rider_id").join(F, how="left").join(tkg, how="left")
for c in ["n_tickets", "n_tickets_unresolved"]: D[c] = D[c].fillna(0)

# static / contextual
D["is_partner"] = (D.rider_group == "partner").astype(int)
D["kyc_verified"] = D.kyc_verified.astype(int)
D["price_regime_post"] = ((D.first_ts >= PRICE_CHANGE_DATE).astype(int) if PRICE_CHANGE_DATE is not None else 0)
D["pilot_exposed"] = ((D.rider_city.isin(PILOT_CITIES)) & (D.first_ts >= PILOT_START) & (D.is_partner == 0)).astype(int)
comp_since = D.f_station.map(stations.set_index("station_id").competitor_within_1_5km_since)
D["competitor_nearby"] = (comp_since.notna() & (D.first_ts >= comp_since)).astype(int)
ctx = city_ctx.set_index(["city", "date"])
mi = pd.MultiIndex.from_arrays([D.rider_city.values, D.first_ts.dt.normalize().values])
D["competitor_promo"] = ctx["competitor_promo_active"].reindex(mi).fillna(False).astype(int).values
D["signup_month_num"] = D.first_ts.dt.year * 12 + D.first_ts.dt.month - 2024 * 12
D["partner_id"] = D.partner_id.fillna("Independent")
D["age_band"] = D.age_band.fillna("unknown"); D["declared_shift"] = D.declared_shift.fillna("unknown")
D["early_fail_bucket"] = pd.cut(D.n_nobatt + D.n_abandon + D.n_syserr, [-1, 0, 1, 2, 1000], labels=["0", "1", "2", "3+"])
print(D.shape, "| missing share of key features:"); print(D[["fail_rate", "queue_mean", "n_attempts", "disc_share", "soh_out_mean"]].isna().mean().round(3).to_dict())
Dobs = D[D.observable].copy()

In [ ]:
# ---------- 9.3 univariate retention gaps (one factor at a time)
def ret_table(df, col, q=None, bins=None, min_n=150):
    s = df[col]
    if q: s = pd.qcut(s, q, duplicates="drop")
    if bins is not None: s = pd.cut(s, bins)
    t = df.groupby(s, observed=True).retained.agg(n="size", retention="mean"); return t[t.n >= min_n]

specs = [("is_partner", None, None), ("rider_group", None, None), ("plan_type", None, None), ("signup_channel", None, None), ("vehicle_class", None, None),
         ("rider_city", None, None), ("kyc_verified", None, None), ("age_band", None, None), ("declared_shift", None, None), ("f_charger_generation", None, None),
         ("f_location_type", None, None), ("f_connectivity_tier", None, None), ("f_expansion_wave", None, None), ("price_regime_post", None, None), ("pilot_exposed", None, None),
         ("competitor_nearby", None, None), ("competitor_promo", None, None), ("first_failed", None, None), ("any_fail_first3", None, None), ("early_fail_bucket", None, None),
         ("n_attempts", 5, None), ("fail_rate", 4, None), ("queue_mean", 5, None), ("list_price_mean", 4, None), ("disc_share", 4, None), ("n_tickets", None, [-1, 0, 1, 100]),
         ("soh_out_mean", 5, None), ("days_signup_to_first", 5, None)]
rows = []; tables = {}
for col, q, bins in specs:
    try:
        t = ret_table(Dobs, col, q=q, bins=bins); tables[col] = t
        if len(t) > 1: rows.append({"factor": col, "levels": len(t), "min_retention": t.retention.min(), "max_retention": t.retention.max(), "gap_pp": (t.retention.max() - t.retention.min()) * 100})
    except Exception as ex:
        print("skip", col, ex)
gaps = pd.DataFrame(rows).sort_values("gap_pp", ascending=False)
print("Retention gap (max-min across levels, levels with n>=150):"); print(gaps.round(3).to_string(index=False))
top = gaps.head(12).factor.tolist()
fig, axes = plt.subplots(3, 4, figsize=(21, 11))
for a, c in zip(axes.ravel(), top):
    t = tables[c]; (t.retention * 100).plot.bar(ax=a, color="tab:blue"); a.axhline(Dobs.retained.mean() * 100, color="k", ls="--", lw=.8); a.set_title(c); a.set_xlabel(""); a.tick_params(axis="x", rotation=30); a.set_ylabel("% retained")
plt.tight_layout(); plt.show()

In [ ]:
# ---------- 9.4 survival-style curves: weekly activity retention of new riders
D["first_wk"] = ((D.first_ts.dt.normalize() - T0W).dt.days // 7).astype(int)
def weekly_retention(seg_col, max_k=26, min_den=60):
    c = D[[seg_col, "first_wk"]].dropna()
    # Drop overlapping columns from rw to prevent duplicate columns with suffixes on merge
    rw_clean = rw.drop(columns=[seg_col], errors="ignore")
    a = rw_clean.merge(c, left_on="rider_id", right_index=True)
    a["k"] = a.wk - a.first_wk; a = a[(a.k >= 0) & (a.k <= max_k)]
    num = a.groupby([seg_col, "k"], observed=True).rider_id.nunique().unstack(0)
    den = pd.DataFrame({s: [(grp.first_wk + k <= LAST_WK).sum() for k in range(max_k + 1)] for s, grp in c.groupby(seg_col, observed=True)}, index=range(max_k + 1))
    r_ = (num / den).reindex(range(max_k + 1)); r_[den < min_den] = np.nan
    return r_
D["early_fail_any"] = np.where((D.n_nobatt + D.n_abandon + D.n_syserr) > 0, "had >=1 failure in first 14d", "no failure in first 14d")
D["price_regime"] = np.where(D.price_regime_post == 1, "after base price change", "before base price change")
fig, axes = plt.subplots(2, 3, figsize=(19, 9))
for ax, sc in zip(axes.ravel(), ["rider_group", "early_fail_any", "rider_city", "signup_channel", "price_regime", "f_charger_generation"]):
    weekly_retention(sc).plot(ax=ax, marker="o", ms=3); ax.set_title(f"Share of cohort active in week k after first swap - by {sc}"); ax.set_xlabel("weeks since first attempt"); ax.set_ylabel("share active")
plt.tight_layout(); plt.show()

In [ ]:
# ---------- 9.5 multivariate model + grouped permutation importance
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score

NUM = ["n_attempts", "n_done", "fail_rate", "n_nobatt", "n_abandon", "n_syserr", "queue_mean", "queue_max", "n_stations", "active_days", "list_price_mean", "amount_mean", "disc_share",
       "soh_out_mean", "km_mean", "any_fail_first3", "first_failed", "first_hour", "n_tickets", "n_tickets_unresolved", "ticket_res_hours", "is_partner", "kyc_verified",
       "price_regime_post", "pilot_exposed", "competitor_nearby", "competitor_promo", "signup_month_num", "days_signup_to_first"]
CAT = ["partner_id", "vehicle_class", "signup_channel", "plan_type", "age_band", "declared_shift", "rider_city", "f_charger_generation", "f_location_type", "f_connectivity_tier", "f_expansion_wave"]
Xn = Dobs[NUM].astype(float)
Xc = pd.get_dummies(Dobs[CAT].astype(str), prefix_sep="__", dtype=float)
X = pd.concat([Xn, Xc], axis=1); y = Dobs.retained.values
cols_of = lambda names: [c for c in X.columns if c in names or any(c.startswith(n + "__") for n in names)]
GROUPS = {
    "early service quality (failures / queues)": cols_of(["fail_rate", "n_nobatt", "n_abandon", "n_syserr", "queue_mean", "queue_max", "any_fail_first3", "first_failed"]),
    "early engagement (mediator: n attempts etc.)": cols_of(["n_attempts", "n_done", "n_stations", "active_days"]),
    "price exposure": cols_of(["list_price_mean", "amount_mean", "disc_share", "price_regime_post", "pilot_exposed", "plan_type"]),
    "support experience": cols_of(["n_tickets", "n_tickets_unresolved", "ticket_res_hours"]),
    "partner affiliation": cols_of(["is_partner", "partner_id"]),
    "rider profile / acquisition": cols_of(["vehicle_class", "signup_channel", "age_band", "declared_shift", "kyc_verified", "days_signup_to_first"]),
    "battery condition received": cols_of(["soh_out_mean", "km_mean"]),
    "first-station equipment & place": cols_of(["rider_city", "f_charger_generation", "f_location_type", "f_connectivity_tier", "f_expansion_wave", "first_hour"]),
    "competition & calendar": cols_of(["competitor_nearby", "competitor_promo", "signup_month_num"]),
}
assert sum(len(v) for v in GROUPS.values()) == X.shape[1], "every column must belong to exactly one group"
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=.3, random_state=SEED, stratify=y)
lr  = make_pipeline(SimpleImputer(strategy="median"), StandardScaler(), LogisticRegression(max_iter=2000, C=.3))
hgb = HistGradientBoostingClassifier(max_depth=4, learning_rate=.06, max_iter=250, l2_regularization=1.0, random_state=SEED)
cv = StratifiedKFold(5, shuffle=True, random_state=SEED)
for name, m in [("logistic", lr), ("grad-boosting", hgb)]:
    s = cross_val_score(m, X, y, cv=cv, scoring="roc_auc"); print(f"{name:14s} 5-fold CV AUC = {s.mean():.3f} +- {s.std():.3f}")
lr.fit(Xtr, ytr); hgb.fit(Xtr, ytr)
print("hold-out AUC: logistic %.3f | boosting %.3f | base rate %.3f" % (roc_auc_score(yte, lr.predict_proba(Xte)[:, 1]), roc_auc_score(yte, hgb.predict_proba(Xte)[:, 1]), y.mean()))

def group_perm(model, Xt, yt, groups, n_rep=8):
    base = roc_auc_score(yt, model.predict_proba(Xt)[:, 1]); out = {}
    for gname, cols in groups.items():
        drops = []
        for _ in range(n_rep):
            Xp = Xt.copy(); Xp[cols] = Xt[cols].values[rng.permutation(len(Xt))]
            drops.append(base - roc_auc_score(yt, model.predict_proba(Xp)[:, 1]))
        out[gname] = (np.mean(drops), np.std(drops))
    return pd.DataFrame(out, index=["auc_drop", "sd"]).T.sort_values("auc_drop", ascending=False)
gi = group_perm(hgb, Xte, yte, GROUPS).join(group_perm(lr, Xte, yte, GROUPS), lsuffix="_boost", rsuffix="_logit")
print("\nGrouped permutation importance (AUC lost when a whole factor group is shuffled):"); print(gi.round(4).to_string())
coef = pd.Series(lr[-1].coef_[0], index=X.columns).sort_values(); print("\nLogistic (standardised) - most negative / most positive:"); print(pd.concat([coef.head(10), coef.tail(10)]).round(3).to_string())
fig, ax = plt.subplots(figsize=(10, 4.5)); gi["auc_drop_boost"].sort_values().plot.barh(xerr=gi.loc[gi["auc_drop_boost"].sort_values().index, "sd_boost"], ax=ax); ax.set_title("What does the model need to predict who returns? (AUC drop when shuffled)")
plt.tight_layout(); plt.show()
FIND["Q6 model AUC (boosting hold-out)"] = round(roc_auc_score(yte, hgb.predict_proba(Xte)[:, 1]), 3)

In [ ]:
# ---------- 9.6 primary vs secondary: stratified checks (is service quality still a driver *within* partner / independent riders?)
print("Retention by number of service failures in the first 14 days x rider group")
tt = Dobs.pivot_table(index="early_fail_bucket", columns="rider_group", values="retained", aggfunc=["mean", "size"], observed=True); print(tt.round(3).to_string())
print("\nRetention by early failure bucket x vehicle class"); print(Dobs.pivot_table(index="early_fail_bucket", columns="vehicle_class", values="retained", aggfunc="mean", observed=True).round(3).to_string())
print("\nRetention by early failure bucket x city"); print(Dobs.pivot_table(index="early_fail_bucket", columns="rider_city", values="retained", aggfunc="mean", observed=True).round(3).to_string())
print("\nIndependent riders: retention by plan type x price regime"); print(Dobs[Dobs.is_partner == 0].pivot_table(index="plan_type", columns="price_regime_post", values="retained", aggfunc=["mean", "size"]).round(3).to_string())
print("\nIndependent riders: retention by pilot exposure"); print(Dobs[(Dobs.is_partner == 0) & Dobs.rider_city.isin(PILOT_CITIES)].groupby("pilot_exposed").retained.agg(["mean", "size"]).round(3).to_string())
print("\nRetention by partner"); print(Dobs.groupby("partner_id").retained.agg(["mean", "size"]).round(3).sort_values("mean").to_string())

fig, axes = plt.subplots(1, 3, figsize=(18, 4.2))
(Dobs.pivot_table(index="early_fail_bucket", columns="rider_group", values="retained", aggfunc="mean", observed=True) * 100).plot(ax=axes[0], marker="o"); axes[0].set_title("Retention % vs early service failures")
(Dobs.pivot_table(index="early_fail_bucket", columns="rider_city", values="retained", aggfunc="mean", observed=True) * 100).plot(ax=axes[1], marker="o"); axes[1].set_title("... by city")
(Dobs[Dobs.is_partner == 0].pivot_table(index="plan_type", columns="price_regime_post", values="retained", aggfunc="mean") * 100).plot.bar(ax=axes[2]); axes[2].set_title("Independent riders: retention by plan x price regime"); axes[2].tick_params(axis="x", rotation=0)
plt.tight_layout(); plt.show()

# driver ranking = model importance + size of the retention gap + whether it survives stratification
driver = gi[["auc_drop_boost", "auc_drop_logit"]].copy()
driver["label_suggestion"] = np.select([driver.auc_drop_boost >= .02, driver.auc_drop_boost >= .005], ["PRIMARY candidate", "secondary / contributing"], "minor")
print("\nDriver summary (label is a starting point - confirm with the stratified tables above, and treat 'early engagement' as a mediator, not a cause):"); print(driver.round(4).to_string())
FIND["Q6 driver ranking"] = driver.label_suggestion.to_dict()

**Reading it — why new riders do not come back** (retained = active between day 30 and 60; 13,844 observable new riders, 85.4 % retained).

**Primary driver — the early service experience.**
* Riders with **≥ 2 service failures in their first 14 days** are retained 77.5 % vs 87.0 % (−9.5 pp); adjusted for city, signup month, vehicle class, partner, competitor and pilot exposure the effect is **−6.7 pp** (−9.5 pp when early activity is also held fixed). The relationship is a step, not a slope: 0 or 1 failure ≈ 87 %, 2 or more ≈ 75–78 %.
* A *failed first attempt* matters most when it is a no-battery failure (−6.3 pp adjusted).
* It holds in every stratum: independents (0.79) and partner riders (0.77) with 2 failures, 2W and 3W, all six cities. Permanent loss (no activity in the last 45 days) rises from 39 % (no failures) to 45 % (2) and 62 % (5+).
* Cohort-month retention tracks the network failure rate (correlation −0.81): the May-24 cohort retained 79 %, April-25 82 %, against ≈ 87 % in cool months.
* In the model, shuffling the early-service group costs 0.05 AUC (0.12 once the engagement mediator is removed) against ≤ 0.008 for any other group.

**Mediator, not a lever.** Early engagement (number of attempts, active days) is the strongest single predictor (0.15 AUC) but it is the *consequence* of a good or bad first experience.

**Secondary / contributing.** A competitor site within 1.5 km (−4.1 pp, p = 0.064); partner affiliation (−2.5 pp); 3W vehicle (−1.9 pp, partly because 3W riders fail more); pilot exposure (−1.8 pp, not significant). **Not drivers here:** price regime (pay-as-you-go 86.1 % → 87.4 % after the price rise, prepaid 87.5 % → 86.1 %), battery condition received, support experience, first-station equipment.

**Limits.** Observational; AUC ≈ 0.70, so most retention variation is not explained by anything observable. Value at stake: 2,246 new riders (16.3 %) had ≥ 2 early failures, ≈ 214 more lost per cohort window than if they retained like the rest, ≈ INR 2.2 m of revenue over 6 months (assumption: 6 months of remaining life per retained rider).

# PART 10 · Optional deep dives

In [ ]:
# ---------- 10.1 expansion waves: did they add service where riders needed it, or where sites were easy?
# NOTE: an earlier zone-level difference-in-differences used a single control zone, and its Wave-2 "+4.7 pp" is just the Apr-Jun heat season
# (Wave 2 opened in Mar 2025). It is removed; the like-for-like same-season comparison is in 11.9.
print("\nWhere were the wave stations opened? (site mix by wave, %)")
for c in ["location_type", "host_type", "connectivity_tier", "charger_generation"]:
    print((pd.crosstab(stations.expansion_wave, stations[c], normalize="index") * 100).round(1).to_string(), "\n")
print(stations.groupby("expansion_wave")[["monthly_rent_inr", "monthly_maintenance_inr", "slots_2w", "inventory_target_2w"]].mean().round(0).to_string())
scw = scorecard.assign(att_day=scorecard.attempts / scorecard.days_active).groupby("expansion_wave").agg(stations=("attempts", "size"), median_attempts_per_day=("att_day", "median"), p90_attempts_per_day=("att_day", lambda s: s.quantile(.9)),
                                                                                                         mean_fail=("service_fail_%", "mean"), worst_fail=("service_fail_%", "max"))
print("\nWave stations: average vs busiest-station picture"); print(scw.round(2).to_string())

In [ ]:
# ---------- 10.2 support-ticket text: does the comment add anything beyond the agent-assigned category?
cm = tickets.groupby("rider_comment").agg(n=("ticket_id", "size"), top_category=("category", lambda s: s.value_counts().index[0]), n_categories=("category", "nunique")).sort_values("n", ascending=False)
print(f"{len(cm)} distinct comments")
print(cm.to_string())       # <- read these, then refine THEME_RULES below
THEME_RULES = [   # first match wins; extend / correct after reading the list above (English + Hinglish)
    ("range / battery drain", r"range|km\b|mileage|jaldi|drain|khatam|utar|kam chal|battery (kam|low|down|weak)|charge (kam|nahi)|bike nahi chal|scooter nahi chal"),
    ("queue / waiting",       r"queue|line|wait|intezaar|lamba|late|der "),
    ("billing / price",       r"amount|extra|double|refund|paisa|price|bill|galat|kata|discount|invoice|charge"),
    ("app / payment",         r"app|otp|login|crash|payment|upi|wallet|update"),
    ("station / no battery",  r"station|cabinet|door|closed|band|khali|no battery|battery nahi|empty|stock"),
    ("vehicle problem",       r"speed|throttle|brake|motor|gaadi|tyre|vehicle"),
]
def theme(txt):
    t = str(txt).lower()
    for name, pat in THEME_RULES:
        if re.search(pat, t): return name
    return "other / unclassified"
cm["text_theme"] = [theme(c) for c in cm.index]
tickets["text_theme"] = tickets.rider_comment.map(cm.text_theme)
print("\nText theme vs agent category (row % of tickets)"); print((pd.crosstab(tickets.text_theme, tickets.category, normalize="index") * 100).round(1).to_string())
mis = tickets[(tickets.text_theme == "range / battery drain") & tickets.category.isin(["other", "app_issue"])]
print(f"\nrange/battery-drain comments filed as 'other'/'app_issue': {len(mis):,} ({len(mis)/max((tickets.text_theme=='range / battery drain').sum(),1):.1%} of range comments)")
tb2 = tickets[(tickets.text_theme == "range / battery drain") & tickets.battery_id.notna()].copy()
tb2["supplier"] = tb2.battery_id.map(bat_i.supplier); tb2["lot"] = tb2.battery_id.map(bat_i.manufacturing_lot)
print("range complaints by supplier (share of range tickets vs share of swaps):")
print(pd.DataFrame({"range_ticket_share": tb2.supplier.value_counts(normalize=True), "swap_share": done.battery_in_id.map(bat_i.supplier).value_counts(normalize=True)}).round(3).to_string())
print("\nTop lots in range complaints (per 1,000 swaps)"); lt = tb2.groupby(["supplier", "lot"]).size().rename("range_tickets").reset_index()
sw_lot = done.battery_in_id.map(bat_i.manufacturing_lot).value_counts().rename("swaps"); lt["swaps"] = lt.lot.map(sw_lot); lt["per_1k"] = lt.range_tickets / lt.swaps * 1000
print(lt[lt.swaps > 2000].nlargest(8, "per_1k").round(2).to_string(index=False))

In [ ]:
# ---------- 10.3 anomalies: stations, battery cohorts, time windows, firmware / connectivity, competitor openings
print("1) Stations > 3 sigma above network failure rate:")
print(stn_perf[stn_perf.z_vs_network > 3][["station_id", "city", "charger_generation", "connectivity_tier", "attempts", "service_fail_%", "no_batt_%", "abandon_%", "sys_err_%", "z_vs_network"]].round(2).to_string(index=False))

print("\n2) Firmware x connectivity (system-error and duplicate-prone areas):")
fc = seg_metrics(swaps, ["station_firmware", "connectivity_tier"])[["attempts", "service_fail_%", "sys_err_%", "queue_mean_s"]]; print(fc.round(2).to_string())
print("offline_batch share by connectivity tier:"); print((pd.crosstab(swaps.connectivity_tier, swaps.sync_mode, normalize="index") * 100).round(2).to_string())

print("\n3) Battery lots > 2 sigma (see part 7):"); print(lot[lot.z_resid.abs() > 2].sort_values("z_resid", ascending=False).round(2).head(10).to_string(index=False))

print("\n4) City-days with unusual failure rates (vs the city's own rolling 28-day median, MAD-scaled):")
cd2 = cd.sort_values(["city", "date"]).copy(); cd2["rate"] = cd2.fails / cd2.attempts * 100
cd2["med"] = cd2.groupby("city").rate.transform(lambda s: s.rolling(28, min_periods=14, center=True).median())
cd2["mad"] = cd2.groupby("city").rate.transform(lambda s: (s - s.rolling(28, min_periods=14, center=True).median()).abs().rolling(28, min_periods=14, center=True).median())
cd2["rz"] = (cd2.rate - cd2.med) / (1.4826 * cd2.mad.replace(0, np.nan))
print(cd2.nlargest(15, "rz")[["city", "date", "attempts", "rate", "med", "rz", "rainfall_mm", "max_temp_c", "grid_outage_hours", "flood_disruption", "heat_alert", "festival_or_event"]].round(2).to_string(index=False))

print("\n5) Competitor openings: weekly attempts at affected stations relative to the rest of the network")
comp = stations.dropna(subset=["competitor_within_1_5km_since"]); comp_ids = set(comp.station_id)
wk_att = swaps.groupby(["wk", "station_id"], observed=True).size().unstack(1); wk_att.columns = wk_att.columns.astype(str)
ctrl = wk_att[[c for c in wk_att.columns if c not in comp_ids]].mean(axis=1)
arr = []
for sid, d in comp.set_index("station_id").competitor_within_1_5km_since.items():
    we = int((d - T0W).days // 7)
    if sid not in wk_att.columns or we - 8 < 0 or we + 8 > wk_att.index.max(): continue
    rel = wk_att[sid] / ctrl; pre = rel.loc[we - 8:we - 1].mean()
    if pre and np.isfinite(pre): arr.append((rel.loc[we - 8:we + 7] / pre).values)
if arr:
    A = np.vstack([a for a in arr if len(a) == 16]); mean_rel = A.mean(axis=0)
    print(f"{len(A)} stations with a competitor opening in the window; relative volume 4-8 weeks after = {mean_rel[12:].mean():.3f} (1.0 = no change vs control)")
    plt.figure(figsize=(8, 3.5)); plt.plot(range(-8, 8), mean_rel, marker="o"); plt.axvline(0, color="k", ls="--"); plt.axhline(1, color="grey"); plt.title("Competitor opens within 1.5 km: relative weekly volume"); plt.xlabel("weeks from opening"); plt.show()
else:
    print("not enough competitor events inside the observation window")

In [ ]:
# ---------- 10.4 the budget decision: evidence table for the four proposals
ev_rows = []
mix_f = pd.Series({"no charged battery (stock)": swaps.is_nobatt.sum(), "abandoned in queue (capacity)": swaps.is_abandon.sum(), "system error (software)": swaps.is_syserr.sum()})
mix_f = mix_f / mix_f.sum() * 100
ev_rows.append(("Failure cause mix", "; ".join(f"{k}: {v:.0f}%" for k, v in mix_f.items()), "-> stock=more batteries / turnaround, queue=slots/stations, error=software"))
ev_rows.append(("Stations: corr(stock-out hours %, no-batt %)", f"{scorecard['stockout_hours_pct'].corr(scorecard['no_batt_%'], method='spearman'):.2f}", "high -> inventory/turnaround problem"))
ev_rows.append(("Stations: corr(attempts per slot-day, abandon %)", f"{scorecard['attempts_per_slot_day'].corr(scorecard['abandon_%'], method='spearman'):.2f}", "high -> capacity problem"))
ev_rows.append(("Stations: corr(avg charge min, stock-out hours %)", f"{scorecard['avg_charge_min'].corr(scorecard['stockout_hours_pct'], method='spearman'):.2f}", "correlated at station level, but only 3.2% of no-battery failures fall in stock-out hours -> mechanism not proven (see 5.4, 11.2)"))
hi = scorecard.attempts_per_slot_day >= scorecard.attempts_per_slot_day.quantile(.8)
ev_rows.append(("Failure % in busiest-utilisation quintile vs rest", f"{scorecard.loc[hi,'service_fail_%'].mean():.2f} vs {scorecard.loc[~hi,'service_fail_%'].mean():.2f}", "supports adding sites only where utilisation is saturated"))
ev_rows.append(("Pilot effect on peak share (independents, DiD)", str(FIND.get("Q5 pilot effect on peak share (independents, DiD)", "n/a")), "supports / rejects a network-wide rollout"))
ev_rows.append(("Largest partner", str(FIND.get("Q5 largest partner", "n/a")), "exclusive deal only if it is also the most valuable per swap"))
ev_rows.append(("Retention drivers", str(FIND.get("Q6 driver ranking", "n/a")), "points to where the budget should go"))
budget = pd.DataFrame(ev_rows, columns=["evidence", "value", "how to read"]); print(budget.to_string(index=False))

**Reading it — the four budget proposals** (final evidence table in 13.2).

| Proposal | Verdict | Deciding number |
|---|---|---|
| **More stations** | **Not supported** | Failure rate in the busiest-utilisation quintile 5.20 % vs 5.23 % for the rest; Spearman(attempts per slot-day, abandon %) = 0.00; only a third of failures are queue-driven. Expansion went where sites were easy, not where service was worst (11.9). |
| **More batteries** | **Partly — replace, don't add** | Only 3.2 % of no-battery failures fall in stock-out hours. But the Kyron lots have already cost INR 47.9 m in replacements — fix procurement and claim against the supplier rather than buy more packs. |
| **Network-wide pricing rollout** | **Not supported as proposed** | Per-swap revenue +INR 7.8, but −0.56 swaps per rider-week (−10 %), net revenue per rider-week +5.6 (not significant); two treated cities; does nothing for failures. |
| **Long-term exclusive with the largest fleet partner** | **Not supported** | ZipDrop is the largest partner and the biggest loss-maker: −INR 16.5 loaded per swap; the 28 % discount bought +2 % volume. |
| **Instead / in addition** | | Fix Gen1 in the heat (91 % of excess failures); recover or replace the Kyron lots; renegotiate ZipDrop; rescue riders after their second failure. |

# PART 11 · Follow-up analyses that test and correct Parts 4–10
Parts 4–10 answer each question once. This part checks the most decision-relevant results harder: what really drives the failure spikes (11.2), whether the margin story survives different battery-wear assumptions (11.3), what moved margin per swap (11.5), whether the pricing pilot and the ZipDrop discount pay for themselves (11.6, 11.7), how robust the retention result is (11.8), whether the expansion waves went where service was worst (11.9), what the ticket text adds (11.10) and which stations to fix first (11.11).

In [ ]:
# ===== 11.1 · why did the de-dup remove 0 rows? probe looser keys =====

def dup_probe(mask, keys, gap_sec=180):
    d = swaps.loc[mask, keys + ["event_ts"]].copy()
    for c in keys:
        if str(d[c].dtype) == "category": d[c] = d[c].cat.codes
    d = d.sort_values(keys + ["event_ts"])
    K = d[keys].values
    same = np.r_[False, (K[1:] == K[:-1]).all(axis=1)]
    t = d.event_ts.values
    gap = np.r_[np.nan, (t[1:] - t[:-1]) / np.timedelta64(1, "s")]
    return int((same & (gap <= gap_sec)).sum())

off_m = swaps.sync_mode == "offline_batch"
KEYSETS = {"rider+station+pack_in+pack_out+type (original)": ["rider_id", "station_id", "battery_in_id", "battery_out_id", "event_type"],
           "rider+station+pack_in+pack_out":                 ["rider_id", "station_id", "battery_in_id", "battery_out_id"],
           "rider+station+pack_in":                          ["rider_id", "station_id", "battery_in_id"],
           "rider+station":                                  ["rider_id", "station_id"],
           "rider+pack_in (any station)":                    ["rider_id", "battery_in_id"]}
rows = []
for name, keys in KEYSETS.items():
    rows.append({"keys": name, "offline_batch": dup_probe(off_m, keys), "realtime": dup_probe(~off_m, keys)})
n_off, n_rt = int(off_m.sum()), int((~off_m).sum())
dp = pd.DataFrame(rows).set_index("keys")
dp["offline per 100k rows"] = dp.offline_batch / n_off * 1e5; dp["realtime per 100k rows"] = dp.realtime / n_rt * 1e5
print("pairs flagged within 180 s, by key set and sync mode"); print(dp.round(1).to_string())
print("\nConclusion: no key set shows an excess of near-duplicates in offline_batch rows relative to realtime rows (see 11.1 reading), so 0 removals is correct.")


**Reading it.** There are no offline-sync duplicates in this export under the strict key sets (0 vs 0–1 flagged pairs), and no excess of near-duplicates in `offline_batch` rows relative to `realtime` rows under any key set. The loosest key (rider + station only) flags pairs in both modes at rates proportional to volume — consecutive legitimate swaps by the same rider — so removing them would delete real transactions. **Zero removals is the right answer**; the check is reported rather than assumed.

In [ ]:
# ===== 11.2 · what drives the failure spikes? heat x charger generation =====
sd = (swaps.groupby(["station_id", "date"], observed=True)
      .agg(att=("is_done", "size"), fails=("service_fail", "sum"), nobatt=("is_nobatt", "sum")).reset_index())
sd["station_id"] = sd.station_id.astype(str)
sd = sd.merge(stations[["station_id", "city", "charger_generation"]], on="station_id", how="left")
sd["charger_generation"] = sd.charger_generation.astype(str)
sd = sd.merge(city_ctx[["city", "date", "max_temp_c"]], on=["city", "date"], how="left")
sd["hot"] = sd.max_temp_c > 36
sd["temp_bin"] = pd.cut(sd.max_temp_c, [-50, 25, 30, 34, 36, 38, 40, 42, 60])
rate = lambda g: g.fails.sum() / g.att.sum() * 100

heat_tab = sd.groupby(["temp_bin", "charger_generation"], observed=True).apply(rate).unstack()
print("Failure % by day max-temperature and charger generation"); print(heat_tab.round(2).to_string())
ch = sd.groupby(["city", "charger_generation", "hot"], observed=True).apply(rate).unstack("hot")
ch.columns = ["normal day", "hot day (>36C)"]; ch["extra_pp"] = ch["hot day (>36C)"] - ch["normal day"]
print("\nCity x generation: failure % on normal vs hot days"); print(ch.round(2).to_string())

# how many failures are 'excess' heat failures?
base = sd[~sd.hot].groupby("charger_generation").apply(lambda g: g.fails.sum() / g.att.sum())
sd["excess"] = sd.fails - sd.att * sd.charger_generation.map(base)
hot_ex = sd[sd.hot].groupby("charger_generation").agg(attempts=("att", "sum"), fails=("fails", "sum"), excess=("excess", "sum"))
tot_f = sd.fails.sum()
print(f"\nhot days = {sd[sd.hot].att.sum()/sd.att.sum():.1%} of attempts but {sd[sd.hot].fails.sum()/tot_f:.1%} of all service failures")
print(hot_ex.round(0).to_string()); print(f"excess heat failures = {hot_ex.excess.sum():,.0f} = {hot_ex.excess.sum()/tot_f:.1%} of all failures")
FIND["Q2 heat: share of attempts / share of failures on >36C days"] = f"{sd[sd.hot].att.sum()/sd.att.sum():.1%} / {sd[sd.hot].fails.sum()/tot_f:.1%}"
FIND["Q2 excess heat failures (count, % of all)"] = f"{hot_ex.excess.sum():,.0f} ({hot_ex.excess.sum()/tot_f:.1%})"

# same-season year-over-year (removes the seasonal illusion) + generation mix
yq = swaps.groupby([swaps.date.dt.year.rename("year"), swaps.date.dt.quarter.rename("quarter")]).service_fail.mean().unstack(0) * 100
mixq = pd.crosstab([swaps.date.dt.year.rename("year"), swaps.date.dt.quarter.rename("quarter")], swaps.charger_generation, normalize="index") * 100
print("\nFailure % by quarter, year over year"); print(yq.round(2).to_string()); print("\nShare of attempts by generation (%)"); print(mixq.round(1).to_string())

# mechanism: ambient temperature -> charge time -> stock-outs -> failures
hh = hourly[["station_id", "hour_start", "ambient_temp_c", "avg_charge_minutes", "stockout2w", "charger_generation"]].copy()
hh["station_id"] = hh.station_id.astype(str); hh["amb_bin"] = pd.cut(hh.ambient_temp_c, [-10, 25, 30, 34, 38, 42, 60])
mech = hh.groupby(["amb_bin", "charger_generation"], observed=True).agg(charge_min=("avg_charge_minutes", "mean"), stockout_pct=("stockout2w", "mean"), n=("avg_charge_minutes", "size")).reset_index()
mech["stockout_pct"] *= 100
shh = sh.merge(hh[["station_id", "hour_start", "ambient_temp_c", "charger_generation"]], on=["station_id", "hour_start"], how="left")
shh["amb_bin"] = pd.cut(shh.ambient_temp_c, [-10, 25, 30, 34, 38, 42, 60])
nb_amb = shh.groupby(["amb_bin", "charger_generation"], observed=True).apply(lambda g: g.nobatt.sum() / g.attempts.sum() * 100).unstack()
shh["stock_bucket"] = pd.cut(shh.charged_2w_min, [-1, 0, 2, 4, 1000], labels=["0", "1-2", "3-4", "5+"])
nb_stock = shh.groupby(["stock_bucket", "charger_generation"], observed=True).apply(lambda g: g.nobatt.sum() / g.attempts.sum() * 100).unstack()
print("\nNo-battery failure % by ambient temp x generation (station-hours)"); print(nb_amb.round(2).to_string())
print("\nNo-battery failure % by charged stock x generation"); print(nb_stock.round(2).to_string())

fig, axes = plt.subplots(2, 3, figsize=(19, 9))
heat_tab.plot(ax=axes[0, 0], marker="o"); axes[0, 0].set_title("Failure % vs day max temp, by generation"); axes[0, 0].tick_params(axis="x", rotation=30)
mech.pivot(index="amb_bin", columns="charger_generation", values="charge_min").plot(ax=axes[0, 1], marker="o"); axes[0, 1].set_title("Avg charge minutes vs ambient temp")
mech.pivot(index="amb_bin", columns="charger_generation", values="stockout_pct").plot(ax=axes[0, 2], marker="o"); axes[0, 2].set_title("% station-hours in stock-out vs ambient temp")
nb_amb.plot(ax=axes[1, 0], marker="o"); axes[1, 0].set_title("No-battery failure % vs ambient temp")
(swaps.groupby(["month", "charger_generation"], observed=True).service_fail.mean().unstack() * 100).plot(ax=axes[1, 1], marker="o", ms=3); axes[1, 1].set_title("Failure % by month and generation"); mark_events(axes[1, 1])
mixq.plot.bar(stacked=True, ax=axes[1, 2]); axes[1, 2].set_title("Generation mix of attempts by quarter (%)")
plt.tight_layout(); plt.show()

In [ ]:
# ===== 11.3 · measured-wear model: month table, sensitivity of the margin story, and the post-retirement integrity check =====
mm = done.groupby("month").agg(swaps=("is_done", "size"), rev=("revenue", "sum"), energy=("energy_cost", "sum"), wear=("wear_cost", "sum"), wear_flat=("wear_cost_flat", "sum"),
                               cm_var=("cm_var", "sum"), cm_loaded=("cm_loaded", "sum"), fixed=("fixed_alloc", "sum"))
for c_ in ["rev", "energy", "wear", "wear_flat", "cm_var", "cm_loaded"]: mm[f"{c_}_ps"] = mm[c_] / mm.swaps
mm["cm_loaded_flat_ps"] = (mm.rev - mm.energy - mm.wear_flat - mm.fixed) / mm.swaps
sup_mix = pd.crosstab(done.month, done.battery_in_id.map(bat_i.supplier).astype(str), normalize="index") * 100
mm["bad_lot_share_%"] = done.battery_in_id.map(bat_i.manufacturing_lot).isin(BAD_LOTS).groupby(done.month).mean() * 100
print("Bad lots:", BAD_LOTS)
print(mm[["swaps", "rev_ps", "energy_ps", "wear_flat_ps", "wear_ps", "cm_loaded_flat_ps", "cm_loaded_ps", "bad_lot_share_%"]].round(2).to_string())

bad = bat2[bat2.manufacturing_lot.isin(BAD_LOTS)]
liab = bad[bad.retired].purchase_cost_inr.sum()
ret_m = bat2[bat2.retired].groupby(bat2.retired_date.dt.to_period("M")).agg(packs=("battery_id", "size"), cost=("purchase_cost_inr", "sum"))
print(f"\nbad-lot packs: {len(bad):,} | retired: {int(bad.retired.sum()):,} | replacement cost already incurred: INR {liab/1e6:,.1f} m "
      f"(= {liab/mm.rev.sum():.1%} of all revenue in the period)")
FIND["Q4 bad lots / packs / retired / replacement cost INR m"] = f"{BAD_LOTS} / {len(bad):,} / {int(bad.retired.sum()):,} / {liab/1e6:,.1f}"
FIND["Q1 loaded CM per swap, first->last month (measured wear)"] = f"{mm.cm_loaded_ps.iloc[0]:.2f} -> {mm.cm_loaded_ps.iloc[-1]:.2f}; (flat-cycle sensitivity {mm.cm_loaded_flat_ps.iloc[0]:.2f} -> {mm.cm_loaded_flat_ps.iloc[-1]:.2f})"

fig, axes = plt.subplots(2, 2, figsize=(16, 9)); x = mm.index
axes[0, 0].plot(x, mm.cm_loaded_flat_ps, "o--", label="flat 1,500-cycle wear (sensitivity)"); axes[0, 0].plot(x, mm.cm_loaded_ps, "o-", label="measured-degradation wear (headline)"); axes[0, 0].axhline(0, color="k", lw=.8)
axes[0, 0].set_title("Loaded contribution per swap (INR)"); axes[0, 0].legend(); mark_events(axes[0, 0])
axes[0, 1].plot(x, mm.wear_flat_ps, "o--", label="flat"); axes[0, 1].plot(x, mm.wear_ps, "o-", label="measured"); axes[0, 1].set_title("Battery wear cost per swap (INR)"); axes[0, 1].legend(); mark_events(axes[0, 1])
sup_mix.plot.area(ax=axes[1, 0], alpha=.85); axes[1, 0].set_title("Supplier mix of returned packs (% of swaps)"); mark_events(axes[1, 0])
ret_m.packs.plot.bar(ax=axes[1, 1], color="tab:red"); axes[1, 1].set_title("Packs retired per month (end of life)")
plt.tight_layout(); plt.show()

# ---- sensitivity: do the two margin MOVEMENTS survive other wear assumptions?  (price rise Jun->Jul-24; Kyron lots Aug->Sep-24)
cnt = done.groupby(["month", "battery_in_id"], observed=True).size().unstack(fill_value=0); cnt.columns = cnt.columns.astype(str)
tot_ = cnt.sum(axis=1).values
fixed_ps = (mm.fixed / mm.swaps).values; rev_ps_, en_ps_ = mm.rev_ps.values, mm.energy_ps.values
def wear_month(kind, p):
    if kind == "measured":
        usable = (bat_i2.initial_soh_pct - p).clip(lower=5); w = bat_i2.purchase_cost_inr * bat_i2.lpc / usable
    else:
        w = bat_i2.purchase_cost_inr / p
    w = w.reindex(cnt.columns); w = w.fillna(w.median())
    return (cnt.values @ w.values) / tot_
M = {k: mm.index.get_loc(pd.Timestamp(k)) for k in ["2024-01-01", "2024-06-01", "2024-07-01", "2024-08-01", "2024-09-01", "2025-06-01"]}
sc_rows = []
for kind, p in [("measured", 55), ("measured", EOL_SOH), ("measured", 65), ("measured", 70), ("measured", 75), ("flat", 1000), ("flat", 1500), ("flat", 2000)]:
    cm = rev_ps_ - en_ps_ - wear_month(kind, p) - fixed_ps
    sc_rows.append({"wear model": f"{kind}, end-of-life SoH {p:.0f}%" if kind == "measured" else f"flat, {p} cycles",
                    "Jan-24": cm[M["2024-01-01"]], "Jun-24": cm[M["2024-06-01"]], "Jul-24": cm[M["2024-07-01"]], "Aug-24": cm[M["2024-08-01"]],
                    "Sep-24": cm[M["2024-09-01"]], "Jun-25": cm[M["2025-06-01"]],
                    "price-rise swing (Jul-Jun)": cm[M["2024-07-01"]] - cm[M["2024-06-01"]], "Kyron-lot swing (Sep-Aug)": cm[M["2024-09-01"]] - cm[M["2024-08-01"]]})
sens = pd.DataFrame(sc_rows).set_index("wear model")
print("\nLoaded contribution per swap (INR) under alternative wear assumptions"); print(sens.round(2).to_string())
meas = sens.loc[[i for i in sens.index if i.startswith("measured")]]
print(f"\nprice-rise swing: {sens['price-rise swing (Jul-Jun)'].min():+.1f} to {sens['price-rise swing (Jul-Jun)'].max():+.1f} across ALL scenarios -> "
      + ("positive in every scenario" if (sens['price-rise swing (Jul-Jun)'] > 0).all() else "NOT positive in every scenario"))
print(f"Kyron-lot swing : {meas['Kyron-lot swing (Sep-Aug)'].min():+.1f} to {meas['Kyron-lot swing (Sep-Aug)'].max():+.1f} across the measured-degradation scenarios -> "
      + ("negative in every one" if (meas['Kyron-lot swing (Sep-Aug)'] < 0).all() else "NOT negative in every one")
      + "; flat-cycle models cannot show it by construction (they assume every pack lives equally long)")
print("level of loaded margin ranges from " f"{sens['Jun-25'].min():.1f} to {sens['Jun-25'].max():.1f} INR per swap in Jun-25: the LEVEL is assumption-dependent, the movements are not.")
FIND["Q1 sensitivity: price-rise swing / Kyron swing (INR per swap)"] = f"{sens['price-rise swing (Jul-Jun)'].min():+.1f}..{sens['price-rise swing (Jul-Jun)'].max():+.1f} / {meas['Kyron-lot swing (Sep-Aug)'].min():+.1f}..{meas['Kyron-lot swing (Sep-Aug)'].max():+.1f}"

# ---- integrity check: completed swaps logged on a pack AFTER its retirement date
rd_ = pd.to_datetime(done.battery_in_id.map(bat_i.retired_date))
post_ret = (done.event_ts > rd_).fillna(False)
n_post = int(post_ret.sum())
print(f"\ncompleted swaps on a pack AFTER its retirement date: {n_post:,} ({n_post/len(done):.2%} of completed swaps)")
da = (done.event_ts - rd_)[post_ret].dt.days
print("days after retirement:", da.describe().round(1).to_dict())
pr = done[post_ret]
print("by supplier:", pr.battery_in_id.map(bat_i.supplier).astype(str).value_counts().to_dict(), "| by pack lot (top):", pr.battery_in_id.map(bat_i.manufacturing_lot).astype(str).value_counts().head(5).to_dict())
print("share of those swaps that are on the flagged lots:", f"{pr.battery_in_id.map(bat_i.manufacturing_lot).isin(BAD_LOTS).mean():.1%}")
print("returned-pack SoH on those swaps vs the rest:", f"{pr.soh_in_pct.mean():.1f} vs {done.loc[~post_ret, 'soh_in_pct'].mean():.1f}")
log("completed swaps on packs after their retirement date", n_post, "kept in all analyses; effect on headline margin tested below")
a_ = done.groupby("month").cm_loaded.mean(); b_ = done[~post_ret].groupby("month").cm_loaded.mean()
dif = (a_ - b_).abs()
print(f"loaded CM per swap with vs without those swaps: max monthly difference INR {dif.max():.2f}")
print("VERDICT:", "headline margin is insensitive to these swaps (< INR 0.50 per swap in every month)" if dif.max() < 0.5 else "headline margin moves by more than INR 0.50 per swap in at least one month - treat month-level levels with care")
FIND["Q4 swaps on packs after retirement date: count / max monthly effect on loaded CM (INR)"] = f"{n_post:,} / {dif.max():.2f}"
del cnt, post_ret, rd_, pr


**Reading it.** Moving to measured wear changes the *level* of loaded margin from positive to negative, so the report states the level as assumption-dependent. The *changes* are what the sensitivity table tests: the July price-rise swing is positive in every scenario, and the Kyron-lot drag appears in every measured-degradation scenario (end-of-life SoH 55–75 %). It cannot appear in a flat-cycle model by construction, because that model assumes every pack lives equally long — an assumption the retirement data contradicts (1,438 Kyron packs retired at ≈ 511 swaps). The post-retirement check at the end prints whether excluding those 45,813 swaps changes the headline; see the printed verdict.

In [ ]:
# ===== 11.4 · why do 3W riders fail twice as often? + unit economics by vehicle class =====
v = seg_metrics(swaps, ["vehicle_class", "charger_generation"])[["attempts", "service_fail_%", "no_batt_%", "abandon_%"]]
print(v.round(2).to_string())
c3 = swaps.groupby(["city", "date", "vehicle_class"], observed=True).agg(att=("is_done", "size"), fails=("service_fail", "sum")).reset_index()
c3["city"] = c3.city.astype(str); c3 = c3.merge(city_ctx[["city", "date", "max_temp_c"]], on=["city", "date"], how="left"); c3["hot"] = c3.max_temp_c > 36
print("\nFailure % by vehicle class, normal vs hot days (>36C)")
print(c3.groupby(["vehicle_class", "hot"]).apply(lambda g: g.fails.sum() / g.att.sum() * 100).unstack().round(2).to_string())
print("\nFailure % by city x vehicle class"); print((swaps.groupby(["city", "vehicle_class"], observed=True).service_fail.mean().unstack() * 100).round(2).to_string())

# 3W demand per 3W pack allocated
s3 = swaps[swaps.vehicle_class == "3W"].groupby("station_id", observed=True).agg(att3=("is_done", "size"), fail3=("service_fail", "mean"), nob3=("is_nobatt", "mean")).reset_index()
s3["station_id"] = s3.station_id.astype(str)
s3 = s3.merge(scorecard.reset_index()[["station_id", "city", "charger_generation", "slots_3w", "inventory_target_3w", "days_active"]], on="station_id")
s3 = s3[(s3.slots_3w > 0) & (s3.inventory_target_3w > 0)].copy()
s3["att3_per_day"] = s3.att3 / s3.days_active; s3["demand_per_pack"] = s3.att3_per_day / s3.inventory_target_3w
s3["dpp_q"] = pd.qcut(s3.demand_per_pack, 4, duplicates="drop")
print(f"\n{len(s3)} stations with 3W slots. 3W failure by demand per allocated 3W pack (quartiles):")
print(s3.groupby("dpp_q", observed=True).agg(stations=("station_id", "size"), fail3_pct=("fail3", lambda s: s.mean() * 100), nobatt3_pct=("nob3", lambda s: s.mean() * 100)).round(2).to_string())
print("spearman(demand per pack, 3W failure):", round(s3[["demand_per_pack", "fail3"]].corr(method="spearman").iloc[0, 1], 3))
print("\n3W failure % by generation / city:"); print((s3.groupby("charger_generation").fail3.mean() * 100).round(2).to_dict()); print((s3.groupby("city").fail3.mean() * 100).round(2).to_dict())

ue = done.groupby("vehicle_class", observed=True).agg(swaps=("is_done", "size"), revenue=("revenue", "mean"), energy=("energy_cost", "mean"), wear=("wear_cost", "mean"), fixed=("fixed_alloc", "mean"), cm_var=("cm_var", "mean"), cm_loaded=("cm_loaded", "mean"))
print("\nUnit economics per completed swap by vehicle class (new wear model)"); print(ue.round(2).to_string())
FIND["Q3 3W vs 2W failure %"] = (swaps.groupby("vehicle_class", observed=True).service_fail.mean() * 100).round(2).to_dict()
FIND["Q5 loaded CM per swap 2W vs 3W"] = ue.cm_loaded.round(2).to_dict()

In [ ]:
# ===== 11.5 · what moved loaded margin per swap? (revenue vs energy vs wear vs site cost) =====
qd = done.groupby(done.month.dt.to_period("Q")).agg(rev=("revenue", "mean"), energy=("energy_cost", "mean"), wear=("wear_cost", "mean"), fixed=("fixed_alloc", "mean"), swaps=("is_done", "size"))
qd["cm_loaded"] = qd.rev - qd.energy - qd.wear - qd.fixed
print(qd.round(2).to_string())

def waterfall(a, b, title, ax):
    steps = [("revenue", b.rev - a.rev), ("energy", -(b.energy - a.energy)), ("battery wear", -(b.wear - a.wear)), ("site fixed cost", -(b.fixed - a.fixed))]
    labels = ["start"] + [s[0] for s in steps] + ["end"]; cum = a.cm_loaded
    ax.bar(0, a.cm_loaded, color="tab:blue"); ax.text(0, a.cm_loaded, f"{a.cm_loaded:.1f}", ha="center", va="bottom")
    for i, (l, vv) in enumerate(steps, start=1):
        ax.bar(i, vv, bottom=cum, color="tab:green" if vv >= 0 else "tab:red"); ax.text(i, cum + vv, f"{vv:+.1f}", ha="center", va="bottom"); cum += vv
    ax.bar(len(labels) - 1, b.cm_loaded, color="tab:blue"); ax.text(len(labels) - 1, b.cm_loaded, f"{b.cm_loaded:.1f}", ha="center", va="bottom")
    ax.set_xticks(range(len(labels))); ax.set_xticklabels(labels); ax.axhline(0, color="k", lw=.8); ax.set_title(title); ax.set_ylabel("INR per completed swap")
fig, axes = plt.subplots(1, 2, figsize=(16, 5))
waterfall(qd.iloc[0], qd.iloc[-1], f"Loaded CM/swap: {qd.index[0]} -> {qd.index[-1]}", axes[0])
waterfall(qd.loc[pd.Period("2024Q2")], qd.loc[pd.Period("2025Q2")], "Same season: 2024Q2 -> 2025Q2", axes[1])
plt.tight_layout(); plt.show()

In [ ]:
# ===== 11.6 · ZipDrop discount counterfactual and partner break-even (partner table `pt` is built once in 8.4, on measured wear) =====
show_cols = ["partner_name", "partner_segment", "swaps", "eff_discount_%", "cm_var_per_swap", "cm_loaded_per_swap", "cm_loaded", "share_of_total_loss_%", "var_break_even_disc_%", "headroom_var_pp"]
print(pt.drop(index="Independent").sort_values("cm_loaded_per_swap", ascending=False)[show_cols].round(2).to_string())
FIND["Q5 partner ranking by loaded CM/swap"] = pt.sort_values("cm_loaded_per_swap", ascending=False).cm_loaded_per_swap.round(1).to_dict()

# ZipDrop: discount 12% -> 28% on 2024-11-01. Did it buy incremental volume, net of network growth?
ZID, d0, W = "FP-03", pd.Timestamp("2024-11-01"), pd.Timedelta(days=90)
pre_all, post_all = done[(done.date >= d0 - W) & (done.date < d0)], done[(done.date >= d0) & (done.date < d0 + W)]
z_pre, z_post = pre_all[pre_all.partner_key == ZID], post_all[post_all.partner_key == ZID]
sh_pre, sh_post = len(z_pre) / len(pre_all), len(z_post) / len(post_all)
cf_swaps = sh_pre * len(post_all)                                   # swaps ZipDrop would have had at its old network share
cm28 = z_post.cm_var.mean(); cm12 = cm28 + z_post.list_price_inr.mean() * (0.28 - 0.12)
print(f"\nZipDrop share of network swaps: {sh_pre:.2%} -> {sh_post:.2%}")
print(f"swaps in the 90d after: actual {len(z_post):,} vs counterfactual (old share) {cf_swaps:,.0f}  -> incremental {len(z_post)-cf_swaps:,.0f} ({len(z_post)/cf_swaps-1:+.1%})")
print(f"variable CM per swap: {cm12:.2f} (had discount stayed 12%) vs {cm28:.2f} (28%)")
print(f"total variable CM over 90d: actual {len(z_post)*cm28:,.0f} vs counterfactual {cf_swaps*cm12:,.0f}")
need_txt = f"x{cm12/cm28:.2f}" if cm28 > 0 else "none - margin per swap is <= 0, so no volume uplift can break even"
print(f"volume uplift needed to break even: {need_txt} | uplift actually achieved: x{len(z_post)/cf_swaps:.2f}")
gain_ps = cm12 - cm28
print(f"moving 28% -> 12%: +INR {gain_ps:.1f} variable CM per swap = INR {gain_ps*len(z_post)/1e6:,.2f} m per 90 days at post-amendment volume "
      f"(ignores any volume loss; the discount bought only {len(z_post)/cf_swaps-1:+.1%} extra volume)")
var_be  = (1 - (z_post.energy_cost.mean() + z_post.wear_cost.mean()) / z_post.list_price_inr.mean()) * 100
load_be = (1 - (z_post.energy_cost.mean() + z_post.wear_cost.mean() + z_post.fixed_alloc.mean()) / z_post.list_price_inr.mean()) * 100
print(f"break-even discount on ZipDrop swaps: variable {var_be:.1f}% | loaded (incl. site cost) {load_be:.1f}%  (28% applied, 12% originally)")
FIND["Q5 ZipDrop break-even discount: variable / loaded %"] = f"{var_be:.1f} / {load_be:.1f}"
FIND["Q5 ZipDrop 28%->12%: variable CM gain per swap / per 90d INR m"] = f"{gain_ps:.1f} / {gain_ps*len(z_post)/1e6:,.2f}"
FIND["Q5 ZipDrop discount: achieved vs needed volume uplift"] = f"x{len(z_post)/cf_swaps:.2f} vs {need_txt}"

ms = done.groupby(["month", "partner_key"], observed=True).size().unstack()
peers = partners[(partners.contract_type == "volume_tiered") & (partners.partner_id != ZID)].partner_id.tolist()
idx = pd.DataFrame({"ZipDrop": ms[ZID], "other volume-tiered partners": ms[peers].sum(axis=1), "network": ms.sum(axis=1)})
idx = idx / idx.loc[pd.Timestamp("2024-10-01")] * 100
fig, axes = plt.subplots(1, 2, figsize=(16, 4.5))
idx.plot(ax=axes[0], marker="o"); axes[0].axvline(d0, color="k", ls="--"); axes[0].set_title("Swaps indexed to Oct-2024 = 100: did the discount outgrow the market?")
pt.sort_values("cm_loaded_per_swap")["cm_loaded_per_swap"].plot.barh(ax=axes[1], color="tab:green"); axes[1].axvline(0, color="k", lw=.8); axes[1].set_title("Loaded CM per swap by partner (INR)")
plt.tight_layout(); plt.show()

**Reading it.** The partner table (built in 8.4) ranks all twelve partners on the measured wear model; ZipDrop is last among 2W partners and the largest total loss. The counterfactual shows that its 28 % discount bought about +2 % volume against a network that grew 3×; at 12 % it would have earned a positive variable margin. No volume uplift can rescue a negative margin per swap, which is what the printout says rather than a negative multiplier.

In [ ]:
# ===== 11.7 · pilot (Bengaluru + Pune): revenue per rider, active riders, hourly shift; rollout estimate =====
import statsmodels.formula.api as smf
ind = done[done.seg == "independent"]
cw = ind.groupby(["city", "wk"], observed=True).agg(rev=("revenue", "sum"), cm=("cm_var", "sum"), swaps=("is_done", "size"), riders=("rider_id", "nunique")).reset_index()
cw["city"] = cw.city.astype(str)
cw["rev_per_rider"] = cw.rev / cw.riders; cw["cm_per_rider"] = cw.cm / cw.riders; cw["swaps_per_rider"] = cw.swaps / cw.riders; cw["log_active_riders"] = np.log(cw.riders)
wkp = int((PILOT_START - T0W).days // 7)
cw = cw[cw.wk.between(wkp - 12, wkp + 11)].copy()
cw["pilot"] = cw.city.isin(PILOT_CITIES).astype(int); cw["post"] = (cw.wk >= wkp).astype(int)
rows = []
for y_ in ["rev_per_rider", "cm_per_rider", "swaps_per_rider", "log_active_riders"]:
    m = smf.ols(f"{y_} ~ pilot:post + C(city) + C(wk)", cw).fit(cov_type="HC1")
    pre_mean = cw.loc[(cw.pilot == 1) & (cw.post == 0), y_].mean()
    rows.append({"outcome (independent riders, weekly)": y_, "DiD effect": m.params["pilot:post"], "std err": m.bse["pilot:post"], "p": m.pvalues["pilot:post"], "pilot pre-mean": pre_mean})
print("Two-way fixed-effects DiD, 12 weeks before vs after, pilot vs non-pilot cities"); print(pd.DataFrame(rows).round(4).to_string(index=False))

# did riders change *when* they swap?
def hour_share(df, d_from, d_to): s = df[(df.date >= d_from) & (df.date < d_to)].hour.value_counts(normalize=True).reindex(range(24), fill_value=0); return s * 100
fig, axes = plt.subplots(1, 2, figsize=(16, 4))
for ax, (lab, mask) in zip(axes, [("pilot cities", ind.city.isin(PILOT_CITIES)), ("other cities", ~ind.city.isin(PILOT_CITIES))]):
    sub = ind[mask]
    hour_share(sub, PILOT_START - pd.Timedelta(weeks=12), PILOT_START).plot(ax=ax, label="12w before"); hour_share(sub, PILOT_START, PILOT_START + pd.Timedelta(weeks=12)).plot(ax=ax, label="12w after")
    for h in PEAK_HOURS: ax.axvspan(h - .5, h + .5, color="red", alpha=.07)
    ax.set_title(f"Independent riders: share of swaps by hour - {lab}"); ax.legend(); ax.set_xlabel("hour")
plt.tight_layout(); plt.show()

# rollout sizing. (a) per-swap price effect only - IGNORES the volume response; (b) volume-adjusted, from revenue per rider-week
d_rev = did(done, "revenue", PILOT_START).loc["independent", "DiD"]
cut = DATA_END.normalize() - pd.Timedelta(days=365)
l12 = ind[(~ind.city.isin(PILOT_CITIES)) & (ind.date > cut)]
tot12 = done[done.date > cut].revenue.sum()
print(f"(a) per-swap price effect only: DiD revenue per independent swap INR {d_rev:.2f} x {len(l12):,} independent swaps in non-pilot cities (last 12m) "
      f"= INR {len(l12)*d_rev/1e6:,.1f} m/yr ({len(l12)*d_rev/tot12:.1%} of revenue). Ignores that riders swapped less - NOT a forecast.")
eff, se = rows[0]["DiD effect"], rows[0]["std err"]          # revenue per independent rider-week
rwk = l12.groupby(["city", "wk"], observed=True).rider_id.nunique().sum()      # independent rider-weeks, non-pilot cities, last 12m
est, lo_, hi_ = rwk * eff / 1e6, rwk * (eff - 1.96 * se) / 1e6, rwk * (eff + 1.96 * se) / 1e6
print(f"(b) volume-adjusted: DiD revenue per rider-week INR {eff:.2f} (p = {rows[0]['p']:.3f}) x {rwk:,} rider-weeks = INR {est:,.1f} m/yr, 95% interval {lo_:,.1f} to {hi_:,.1f} m")
print("    -> the interval " + ("includes zero: the revenue gain is not distinguishable from nothing" if lo_ < 0 < hi_ else "excludes zero"))
print("Caveats: only 2 treated cities; assumes other cities respond like Bengaluru/Pune; partner segments respond differently (see 8.3).")
FIND["Q5 pilot: per-swap price effect x non-pilot swaps, INR m/yr (ignores volume loss)"] = round(len(l12) * d_rev / 1e6, 1)
FIND["Q5 pilot: volume-adjusted rollout revenue INR m/yr (point; 95% CI)"] = f"{est:,.1f} ({lo_:,.1f} to {hi_:,.1f})"


**Reading it.** The per-swap price effect (INR 7.8 per independent swap) multiplied by non-pilot swaps gives ≈ INR 5.1 m a year, but that number **ignores the volume response**: independents swapped 0.56 fewer times per week in the pilot cities. Using revenue per rider-week (+INR 5.6, p = 0.155) the volume-adjusted estimate is smaller and its 95 % interval includes zero. With only two treated cities this is a reason to extend the pilot with a staggered design, not to roll out network-wide.

In [ ]:
# ===== 11.8 · retention: failure threshold, adjusted effects, cohort-month link, no-mediator model, permanent churn, value at stake =====
import statsmodels.formula.api as smf
fa = (e[e.k == 0].set_index("rider_id")[["is_nobatt", "is_abandon", "is_syserr", "queue_wait_sec"]].astype({"is_nobatt": int, "is_abandon": int, "is_syserr": int}).add_prefix("fa_"))
R = Dobs.join(fa)
R["n_fail"] = R.n_nobatt + R.n_abandon + R.n_syserr; R["ge2"] = (R.n_fail >= 2).astype(int); R["is3w"] = (R.vehicle_class == "3W").astype(int)
R["fm"] = R.first_ts.dt.to_period("M").astype(str)
vc = R.fm.value_counts(); R = R[R.fm.isin(vc[vc >= 150].index)].copy()

# (a) dose-response
R["nf_b"] = R.n_fail.clip(upper=5)
dr = R.groupby("nf_b").retained.agg(n="size", ret="mean"); dr["se"] = np.sqrt(dr.ret * (1 - dr.ret) / dr.n)
print("Retention by # service failures in first 14 days"); print(dr.round(3).to_string())
# (b) first-attempt failure type, adjusted for city, signup month, vehicle, partner, competitor, pilot
m1 = smf.logit("retained ~ fa_is_nobatt + fa_is_abandon + fa_is_syserr + is_partner + is3w + competitor_nearby + pilot_exposed + C(rider_city) + C(fm)", R).fit(disp=0)
ame1 = m1.get_margeff().summary_frame(); print("\nAverage marginal effect on P(retained) of the FIRST attempt's outcome (adjusted):")
print(ame1.loc[~ame1.index.str.startswith("C("), ["dy/dx", "Pr(>|z|)"]].round(4).to_string())
# (c) >=2 failures, with and without holding early activity fixed
m2 = smf.logit("retained ~ ge2 + is_partner + is3w + competitor_nearby + pilot_exposed + C(rider_city) + C(fm)", R).fit(disp=0)
m3 = smf.logit("retained ~ ge2 + np.log1p(n_attempts) + is_partner + is3w + competitor_nearby + pilot_exposed + C(rider_city) + C(fm)", R).fit(disp=0)
for nm, mm_ in [("adjusted", m2), ("adjusted + early activity held fixed", m3)]:
    a = mm_.get_margeff().summary_frame(); print(f">=2 early failures, {nm}: dy/dx = {a.loc['ge2','dy/dx']:.4f} (p={a.loc['ge2','Pr(>|z|)']:.4f})")
FIND["Q6 adjusted effect of >=2 early failures on retention (pp)"] = round(float(m2.get_margeff().summary_frame().loc["ge2", "dy/dx"]) * 100, 1)
FIND["Q6 adjusted effect of competitor_nearby (pp)"] = round(float(m2.get_margeff().summary_frame().loc["competitor_nearby", "dy/dx"]) * 100, 1)

# (d) cohort-month retention vs the network failure rate that month
cmn = R.groupby("fm").agg(n=("retained", "size"), retention=("retained", "mean"), mean_fails_per_rider=("n_fail", "mean"))
net = monthly["svc_fail_%"].copy(); net.index = net.index.to_period("M").astype(str); cmn["network_fail_%"] = cmn.index.map(net)
print("\nCohort month: retention vs network failure rate"); print(cmn.round(3).to_string()); print("corr(retention, network failure %):", round(cmn.retention.corr(cmn["network_fail_%"]), 3))

# (e) model WITHOUT the early-engagement mediator: what predicts return once you remove 'how much they rode'?
med = [k for k in GROUPS if "mediator" in k][0]
X2 = X.drop(columns=GROUPS[med]); G2 = {k: v for k, v in GROUPS.items() if k != med}
X2tr, X2te, y2tr, y2te = train_test_split(X2, y, test_size=.3, random_state=SEED, stratify=y)
hgb2 = HistGradientBoostingClassifier(max_depth=4, learning_rate=.06, max_iter=250, l2_regularization=1.0, random_state=SEED).fit(X2tr, y2tr)
print(f"\nhold-out AUC without engagement features: {roc_auc_score(y2te, hgb2.predict_proba(X2te)[:, 1]):.3f}")
gi2 = group_perm(hgb2, X2te, y2te, G2); print(gi2.round(4).to_string())

# (f) permanent churn definition (no activity in the last 45 days of data), riders with >=120 days of history
last_ts = ev.groupby("rider_id").event_ts.max()
PC = R[R.first_ts <= DATA_END - pd.Timedelta(days=120)].copy(); PC["last_ts"] = PC.index.map(last_ts)
PC["lost"] = (PC.last_ts < DATA_END - pd.Timedelta(days=45)).astype(int)
print("\nPermanently lost (no activity in last 45 days) by early failures:"); print(PC.groupby("nf_b").lost.agg(n="size", lost="mean").round(3).to_string())

# (g) value at stake (ASSUMPTION: LIFETIME_M months of remaining life per retained rider)
LIFETIME_M = 6
rev_pr_m = (monthly.revenue / monthly.active_riders).iloc[-6:].mean()
g_lo, g_hi = R[R.n_fail <= 1].retained.mean(), R[R.n_fail >= 2].retained.mean()
n_hi = int((R.n_fail >= 2).sum()); lost = n_hi * (g_lo - g_hi)
print(f"\nriders with >=2 early failures: {n_hi:,} ({n_hi/len(R):.1%} of new riders) | retention {g_hi:.1%} vs {g_lo:.1%} -> ~{lost:,.0f} riders lost per cohort window")
print(f"revenue per active rider-month (last 6m): INR {rev_pr_m:,.0f} -> revenue at risk ~ INR {lost*rev_pr_m*LIFETIME_M/1e6:,.2f} m over {LIFETIME_M} months (assumption-driven)")
FIND["Q6 riders with >=2 early failures: share / retention gap"] = f"{n_hi/len(R):.1%} / {(g_lo-g_hi)*100:.1f}pp"

In [ ]:
# ===== 11.9 · did the waves go where service was worst? (replaces the 1-control-zone DiD) =====
print("Stations by wave and city"); print(pd.crosstab(stations.expansion_wave, stations.city).to_string())
print("\nStations by generation and city"); print(pd.crosstab(stations.charger_generation, stations.city).to_string())
city_need = pd.DataFrame({"failure_%": swaps.groupby("city", observed=True).service_fail.mean() * 100,
                          "gen1_share_of_stations_%": stations.groupby("city").charger_generation.apply(lambda s: (s == "Gen1").mean() * 100),
                          "wave_stations": stations[stations.expansion_wave != "Launch"].groupby("city").size()}).fillna(0)
city_need["wave_station_share_%"] = city_need.wave_stations / city_need.wave_stations.sum() * 100
print("\nNeed vs placement by city"); print(city_need.round(1).sort_values("failure_%", ascending=False).to_string())

hs_ = swaps[swaps.month.dt.month.isin([4, 5, 6])]
t = hs_.groupby([hs_.month.dt.year.rename("year"), "city", "charger_generation"], observed=True).agg(att=("is_done", "size"), fail=("service_fail", "mean"), stations=("station_id", "nunique"))
t["fail"] *= 100; t["att_per_station_day"] = t.att / t.stations / 91
print("\nHeat season (Apr-Jun) same-city comparison, 2024 vs 2025: failure % and attempts per station-day")
print(t[["fail", "att_per_station_day", "stations"]].unstack(0).round(2).to_string())
g1 = swaps[swaps.charger_generation == "Gen1"].groupby(["city", "month"], observed=True).agg(att=("is_done", "size"), st=("station_id", "nunique")).reset_index()
g1["att_per_station_month"] = g1.att / g1.st
print("\nDid Gen1 stations get relief from the waves? attempts per Gen1 station per month, by city:")
print(g1.pivot(index="month", columns="city", values="att_per_station_month").iloc[[0, 6, 12, 17]].round(0).to_string())
FIND["Q3 waves: station share by city vs failure %"] = city_need[["failure_%", "wave_station_share_%"]].round(1).to_dict("index")

In [ ]:
# ===== 11.10 · ticket text: families, typos, parsed km, link to batteries =====
FIX = {"staion": "station", "battry": "battery", "rnage": "range"}
def norm(t):
    t = str(t).lower()
    for a, b in FIX.items(): t = t.replace(a, b)
    return t
FAMILIES = [("range_km_report", r"dying at"), ("vehicle_lag", r"vehicle performance|lag raha|weak lag"),
            ("no_battery", r"charged batt|batteries discharge|no battery|empty station|sab batt|battery nahi"),
            ("queue", r"queue|line"), ("billing", r"amount|surcharge|charged extra"), ("low_range", r"range|discharge fast"), ("app", r"app |qr")]
def family(t):
    n = norm(t)
    for name, pat in FAMILIES:
        if re.search(pat, n): return name
    return "unclassified"
uc = pd.Series(tickets.rider_comment.unique())
fam_map = {c: family(c) for c in uc}; typo_map = {c: norm(c) != str(c).lower() for c in uc}
tickets["family"] = tickets.rider_comment.map(fam_map); tickets["typo"] = tickets.rider_comment.map(typo_map)
tickets["rep_km"] = pd.to_numeric(tickets.rider_comment.str.extract(r"dying at (-?\d+) km")[0], errors="coerce").where(lambda s: s > 0)
print("unclassified comments:", int((tickets.family == "unclassified").sum()), "(should be 0)")
print("\nText family vs agent category (counts)"); print(pd.crosstab(tickets.family, tickets.category).to_string())
print(f"\n'other' tickets that are really vehicle_lag: {((tickets.category=='other')&(tickets.family=='vehicle_lag')).sum():,} of {(tickets.category=='other').sum():,} "
      f"({(tickets.family=='vehicle_lag').mean():.1%} of ALL tickets)")
print("\nTypo share by channel / city:"); print((tickets.groupby("channel").typo.mean() * 100).round(1).to_dict()); print((tickets.groupby("city").typo.mean() * 100).round(1).to_dict())

# do the 'mystery' families point at weak batteries?
tk = tickets.dropna(subset=["battery_id"]).copy()
tk["supplier"] = tk.battery_id.map(bat_i.supplier); tk["lot"] = tk.battery_id.map(bat_i.manufacturing_lot); tk["cur_soh"] = tk.battery_id.map(bat_i.current_soh_pct); tk["bad_lot"] = tk.lot.isin(BAD_LOTS)
fam = tk.groupby("family").agg(tickets=("ticket_id", "size"), mean_pack_soh=("cur_soh", "mean"), pct_kyron=("supplier", lambda s: (s == "Kyron").mean() * 100), pct_bad_lot=("bad_lot", lambda s: s.mean() * 100))
share_bad_swaps = done.battery_in_id.map(bat_i.manufacturing_lot).isin(BAD_LOTS).mean() * 100
print(f"\nTickets that cite a pack: pack health / supplier / bad-lot share by text family  (bad lots carry {share_bad_swaps:.1f}% of swaps; average pack SoH {bat.current_soh_pct.mean():.1f}%)")
print(fam.round(1).to_string())
bat_fams = ["vehicle_lag", "low_range", "range_km_report"]
sw_lot = done.battery_in_id.map(bat_i.manufacturing_lot).value_counts().rename("swaps")
lt = tk[tk.family.isin(bat_fams)].groupby(["supplier", "lot"]).size().rename("tickets").reset_index(); lt["swaps"] = lt.lot.map(sw_lot); lt["per_1k_swaps"] = lt.tickets / lt.swaps * 1000
print("\nBattery-type complaints (vehicle_lag + low_range + dying-at-km) per 1,000 swaps, worst lots:"); print(lt[lt.swaps > 2000].nlargest(8, "per_1k_swaps").round(2).to_string(index=False))
print("\nReported 'dying at' km by supplier (median):", tk[tk.family == "range_km_report"].groupby("supplier").rep_km.median().to_dict(), "| vs observed median km per swap:", round(done.km_since_last_swap.median(), 1))

# billing disputes around the pilot (surcharge comments)
sur = tickets[tickets.rider_comment.str.contains("surcharge", case=False, na=False)].copy(); sur["pilot_city"] = sur.city.isin(PILOT_CITIES)
sur["period"] = np.where(sur.created_ts >= PILOT_START, "after pilot", "before pilot")
print("\n'surcharge' comments by pilot city and period:"); print(pd.crosstab(sur.pilot_city, sur.period).to_string())
FIND["Q3 tickets: vehicle_lag share of all tickets (all filed as 'other')"] = f"{(tickets.family=='vehicle_lag').mean():.1%}"

In [ ]:
# ===== 11.11 · which stations to fix first, and what the failures cost =====
base_rate = swaps[swaps.charger_generation != "Gen1"].service_fail.mean()
S = scorecard.reset_index()[["station_id", "city", "zone", "charger_generation", "connectivity_tier", "attempts", "svc_fails", "days_active"]].copy()
S["excess_fails"] = (S.svc_fails - S.attempts * base_rate).clip(lower=0)
rev_sw, cm_sw = done.revenue.mean(), done.cm_var.mean()
S["lost_revenue_per_year"] = S.excess_fails * rev_sw / S.days_active * 365
S["lost_cm_per_year"] = S.excess_fails * cm_sw / S.days_active * 365
S = S.sort_values("lost_cm_per_year", ascending=False)
print(f"benchmark failure rate (non-Gen1 stations) = {base_rate:.2%} | revenue per swap {rev_sw:.1f} | variable CM per swap {cm_sw:.1f}")
print(S.head(25).round(0).to_string(index=False))
print("\nBy city and generation"); print(S.groupby(["city", "charger_generation"]).agg(stations=("station_id", "size"), excess_fails=("excess_fails", "sum"), lost_cm_per_year=("lost_cm_per_year", "sum")).round(0).to_string())
print(f"\nGen1 stations: {int((S.charger_generation=='Gen1').sum())} | share of all excess failures: {S[S.charger_generation=='Gen1'].excess_fails.sum()/S.excess_fails.sum():.1%} | "
      f"lost variable CM per year: INR {S[S.charger_generation=='Gen1'].lost_cm_per_year.sum()/1e6:,.1f} m")
RETROFIT_COST_INR = None          # <- fill in per-station retrofit/replacement cost when you have a figure
if RETROFIT_COST_INR:
    g1s = S[S.charger_generation == "Gen1"].copy(); g1s["payback_years"] = RETROFIT_COST_INR / g1s.lost_cm_per_year.replace(0, np.nan)
    print(g1s[["station_id", "city", "lost_cm_per_year", "payback_years"]].head(20).round(2).to_string(index=False))
print("Caveat: assumes a failed attempt is a fully lost swap (upper bound - some riders retry elsewhere), and excludes churn cost (see retention cell).")
FIND["Q3 Gen1: share of excess failures / lost CM per year (INR m)"] = f"{S[S.charger_generation=='Gen1'].excess_fails.sum()/S.excess_fails.sum():.1%} / {S[S.charger_generation=='Gen1'].lost_cm_per_year.sum()/1e6:,.1f}"

# PART 12 · Models: forecast, early warning, and what a Gen1 fix could be worth
The analysis above says *where* and *why* failures happen. This part turns it into three operating tools and states plainly how far each can be trusted:

* **12.1 Next-day station failure-risk forecast** (gradient boosting, time-based hold-out on the Apr–Jun 2025 heat season) — a watch-list so operations can pre-position packs or dispatch support to the stations most likely to fail tomorrow. It is judged against two simple baselines, because a model that cannot beat "Gen1 on a hot day" is not worth running. The hold-out also tests generalisation: Gen3 stations barely existed in the training window.
* **12.2 Rider churn early warning** — kept as a *supporting exhibit* (AUC ≈ 0.69, recall ≈ 25 % at the chosen threshold), compared with the deterministic rule "≥ 2 failures in the first 14 days".
* **12.3 Gen1 fix break-even** — what a Gen1 retrofit could cost per station before it stops paying for itself on margin alone. No retrofit price is in the data, so none is assumed.


In [ ]:
# ===== 12.1 · MODEL · next-day station failure-risk forecast (pre-position packs / dispatch support where risk is highest) =====
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.inspection import permutation_importance
from sklearn.metrics import r2_score

# ---------- 1. station-day table
fd = (swaps.groupby(["station_id", "date"], observed=True)
      .agg(att=("is_done", "size"), fails=("service_fail", "sum")).reset_index())
fd["station_id"] = fd.station_id.astype(str)
fd["rate"] = fd.fails / fd.att

# lagged telemetry (daily aggregates of the hourly table)
hd = hourly[["station_id", "hour_start", "avg_charge_minutes", "stockout2w", "outage_minutes", "cabinet_temp_c", "charged_2w_avg"]].copy()
hd["station_id"] = hd.station_id.astype(str); hd["date"] = hd.hour_start.dt.normalize()
hd = hd.groupby(["station_id", "date"]).agg(charge_min=("avg_charge_minutes", "mean"), stockout=("stockout2w", "mean"), outage=("outage_minutes", "sum"),
                                            cab_temp=("cabinet_temp_c", "mean"), stock_avg=("charged_2w_avg", "mean")).reset_index()
fd = fd.merge(hd, on=["station_id", "date"], how="left")

# static station attributes
ST = ["station_id", "city", "charger_generation", "location_type", "host_type", "connectivity_tier", "slots_2w", "slots_3w",
      "inventory_target_2w", "inventory_target_3w", "grid_tariff_inr_kwh", "commissioned_date"]
fd = fd.merge(stations[ST], on="station_id", how="left")
fd["age_m"] = (fd.date - fd.commissioned_date).dt.days / 30.44

# weather / calendar (max temp = stand-in for a day-ahead forecast) + city temperature lags
cc = city_ctx.sort_values(["city", "date"]).copy()
cc["temp_l1"] = cc.groupby("city").max_temp_c.shift(1)
cc["temp_3d"] = cc.groupby("city").max_temp_c.transform(lambda s: s.shift(1).rolling(3, min_periods=1).mean())
cc["festival"] = cc.festival_or_event.notna().astype(int)
for c_ in ["is_public_holiday", "ecommerce_sale_event", "heat_alert"]: cc[c_] = cc[c_].astype(int)
fd = fd.merge(cc[["city", "date", "max_temp_c", "min_temp_c", "rainfall_mm", "heat_alert", "is_public_holiday", "festival", "ecommerce_sale_event", "temp_l1", "temp_3d"]],
              on=["city", "date"], how="left")
fd["dow"] = fd.date.dt.dayofweek; fd["month_n"] = fd.date.dt.month

# ---------- 2. lag features (strictly from previous days)
fd = fd.sort_values(["station_id", "date"]).reset_index(drop=True)
gfd = fd.groupby("station_id")
fd["rate_l1"] = gfd.rate.shift(1)
fd["rate_l7"] = gfd.rate.transform(lambda s: s.shift(1).rolling(7, min_periods=3).mean())
fd["att_l7"] = gfd.att.transform(lambda s: s.shift(1).rolling(7, min_periods=3).mean())
for c_ in ["charge_min", "stockout", "outage", "cab_temp", "stock_avg"]: fd[c_ + "_l1"] = gfd[c_].shift(1)
fd["att_per_slot_l7"] = fd.att_l7 / (fd.slots_2w + fd.slots_3w).replace(0, np.nan)

F_CAT = ["city", "charger_generation", "location_type", "host_type", "connectivity_tier"]
for c_ in F_CAT: fd[c_ + "_c"] = fd[c_].astype("category").cat.codes
F_NUM = ["max_temp_c", "min_temp_c", "temp_l1", "temp_3d", "rainfall_mm", "heat_alert", "is_public_holiday", "festival", "ecommerce_sale_event", "dow", "month_n", "age_m",
         "slots_2w", "slots_3w", "inventory_target_2w", "inventory_target_3w", "grid_tariff_inr_kwh", "rate_l1", "rate_l7", "att_l7", "att_per_slot_l7",
         "charge_min_l1", "stockout_l1", "outage_l1", "cab_temp_l1", "stock_avg_l1"]
FEATS = F_NUM + [c_ + "_c" for c_ in F_CAT]; cat_idx = [FEATS.index(c_ + "_c") for c_ in F_CAT]

# ---------- 3. time-based split: train before Mar-2025, test = the Apr-Jun 2025 heat season (March is an embargo gap)
tr = fd[fd.date < "2025-03-01"].dropna(subset=["rate"]); te = fd[fd.date >= "2025-04-01"].dropna(subset=["rate"]).copy()
print(f"train station-days {len(tr):,} ({tr.date.min().date()} to {tr.date.max().date()}) | test {len(te):,} ({te.date.min().date()} to {te.date.max().date()})")

model = HistGradientBoostingRegressor(max_depth=5, learning_rate=.05, max_iter=300, l2_regularization=1.0, categorical_features=cat_idx, random_state=SEED)
model.fit(tr[FEATS], tr.rate, sample_weight=tr.att)
te["pred"] = model.predict(te[FEATS])

# ---------- 4. baselines the model must beat
tr_hot = tr.assign(hot=tr.max_temp_c > 36)
rule_tab = tr_hot.groupby(["charger_generation", "hot"]).apply(lambda d: d.fails.sum() / d.att.sum()).rename("pred_rule").reset_index()
rule_all = tr_hot.groupby("hot").apply(lambda d: d.fails.sum() / d.att.sum())          # fallback: Gen3 saw (almost) no hot days in training
te["hot"] = te.max_temp_c > 36
te = te.merge(rule_tab, on=["charger_generation", "hot"], how="left")
te["pred_rule"] = te.pred_rule.fillna(te.hot.map(rule_all))
te["pred_last7"] = te.rate_l7.fillna(te.rate_l7.median())
base_rate = tr[tr.max_temp_c <= 34].fails.sum() / tr[tr.max_temp_c <= 34].att.sum()
te["excess"] = te.fails - te.att * base_rate

def evaluate(col):
    out = {"weighted MAE (pp)": np.average((te.rate - te[col]).abs(), weights=te.att) * 100,
           "weighted R2": r2_score(te.rate, te[col], sample_weight=te.att)}
    for q in (.10, .20):
        top = te.sort_values(col, ascending=False).head(int(len(te) * q))
        out[f"top {int(q*100)}% flagged: share of failures"] = top.fails.sum() / te.fails.sum()
        out[f"top {int(q*100)}% flagged: share of EXCESS failures"] = top.excess.clip(lower=0).sum() / te.excess.clip(lower=0).sum()
    return out
cmp_ = pd.DataFrame({"gradient boosting": evaluate("pred"), "rule: generation x hot day": evaluate("pred_rule"), "last-7-day station average": evaluate("pred_last7")}).T
print("\nHeat-season hold-out (random flagging would capture 10% / 20%)"); print(cmp_.round(3).to_string())
FIND["MODEL top-10% flagged station-days: share of failures / excess failures"] = f"{cmp_.loc['gradient boosting','top 10% flagged: share of failures']:.1%} / {cmp_.loc['gradient boosting','top 10% flagged: share of EXCESS failures']:.1%}"
m_cap = cmp_.loc["gradient boosting", "top 10% flagged: share of EXCESS failures"]; r_cap = cmp_.loc["rule: generation x hot day", "top 10% flagged: share of EXCESS failures"]
print(f"\nVERDICT: the model captures {m_cap:.1%} of excess failures in its top 10% vs {r_cap:.1%} for the simple rule ({(m_cap - r_cap)*100:+.1f} pp). "
      + ("The model adds meaningful targeting value over the rule." if m_cap - r_cap >= .03 else "The simple rule is almost as good - run the rule; the model's extra value is small."))
FIND["MODEL vs rule: top-10% excess-failure capture (pp difference)"] = f"{(m_cap - r_cap)*100:+.1f}"

# ---------- 5. calibration + what the model uses
te["temp_bin"] = pd.cut(te.max_temp_c, [-50, 30, 34, 36, 38, 40, 60])
cal = te.groupby(["temp_bin", "charger_generation"], observed=True).apply(lambda d: pd.Series({"actual_%": d.fails.sum() / d.att.sum() * 100, "predicted_%": np.average(d.pred, weights=d.att) * 100, "station_days": len(d)}))
print("\nCalibration by temperature and generation"); print(cal.round(2).to_string())
samp = te.sample(min(15000, len(te)), random_state=SEED)
pi = permutation_importance(model, samp[FEATS], samp.rate, scoring="neg_mean_absolute_error", n_repeats=3, random_state=SEED, sample_weight=samp.att)
imp = pd.Series(pi.importances_mean, index=FEATS).sort_values(ascending=False)
print("\nPermutation importance (MAE increase when shuffled)"); print(imp.head(12).round(5).to_string())

# ---------- 6. operational value (ASSUMPTION: extra packs / dispatch on flagged days remove ACTION_EFFECT of failures) - shown as a sensitivity
FLAG_SHARE = 0.10
flag = te.sort_values("pred", ascending=False).head(int(len(te) * FLAG_SHARE)); cm_sw = done.cm_var.mean()
print(f"\nFlagging the top {FLAG_SHARE:.0%} station-days ({len(flag)/te.date.nunique():.0f} stations per day). Value depends on an ASSUMED action effect (share of failures removed on flagged days):")
val = pd.DataFrame({"action effect": [.10, .20, .30, .50]})
val["failures avoided"] = flag.fails.sum() * val["action effect"]
val["% of all test-period failures"] = val["failures avoided"] / te.fails.sum() * 100
val["variable CM recovered, INR m (upper bound)"] = val["failures avoided"] * cm_sw / 1e6
print(val.round(2).to_string(index=False))
ACTION_EFFECT = 0.30
avoided = flag.fails.sum() * ACTION_EFFECT
FIND["MODEL value at assumed 30% action effect: failures avoided / CM recovered INR m (test period, assumption-driven)"] = f"{avoided:,.0f} / {avoided*cm_sw/1e6:,.2f}"

# ---------- 7. the daily watchlist an ops team would receive
def watchlist(day, k=15):
    d = te[te.date == pd.Timestamp(day)].sort_values("pred", ascending=False).head(k)
    return d[["station_id", "city", "charger_generation", "max_temp_c", "pred", "rate"]].assign(pred=lambda x: x.pred * 100, rate=lambda x: x.rate * 100).rename(columns={"pred": "predicted_fail_%", "rate": "actual_fail_%"})
print("\nExample watchlist for the hottest test day:"); print(watchlist(te.loc[te.max_temp_c.idxmax(), "date"]).round(1).to_string(index=False))

fig, axes = plt.subplots(1, 3, figsize=(19, 4.5))
imp.head(12)[::-1].plot.barh(ax=axes[0]); axes[0].set_title("Top drivers of predicted risk")
cal_p = cal.reset_index()
for gname, d in cal_p.groupby("charger_generation"):
    axes[1].plot(d.temp_bin.astype(str), d["actual_%"], "o-", label=f"{gname} actual"); axes[1].plot(d.temp_bin.astype(str), d["predicted_%"], "x--", label=f"{gname} predicted")
axes[1].set_title("Calibration by temperature"); axes[1].legend(fontsize=7); axes[1].tick_params(axis="x", rotation=30)
ql = te.assign(dec=pd.qcut(te.pred, 10, labels=False, duplicates="drop")).groupby("dec").apply(lambda d: d.fails.sum() / d.att.sum() * 100)
ql.plot.bar(ax=axes[2], color="tab:red"); axes[2].set_title("Actual failure % by predicted-risk decile (test)"); axes[2].set_xlabel("decile (9 = highest predicted risk)")
plt.tight_layout(); plt.show()


**How to read it.**
* *Baselines:* "rule: generation × hot day" and "last-7-day station average" are what ops could do without a model. The model earns its place only if the printed **VERDICT** line says it adds a meaningful share of excess failures captured in the top 10 % of station-days.
* *Calibration:* the table and middle chart show whether predictions hold by temperature band and generation. Gen3 had almost no hot days in the training window, so a miss there is expected and is itself a finding — retrain after each heat season.
* *Weather inputs:* the model uses the day's maximum temperature, min temperature, rainfall and heat-alert flag as stand-ins for a day-ahead forecast. In production, forecast error will cost some accuracy; the lagged features (previous-day failure rate, 7-day average, charge time, stock) are strictly from earlier days.
* *Operational value:* the avoided-failure figures rest on an **assumed** action effect (the share of failures removed on flagged days). Packs-in-place is not proven to fix failures — only 3.2 % of no-battery failures fall in stock-out hours — so the sensitivity table shows the value across effect sizes, and the recommendation is to **trial** the watch-list on a sample of stations before relying on it.
* The watch-list at the end is the format an operations team would receive each morning.

In [ ]:
# ===== 12.2 · RIDER CHURN EARLY-WARNING SCORE (supporting exhibit) vs the simple rule ">= 2 failures in the first 14 days" =====
from sklearn.metrics import classification_report, roc_curve, auc, precision_score, recall_score

X_ew, y_ew = X.copy(), 1 - y                                   # 1 = churner (same riders and features as 9.5)
X_train_ew, X_test_ew, y_train_ew, y_test_ew = train_test_split(X_ew, y_ew, test_size=0.3, random_state=SEED, stratify=y_ew)
churn_model = HistGradientBoostingClassifier(max_depth=4, learning_rate=0.05, max_iter=300, random_state=SEED).fit(X_train_ew, y_train_ew)
preds_prob = churn_model.predict_proba(X_test_ew)[:, 1]
THRESH = 0.35
preds_class = (preds_prob >= THRESH).astype(int)
print("=== EARLY-WARNING SCORE (hold-out) ===")
print(f"ROC-AUC: {roc_auc_score(y_test_ew, preds_prob):.3f} | base churn rate {y_test_ew.mean():.1%}")
print(f"Classification report at threshold {THRESH}:"); print(classification_report(y_test_ew, preds_class, digits=2))

# lift: how concentrated is churn in the highest-scored riders?
yt = np.asarray(y_test_ew); order = np.argsort(-preds_prob)
lift_rows = []
for q in (.05, .10, .20):
    k = int(len(order) * q); top = yt[order[:k]]
    lift_rows.append({"top share of riders flagged": f"{q:.0%}", "churn rate in flagged": top.mean(), "lift vs base": top.mean() / yt.mean(), "share of all churners captured": top.sum() / yt.sum()})
print("Lift of the score"); print(pd.DataFrame(lift_rows).round(3).to_string(index=False))

# the simple rule
n_fail_te = X_test_ew.n_nobatt + X_test_ew.n_abandon + X_test_ew.n_syserr
rule = (n_fail_te >= 2).astype(int).values
cmp_ew = pd.DataFrame({"flagged share of riders": [preds_class.mean(), rule.mean()],
                       "precision": [precision_score(yt, preds_class), precision_score(yt, rule)],
                       "recall": [recall_score(yt, preds_class), recall_score(yt, rule)]}, index=["early-warning score (>= 0.35)", "rule: >= 2 failures in first 14 days"])
print("\nScore vs simple rule"); print(cmp_ew.round(3).to_string())
FIND["Q6 early-warning score: AUC / recall at 0.35 / rule recall"] = f"{roc_auc_score(y_test_ew, preds_prob):.3f} / {recall_score(yt, preds_class):.2f} / {recall_score(yt, rule):.2f}"

fpr, tpr, _ = roc_curve(y_test_ew, preds_prob)
plt.figure(figsize=(7, 5))
plt.plot(fpr, tpr, color="darkorange", lw=2, label=f"score (AUC = {auc(fpr, tpr):.3f})")
plt.plot([0, 1], [0, 1], color="navy", lw=1.5, ls="--")
plt.scatter([rule[yt == 0].mean()], [rule[yt == 1].mean()], color="red", zorder=5, label="rule: >= 2 early failures")
plt.xlabel("False positive rate"); plt.ylabel("True positive rate"); plt.title("Rider churn early warning: score vs simple rule"); plt.legend(loc="lower right"); plt.show()


**How to read it.** The score ranks riders better than chance but is a weak classifier (low recall at the 0.35 threshold), which is what the retention analysis predicted: most retention variation is not explained by anything observed in the first 14 days. The lift and rule comparison show the practical choice: a simple, explainable trigger (≥ 2 failures, or a failed first attempt for no battery) is easier to run and the model is a supplement, not a replacement.

In [ ]:
# ===== 12.3 · WHAT COULD A GEN1 FIX BE WORTH? break-even cost per station (no retrofit price is assumed - none is in the data) =====
g1 = S[(S.charger_generation == "Gen1")].copy()
g1["max_cost_3y_margin_only"] = g1.lost_cm_per_year * 3
g1["max_cost_5y_margin_only"] = g1.lost_cm_per_year * 5
city_tab = g1.groupby("city").agg(stations=("station_id", "size"), stations_with_excess=("excess_fails", lambda s: int((s > 0).sum())), excess_failures_total=("excess_fails", "sum"),
                                  lost_revenue_per_yr=("lost_revenue_per_year", "sum"), lost_cm_per_yr=("lost_cm_per_year", "sum"))
city_tab["lost_cm_per_station_yr"] = city_tab.lost_cm_per_yr / city_tab.stations_with_excess.replace(0, np.nan)
city_tab["max_cost_per_station_3y"] = city_tab.lost_cm_per_station_yr * 3
city_tab["max_cost_per_station_5y"] = city_tab.lost_cm_per_station_yr * 5
print("Gen1 stations: value of removing their EXCESS failures (vs the non-Gen1 failure rate), by city  [INR; assumes every avoided failure is a recovered swap = upper bound]")
print(city_tab.round(0).to_string())
print("\nTen Gen1 stations with the largest recoverable margin")
print(g1.sort_values("lost_cm_per_year", ascending=False)[["station_id", "city", "zone", "connectivity_tier", "excess_fails", "lost_revenue_per_year", "lost_cm_per_year", "max_cost_3y_margin_only", "max_cost_5y_margin_only"]].head(10).round(0).to_string(index=False))
tot_cm = g1.lost_cm_per_year.sum()
print(f"\nAll Gen1 stations: lost variable CM INR {tot_cm/1e6:,.2f} m a year, lost revenue INR {g1.lost_revenue_per_year.sum()/1e6:,.2f} m a year "
      f"-> a margin-only payback within 3 years allows at most INR {tot_cm*3/len(g1[g1.excess_fails > 0]):,.0f} per affected station.")
print("A retrofit is therefore a SERVICE / RETENTION decision: the margin it recovers is small, but >= 2 early failures cost new riders 6.7-9.5 pp of retention (11.8).")
FIND["Q3 Gen1 fix: break-even cost per affected station, 3y margin-only (INR)"] = f"{tot_cm*3/max(len(g1[g1.excess_fails > 0]), 1):,.0f}"


**How to read it.** The margin recovered by fixing Gen1 is small (the table gives it per station and per city), so the case for a retrofit is **service and retention**, not margin: ≥ 2 early failures cost new riders 6.7–9.5 pp of retention (11.8). The "max cost" columns are the break-even retrofit cost per station if the margin alone had to pay back in 3 or 5 years; any real retrofit quote can be compared with them directly. Stations without excess failures (Bengaluru, Mumbai) are not candidates.

# PART 13 · Key findings, recommendations, limitations, exports

In [ ]:
# ---------- 13.1 every number collected along the way, plus the cleaning log
print(f"notebook runtime so far: {(time.time()-T_NB)/60:.1f} min\n")
print("=== numbers collected for the report ===")
for k, v in FIND.items(): print(f"- {k}: {v}")
print("\n=== cleaning log ===")
print(pd.DataFrame(CLEAN_LOG).to_string(index=False))
print("\nEVENTS:", {k: (v.date() if isinstance(v, pd.Timestamp) else v) for k, v in EVENTS.items()})

In [ ]:
# ===== 13.2 · evidence table for the four budget proposals + export =====
ev_rows = [
 ("MORE STATIONS", "failure cause mix (stock / queue / error)", "63% / 32% / 5%", "queue-driven failures are a third of the total"),
 ("MORE STATIONS", "utilisation vs failure (busiest quintile vs rest; spearman with abandon)", "5.20% vs 5.23%; 0.00", "no capacity signal -> not supported"),
 ("MORE STATIONS", "wave stations: station share vs city failure rate", str(FIND.get("Q3 waves: station share by city vs failure %", "n/a")), "were they placed where service was worst?"),
 ("MORE BATTERIES", "no-battery failures that fall in telemetry stock-out hours", "3.2%", "stock-out hours are not the main cause"),
 ("MORE BATTERIES", "Kyron bad lots: packs / retired / replacement cost INR m", str(FIND.get("Q4 bad lots / packs / retired / replacement cost INR m", "n/a")), "replace-and-claim supplier, do not just buy more"),
 ("PRICING ROLLOUT", "pilot: peak share DiD / revenue per swap", f"{FIND.get('Q5 pilot effect on peak share (independents, DiD)','n/a')} / see NEXT-7", "peak share -2.6pp, but swaps per rider -10%"),
 ("PRICING ROLLOUT", "per-swap price effect x non-pilot swaps, INR m/yr (ignores volume loss) | volume-adjusted (95% CI)", f"{FIND.get('Q5 pilot: per-swap price effect x non-pilot swaps, INR m/yr (ignores volume loss)', 'n/a')} | {FIND.get('Q5 pilot: volume-adjusted rollout revenue INR m/yr (point; 95% CI)', 'n/a')}", "independents swapped 10% less per week; gain not significant; does not fix failures"),
 ("EXCLUSIVE WITH LARGEST PARTNER", "ZipDrop loaded CM per swap rank / volume uplift achieved vs needed", f"{FIND.get('Q5 ZipDrop discount: achieved vs needed volume uplift','n/a')}", "largest partner is the biggest loss-maker; 28% discount bought ~2% volume"),
 ("INSTEAD: FIX GEN1 IN HEAT", "Gen1 share of excess failures / lost CM per year (INR m)", str(FIND.get("Q3 Gen1: share of excess failures / lost CM per year (INR m)", "n/a")), "service/retention case (margin recovery is small); mechanism to be confirmed in a pilot"),
 ("INSTEAD: FAILURE-RISK WATCH-LIST (trial)", "top-10% flagged station-days: share of failures / excess failures | model vs rule (pp)", f"{FIND.get('MODEL top-10% flagged station-days: share of failures / excess failures', 'n/a')} | {FIND.get('MODEL vs rule: top-10% excess-failure capture (pp difference)', 'n/a')}", "targeting tool; value rests on an assumed action effect"),
 ("INSTEAD: EARLY-FAILURE RESCUE", "retention gap for >=2 early failures (adjusted, pp)", str(FIND.get("Q6 adjusted effect of >=2 early failures on retention (pp)", "n/a")), "credit / priority battery after the 2nd failed attempt"),
]
budget = pd.DataFrame(ev_rows, columns=["proposal", "evidence", "value", "reading"]); print(budget.to_string(index=False, max_colwidth=90))

# export tables for the report
tbls = {"monthly": monthly, "monthly_newwear": mm, "scorecard": scorecard.reset_index(), "partners": pt, "forecast_model": cmp_, "gen1_break_even_by_city": city_tab, "wear_sensitivity": sens, "early_warning_vs_rule": cmp_ew, "station_actions": S, "drivers": driver, "group_importance": gi,
        "lots": lot, "supplier": sup_t, "heat_by_gen": heat_tab.reset_index(), "budget_evidence": budget}
with pd.ExcelWriter(f"{OUT}/voltrelay_tables.xlsx") as xw:
    for k, v in tbls.items():
        try:
            vv = v.copy(); vv.index = vv.index.astype(str) if not isinstance(vv.index, pd.RangeIndex) else vv.index
            vv.columns = [str(c) for c in vv.columns]; vv.to_excel(xw, sheet_name=k[:31])
        except Exception as ex_: print("skip", k, ex_)
pd.Series({k: str(v) for k, v in FIND.items()}).to_csv(f"{OUT}/key_numbers.csv")
import shutil; shutil.make_archive("/content/voltrelay_outputs", "zip", OUT)
print("\nsaved:", sorted(os.listdir(OUT))[:8], "... -> /content/voltrelay_outputs.zip (download from the Colab file browser)")

## Key findings — one headline and the number that proves it

| # | Question | Headline | Evidence |
|---|---|---|---|
| 1 | Network performance | Volume and revenue grew ~3× (324 k → 896 k swaps per quarter; INR 19.4 m → 58.2 m). The apparent doubling of the failure rate is **seasonal** (same-season Q2: 9.5 % → 7.9 %). Variable margin per swap is flat (INR 7.4 → 7.0); loaded margin improved only through fixed-cost dilution and stays negative on measured wear. The **July price rise added ≈ INR 7 per swap; the Kyron lots took ≈ INR 6.5 back**. | 4.2–4.4, 11.2, 11.3, 11.5 |
| 2 | Service failures | Heat-driven and concentrated: days > 36 °C are 17 % of attempts but 41 % of failures; the worst 10 % of stations produce 25 % of failures; 63 % of failures are no-battery; 3W riders fail twice as often as 2W (9.95 % vs 5.02 %). | 5.1–5.4, 11.2, 11.4 |
| 3 | Stations & geography | **Gen1 chargers** (51 stations) carry 91 % of excess failures: 91.8-minute charge time vs 64 / 42, failure 14–19 % above 36 °C vs 5 % on Gen3. Geography is mostly Gen1 mix (Jaipur, Delhi NCR, Hyderabad). Gen1 is not hotter; mechanism not fully identified. | 6.1–6.4, 11.2, 11.11 |
| 4 | Batteries | **Kyron lots KY-2407/08/09** (1,461 packs) degrade 2× faster, deliver ≈ 5 % less range at the same SoH, retire at ≈ 511 swaps vs ≈ 1,217; replacement cost INR 47.9 m (20.9 % of revenue). | 7.1–7.2, 3.12, 11.3, 11.10 |
| 5 | Pricing & partners | Base-price rise had no visible volume cost. The pilot shifts timing (−2.6 pp peak share) but cuts swaps per rider 10 % with no significant revenue-per-rider gain. **ZipDrop, the largest partner, is the biggest loss-maker** (−INR 16.5 per swap loaded, ≈ −INR 9.1 m; discount 12 % → 28 % bought +2 % volume; variable break-even 17 %). | 8.1–8.5, 11.6, 11.7 |
| 6 | Retention | **Primary:** ≥ 2 failures in the first 14 days (−6.7 pp adjusted; cohort retention vs network failure rate r = −0.81). **Mediator:** early engagement. **Secondary:** nearby competitor (−4.1 pp), partner affiliation, 3W. **Not drivers:** price, support, battery condition. | 9.3–9.6, 11.8 |
| 7 | Models | A next-day failure-risk watch-list and a rider early-warning score are provided as targeting tools; both are judged against simple rules (12.1–12.2). | 12.1, 12.2 |

## Recommendations (in priority order)
1. **Fix Gen1 in the heat — as a service and retention investment.** Start with the stations > 3σ above the network rate (36 of 150; the worst 15 are all Gen1 sites in Delhi NCR and Jaipur, followed by Hyderabad). Margin recovery is small (INR ≈ 0.2 m a year of variable contribution), so the case rests on retention (2 early failures = −6.7 to −9.5 pp). Because the mechanism is not proven, **pilot on ~5 stations through the next heat season** (charger swap or cooling), compare with matched untreated Gen1 stations, then scale. Use 12.3 to compare any retrofit quote with the break-even cost.
2. **Recover or replace the Kyron lots KY-2407/08/09.** Open a supplier claim, hold new Kyron orders until lot QA is shown, and add a lot-level degradation z-score alert. Impact: ≈ INR 6.5 per swap of loaded margin and 47.9 m of replacement cost already incurred.
3. **Renegotiate ZipDrop to ≤ 12 % (its original level)** and decline the long-term exclusive. Worth ≈ INR 10.7 per swap ≈ INR 1.2 m per quarter at current volume; 17 % is the variable break-even, below which each swap loses money before site cost.
4. **Rescue new riders after their second failure** (credit and priority battery) — 16.3 % of new riders, ≈ 214 riders per cohort window. Use the deterministic trigger; the early-warning score (12.2) is a supplement.
5. **Do not roll out peak/off-peak pricing network-wide.** Extend it to a third city with a staggered design and track swaps per rider, not only revenue per swap.
6. **Target the next expansion wave at the need, with Gen3.** Wave stations were not placed where service was worst (Jaipur 7.4 % failure got 13 % of wave stations, Bengaluru 4.1 % got 20 %; 90 % of Wave 1 went to fuel pumps and residential sites) and Gen1 sites received no relief (monthly attempts per Gen1 station roughly doubled from Jan-24 to Jan-25 and stayed at ≈ 2,200–2,800).
7. **Investigate 3W separately.** Failure is twice 2W even on cool days and not explained by demand per pack; 3W loaded margin is ≈ −INR 20 per swap against −5 for 2W.
8. **Measure what matters:** heat-adjusted failure rate by generation, station z-score, lot degradation z-score, early-failure flag per new rider, 3W failure rate, and the ticket text family (21.6 % of tickets are battery/vehicle complaints filed as `other`).
9. **Operate the failure-risk watch-list as a trial** on a sample of stations during the next heat season (12.1) and measure whether pre-positioning actually reduces failures.

## Limitations
* Synthetic, observational data: patterns are internally consistent, but associations are not proof of causation.
* Loaded margin depends on the battery-wear model and the end-of-life SoH (extrapolated for Cellora and Amptek, whose packs are not retired); conclusions are stated for changes, not levels.
* The pricing pilot has two treated cities; the early-failure retention effect is adjusted but not randomised; value-at-stake figures rest on a stated assumption (6-month remaining rider life).
* 45,813 completed swaps are logged on packs after their retirement date; their effect is tested in 11.3.
* The failure-risk model's operational value rests on an assumed action effect and on weather inputs standing in for forecasts.


In [ ]:
# ===== 13.3 · write the one-page executive summary and zip all outputs (numbers match the printed results above) =====
summary = f"""VOLTRELAY ENERGY - EXECUTIVE SUMMARY  (generated {pd.Timestamp.now():%Y-%m-%d})

1. NETWORK PERFORMANCE
Completed swaps per quarter grew ~3x (324k -> 896k) and revenue ~3x (INR 19.4m -> 58.2m). The apparent doubling of the failure rate (3.98% -> 7.92%) compares
Jan-Mar 2024 with Apr-Jun 2025 and is SEASONAL: like-for-like, Q2 improved from 9.52% to 7.92%; outside the Apr-Jun heat season the rate is flat at ~3.9%.
Variable margin per swap is flat (INR 7.4 -> 7.0); loaded margin improves only through fixed-cost dilution. The July-2024 price rise added ~INR 7 per swap;
the Kyron lots KY-2407/08/09 took ~INR 6.5 back. The LEVEL of loaded margin depends on the battery-wear model; the movements do not.

2. WHAT DRIVES FAILURES
Days above 36C: 17% of attempts, 41% of failures. Gen1 chargers (51 stations, 91.8-min charge time vs 64 / 42) carry 91% of excess failures and fail 14-19%
above 36C vs ~5% on Gen3. Gen1 is NOT hotter (cabinet 24.9C vs 24.6 / 28.3); the mechanism is not fully identified. 3W riders fail 2x as often as 2W.

3. RETENTION
Primary driver: >= 2 service failures in a new rider's first 14 days (-6.7pp adjusted; 16% of new riders). Early engagement is a mediator. Secondary: nearby
competitor, partner affiliation, 3W. Not drivers: price, support, battery condition.

4. PARTNERS AND PRICING
ZipDrop (largest partner) loses INR 16.5 per swap after energy, wear and site cost (about -INR 9.1m); its 12% -> 28% discount bought ~2% volume; variable
break-even discount is 17%. The peak/off-peak pilot shifts timing (-2.6pp peak share) but independents swap 10% less per week; revenue per rider-week is not
significantly higher.

5. RECOMMENDATIONS
(1) Fix Gen1 in the heat as a service/retention investment, piloted first. (2) Claim against / replace the Kyron lots; add a lot z-score alert. (3) Renegotiate
ZipDrop to <= 12%; decline the exclusive. (4) Rescue new riders after their second failure. (5) No network-wide pricing rollout yet; extend the pilot.
(6) Point the next expansion wave at high-failure cities with Gen3. (7) Investigate 3W separately. (8) Trial the failure-risk watch-list.

MODEL: top-10% flagged station-days capture (failures / excess failures): {FIND.get('MODEL top-10% flagged station-days: share of failures / excess failures', 'n/a')}; model vs simple rule: {FIND.get('MODEL vs rule: top-10% excess-failure capture (pp difference)', 'n/a')} pp.
"""
with open(f"{OUT}/voltrelay_executive_summary.txt", "w") as f: f.write(summary)
print(summary)
import shutil
shutil.make_archive("/content/voltrelay_outputs", "zip", OUT)
print("saved ->", sorted(os.listdir(OUT))[:6], "... /content/voltrelay_outputs.zip (download from the Colab file browser)")
